# R2 Migrate (CPU · Resumable Cloudflare R2 → Google Drive Migration + Web UI)

Runs a pure-stdlib, high-concurrency Cloudflare R2 S3 → Google Drive migration engine + Web UI on a zero-GPU Colab CPU runtime (`gpu=None`, optional `--high-mem`) behind Caddy Basic Auth and Bearer token auth. Preserves multi-bucket hierarchy, checkpoints local NVMe SQLite WAL state + `StartAfter` cursors to Google Drive for exact interruption resumption, verifies size + MD5 ETag before optional Free 1,000-key batch `DeleteObjects`, and minimizes R2 API costs (`free-first` / `free-only` / `unlimited` + CDN cache bypass).

Run it with the service script (`colabllm <name> up`), which creates the Colab session, mounts Drive with `colab dfs`, optionally creates a named Cloudflare tunnel, and injects `CT_*` settings. Cell order is the standard contract -- see `.agents/skills/colab-llm-service/SKILL.md`.

In [ ]:
# @title 0. Config -- values come from `colab exec --env` (defaults allow running in the Colab UI)
import json, os, re, secrets, shutil, stat, subprocess, sys, time, urllib.request, urllib.error

SERVICE      = os.environ.get("CT_SERVICE", "r2-migrate")
SERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "8000"))
AUTH         = os.environ.get("CT_AUTH", "caddy")            # caddy | builtin | none
PUBLIC_PORT  = int(os.environ.get("CT_PUBLIC_PORT", "18080"))  # what cloudflared targets
AUTH_USER    = os.environ.get("CT_AUTH_USER", "admin")
AUTH_PASS    = os.environ.get("CT_AUTH_PASS") or secrets.token_urlsafe(18)
API_TOKEN    = os.environ.get("CT_API_TOKEN") or "sk-" + secrets.token_urlsafe(32)
TUNNEL_TOKEN = os.environ.get("CT_TUNNEL_TOKEN", "")            # set -> fixed domain (named tunnel)
PUBLIC_HOST  = os.environ.get("CT_PUBLIC_HOSTNAME", "")
CREDS_FROM_ENV = bool(os.environ.get("CT_AUTH_PASS"))

WORK = "/content/ct"                                            # logs + binaries of this notebook
os.makedirs(WORK, exist_ok=True)
for _p in ("/content/drive/MyDrive/colab-llm/hf_token.txt",
           "/content/drive/MyDrive/colab-services/hf_token.txt",
           "/content/drive/MyDrive/colab-services/qwen-flash/hf_token.txt"):
    if not os.environ.get("HF_TOKEN") and os.path.isfile(_p):
        _tok = open(_p).read().strip()
        if _tok:
            os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
            break
MOUNT_DRIVE = os.environ.get("CT_MOUNT_DRIVE", "1") != "0"
DATA_DIR = os.environ.get("CT_DATA_DIR", f"/content/drive/MyDrive/colab-llm/{SERVICE}")
if not MOUNT_DRIVE or not os.path.isdir("/content/drive/MyDrive"):   # Drive not mounted -> local fallback
    if DATA_DIR.startswith("/content/drive/"):
        DATA_DIR = f"/content/data/{SERVICE}"
os.makedirs(DATA_DIR, exist_ok=True)
PROCS = {}                                                      # name -> (Popen, restart function)

def ct_spawn(name, args, env=None, cwd=None):
    """Start a background process; output goes to /content/ct/<name>.log."""
    with open(f"{WORK}/{name}.log", "a") as logf:   # child keeps its own dup of the fd
        return subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, cwd=cwd,
                                env={**os.environ, **(env or {})}, start_new_session=True)

def ct_stop_proc(proc, timeout=15):
    """Terminate a ct_spawn process group, escalating to SIGKILL after a bounded wait."""
    import signal
    if proc is None or proc.poll() is not None:
        return
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(proc.pid, sig)
        except (ProcessLookupError, PermissionError):
            try:
                proc.send_signal(sig)
            except ProcessLookupError:
                return
        try:
            proc.wait(timeout=timeout)
            return
        except subprocess.TimeoutExpired:
            continue

def ct_gpu_stat(query="memory.used,memory.total,utilization.gpu"):
    """nvidia-smi summary that never raises (CPU runtimes / transient driver errors)."""
    try:
        return subprocess.check_output(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader"],
                                       text=True, stderr=subprocess.DEVNULL, timeout=20).strip()
    except (OSError, subprocess.SubprocessError):
        return "cpu/no-gpu"

def ct_log_size(name):
    p = f"{WORK}/{name}.log"
    return os.path.getsize(p) if os.path.exists(p) else 0

def ct_log_since(name, offset):
    p = f"{WORK}/{name}.log"
    if not os.path.exists(p):
        return ""
    with open(p, "rb") as handle:
        handle.seek(min(offset, os.path.getsize(p)))
        return handle.read().decode("utf-8", errors="replace")

def ct_public_gate(public_url, probe, verify_auth, log, timeout=120, interval=8):
    """Wait for the public endpoint; auth failures are fatal, network errors are retried."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            code = probe()
        except Exception as err:   # DNS / edge propagation / TLS warm-up
            log(f"Public probe pending ({type(err).__name__}: {err})")
            time.sleep(interval)
            continue
        if code == 200:
            verify_auth()            # RuntimeError propagates: never report CT_READY with broken auth
            return
        log(f"Public probe pending (HTTP {code})")
        time.sleep(interval)
    raise RuntimeError(f"Public endpoint {public_url} failed readiness verification within {timeout}s")

def ct_watchdog(health, start_service, log, *, service_key="service", log_name="service",
                interval=60, grace=900, dead_markers=(), heartbeat=None):
    """Supervise PROCS forever: restart dead processes, debounce health failures after reloads."""
    started = time.monotonic()
    failures = 0
    grace_until = time.monotonic()
    log_offset = ct_log_size(log_name)

    def restart_service(reason):
        nonlocal failures, grace_until, log_offset
        log(f"[watchdog] {reason}; restarting {service_key}")
        ct_stop_proc(PROCS[service_key][0])
        PROCS[service_key] = (start_service(), start_service)
        failures = 0
        grace_until = time.monotonic() + grace
        log_offset = ct_log_size(log_name)

    while True:
        time.sleep(interval)
        for name, (proc, restart) in list(PROCS.items()):
            if proc.poll() is not None:
                log(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 20)}")
                PROCS[name] = (restart(), restart)
                if name == service_key:
                    failures = 0
                    grace_until = time.monotonic() + grace
                    log_offset = ct_log_size(log_name)
        if dead_markers:
            fresh = ct_log_since(log_name, log_offset)
            log_offset += len(fresh.encode("utf-8", errors="replace"))
            if any(marker in fresh for marker in dead_markers):
                restart_service("Detected dead worker marker in service log")
                continue
        try:
            healthy = bool(health())
        except Exception:
            healthy = False
        if healthy:
            failures = 0
        elif time.monotonic() >= grace_until:
            failures += 1
        if failures >= 6:
            restart_service("6 consecutive failed health checks")
        if heartbeat is not None:
            try:
                heartbeat(int(time.monotonic() - started))
            except Exception as err:
                log(f"[watchdog] heartbeat warning: {err}")

def ct_wait_http(url, timeout=600, ok=lambda code: code < 500):
    """Wait until url answers with a non-5xx status (401/404 count as 'listening')."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r.status): return r.status
        except urllib.error.HTTPError as e:
            if ok(e.code): return e.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def ct_tail(name, n=30):
    p = f"{WORK}/{name}.log"
    return "".join(open(p, errors="replace").readlines()[-n:]) if os.path.exists(p) else ""

def ct_fetch_binary(path, url):
    if not os.path.exists(path):
        partial = f"{path}.{os.getpid()}.partial"
        with urllib.request.urlopen(url, timeout=60) as src, open(partial, "wb") as dst:
            shutil.copyfileobj(src, dst)
        os.chmod(partial, os.stat(partial).st_mode | stat.S_IEXEC)
        os.replace(partial, path)
    return path

print(f"service={SERVICE} port={SERVICE_PORT} auth={AUTH} public_port={PUBLIC_PORT} "
      f"tunnel={'named ' + PUBLIC_HOST if TUNNEL_TOKEN else 'quick'} data={DATA_DIR}")


In [ ]:
# @title 1. Setup -- stage pure-stdlib R2 migration engine & verify Drive mount (CPU VM, zero pip install)
from pathlib import Path

R2_WORK = Path(os.environ.get('CT_R2_WORK_DIR', '/content/r2-migrate'))
R2_SRC = R2_WORK / 'src'
for _d in (R2_WORK, R2_SRC, R2_WORK / 'state', R2_WORK / 'spool'):
    _d.mkdir(parents=True, exist_ok=True)

if MOUNT_DRIVE and not os.path.isdir('/content/drive/MyDrive'):
    raise RuntimeError('Google Drive is not mounted at /content/drive/MyDrive; run `colab dfs mount` or pass `--no-drive` for local testing.')

(R2_SRC / 'r2_s3.py').write_text('"""Pure-stdlib Cloudflare R2 S3 SigV4 client, cost-aware ledger, and safe Drive path mapper."""\nfrom __future__ import annotations\n\nimport base64\nimport concurrent.futures\nimport contextlib\nimport dataclasses\nimport datetime as dt\nimport hashlib\nimport hmac\nimport http.client\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport re\nimport ssl\nimport threading\nimport time\nfrom typing import Any, Callable, Sequence\nimport urllib.parse\nimport xml.etree.ElementTree as ET\nfrom xml.sax.saxutils import escape as xml_escape\n\nFREE_CLASS_A_MONTHLY = 1_000_000\nFREE_CLASS_B_MONTHLY = 10_000_000\nDEFAULT_SAFE_FREE_CLASS_A = 950_000\nDEFAULT_SAFE_FREE_CLASS_B = 9_500_000\n\nPRICE_CLASS_A_PER_MILLION = 4.50\nPRICE_IA_CLASS_A_PER_MILLION = 9.00\nPRICE_CLASS_B_PER_MILLION = 0.36\nPRICE_IA_CLASS_B_PER_MILLION = 0.90\nPRICE_IA_RETRIEVAL_PER_GB = 0.01  # decimal 10^9 bytes per R2 pricing\n\n_HEX32_RE = re.compile(r"^[0-9a-fA-F]{32}$")\n_ACCOUNT_ID_RE = re.compile(r"^[A-Za-z0-9_-]{8,64}$")\n_XML_ILLEGAL_CTRL_RE = re.compile(r"[\\x00-\\x08\\x0b\\x0c\\x0e-\\x1f]")\n_LOOPBACK_HOSTS = {"127.0.0.1", "localhost", "::1", "[::1]"}\n_RETRY_STATUSES = {429, 500, 502, 503, 504}\n_CF_CACHE_HIT_STATUSES = {"HIT", "REVALIDATED", "UPDATING", "STALE"}\n\n\nclass R2Error(RuntimeError):\n    """Base error for Cloudflare R2 S3 operations."""\n\n\nclass R2HttpError(R2Error):\n    """HTTP error returned by R2 / S3 or CDN endpoint."""\n\n    def __init__(self, status: int, message: str, *, code: str = "", body: str = "") -> None:\n        super().__init__(f"HTTP {status}" + (f" ({code})" if code else "") + f": {message}")\n        self.status = status\n        self.code = code\n        self.body = body\n\n\nclass R2IntegrityError(R2Error):\n    """Downloaded object failed size or MD5 ETag integrity verification."""\n\n\nclass R2TruncatedReadError(R2IntegrityError):\n    """Downloaded object stream ended early before expected Size."""\n\n\ndef s3_quote(value: str, safe: str = "-_.~") -> str:\n    """Percent-encode a string strictly according to AWS SigV4 RFC 3986 rules."""\n    return urllib.parse.quote(value, safe=safe, encoding="utf-8", errors="strict")\n\n\ndef s3_quote_path(key: str) -> str:\n    """Percent-encode an S3 object key while preserving \'/\' hierarchy delimiters."""\n    return urllib.parse.quote(key, safe="/-_.~", encoding="utf-8", errors="strict")\n\n\ndef mask_secret(value: str | None) -> str:\n    """Return a non-sensitive masked representation of an access key or secret."""\n    text = (value or "").strip()\n    if not text:\n        return ""\n    if len(text) <= 8:\n        return "*" * len(text)\n    return f"{text[:4]}...{text[-4:]}"\n\n\ndef current_utc_month() -> str:\n    return dt.datetime.now(dt.timezone.utc).strftime("%Y-%m")\n\n\ndef parse_public_urls(items: Sequence[str] | dict[str, str] | None) -> dict[str, str]:\n    """Validate and normalize per-bucket CDN public URL mappings."""\n    mapping: dict[str, str] = {}\n    if not items:\n        return mapping\n    pairs: list[tuple[str, str]] = []\n    if isinstance(items, dict):\n        pairs = [(str(k), str(v)) for k, v in items.items()]\n    else:\n        for entry in items:\n            raw = str(entry or "").strip()\n            if not raw:\n                continue\n            if "=" not in raw:\n                raise ValueError(f"--public-url 需要 BUCKET=URL 格式，收到: {raw}")\n            b, u = raw.split("=", 1)\n            pairs.append((b, u))\n\n    for bucket, url in pairs:\n        b = bucket.strip()\n        u = url.strip().rstrip("/")\n        if not b or "/" in b:\n            raise ValueError(f"无效的 bucket 名称: {bucket!r}")\n        parsed = urllib.parse.urlsplit(u)\n        host = (parsed.hostname or "").lower()\n        if parsed.scheme not in ("http", "https") or not host or parsed.username or parsed.password:\n            raise ValueError(f"CDN public-url 必须是无内嵌凭据的 HTTP(S) URL: {u!r}")\n        if parsed.scheme == "http" and host not in _LOOPBACK_HOSTS:\n            raise ValueError(f"非本地 CDN public-url 必须使用 https://: {u!r}")\n        mapping[b] = u\n    return mapping\n\n\n@dataclasses.dataclass\nclass R2Credentials:\n    access_key_id: str\n    secret_access_key: str\n    account_id: str = ""\n    endpoint_url: str = ""\n    jurisdiction: str = ""  # "", "eu", "fedramp"\n    region: str = "auto"\n\n    def resolve_endpoint_url(self) -> str:\n        raw = (self.endpoint_url or "").strip()\n        if raw:\n            parsed = urllib.parse.urlsplit(raw)\n            host = (parsed.hostname or "").lower()\n            if parsed.scheme not in ("http", "https") or not host or parsed.username or parsed.password:\n                raise ValueError(f"无效的 R2 endpoint URL: {raw!r}")\n            if parsed.query or parsed.fragment:\n                raise ValueError("R2 endpoint URL 不应包含 query 或 fragment")\n            if parsed.scheme == "http" and host not in _LOOPBACK_HOSTS:\n                raise ValueError("非本地 R2 endpoint 必须使用 https://")\n            return f"{parsed.scheme.lower()}://{parsed.netloc}{parsed.path.rstrip(\'/\')}"\n\n        acct = (self.account_id or "").strip()\n        if not _ACCOUNT_ID_RE.fullmatch(acct):\n            raise ValueError("请提供有效的 R2 account_id 或 --endpoint-url")\n        jur = (self.jurisdiction or "").strip().lower()\n        if jur not in ("", "eu", "fedramp"):\n            raise ValueError(f"不支持的 R2 jurisdiction: {jur!r} (可选: eu, fedramp)")\n        jur_part = f"{jur}." if jur else ""\n        return f"https://{acct}.{jur_part}r2.cloudflarestorage.com"\n\n    def validate(self) -> None:\n        if not (self.access_key_id or "").strip():\n            raise ValueError("缺少 R2 access_key_id")\n        if not (self.secret_access_key or "").strip():\n            raise ValueError("缺少 R2 secret_access_key")\n        self.resolve_endpoint_url()\n\n    def scope_key(self) -> str:\n        return self.resolve_endpoint_url()\n\n    def to_dict(self) -> dict[str, str]:\n        return {\n            "account_id": (self.account_id or "").strip(),\n            "endpoint_url": (self.endpoint_url or "").strip(),\n            "jurisdiction": (self.jurisdiction or "").strip().lower(),\n            "access_key_id": (self.access_key_id or "").strip(),\n            "secret_access_key": (self.secret_access_key or "").strip(),\n            "region": (self.region or "auto").strip() or "auto",\n        }\n\n    def redacted_summary(self, *, source: str = "configured") -> dict[str, Any]:\n        endpoint = ""\n        try:\n            endpoint = self.resolve_endpoint_url()\n        except ValueError:\n            endpoint = (self.endpoint_url or "").strip()\n        return {\n            "configured": bool((self.access_key_id or "").strip() and (self.secret_access_key or "").strip()),\n            "source": source,\n            "account_id": (self.account_id or "").strip(),\n            "endpoint_url": endpoint,\n            "jurisdiction": (self.jurisdiction or "").strip().lower(),\n            "access_key_id_masked": mask_secret(self.access_key_id),\n            "region": (self.region or "auto").strip() or "auto",\n        }\n\n    @classmethod\n    def from_dict(cls, data: dict[str, Any]) -> "R2Credentials":\n        return cls(\n            access_key_id=str(data.get("access_key_id") or "").strip(),\n            secret_access_key=str(data.get("secret_access_key") or "").strip(),\n            account_id=str(data.get("account_id") or "").strip(),\n            endpoint_url=str(data.get("endpoint_url") or "").strip(),\n            jurisdiction=str(data.get("jurisdiction") or "").strip().lower(),\n            region=str(data.get("region") or "auto").strip() or "auto",\n        )\n\n\ndef _hmac_sha256(key: bytes, msg: str) -> bytes:\n    return hmac.new(key, msg.encode("utf-8"), hashlib.sha256).digest()\n\n\ndef sign_s3_request(\n    creds: R2Credentials,\n    method: str,\n    host_header: str,\n    canonical_uri: str,\n    query_params: Sequence[tuple[str, str]] = (),\n    headers: dict[str, str] | None = None,\n    payload: bytes = b"",\n    *,\n    now: dt.datetime | None = None,\n) -> tuple[str, dict[str, str]]:\n    """Sign an HTTP request for S3 / Cloudflare R2 with AWS Signature Version 4.\n\n    Returns `(request_target, signed_headers_dict)`.\n    """\n    stamp = now or dt.datetime.now(dt.timezone.utc)\n    amz_date = stamp.strftime("%Y%m%dT%H%M%SZ")\n    date_stamp = stamp.strftime("%Y%m%d")\n    region = (creds.region or "auto").strip() or "auto"\n    service = "s3"\n\n    payload_hash = hashlib.sha256(payload).hexdigest()\n    hdrs: dict[str, str] = {}\n    for k, v in (headers or {}).items():\n        hdrs[k.lower().strip()] = " ".join(str(v).strip().split())\n    hdrs["host"] = host_header\n    hdrs["x-amz-date"] = amz_date\n    hdrs["x-amz-content-sha256"] = payload_hash\n\n    encoded_params = sorted((s3_quote(k), s3_quote(v)) for k, v in query_params)\n    canonical_query = "&".join(f"{ek}={ev}" for ek, ev in encoded_params)\n\n    signed_names = sorted(hdrs.keys())\n    canonical_headers = "".join(f"{name}:{hdrs[name]}\\n" for name in signed_names)\n    signed_headers = ";".join(signed_names)\n\n    canon_uri = canonical_uri if canonical_uri.startswith("/") else "/" + canonical_uri\n    canonical_request = "\\n".join(\n        (\n            method.upper(),\n            canon_uri,\n            canonical_query,\n            canonical_headers,\n            signed_headers,\n            payload_hash,\n        )\n    )\n    credential_scope = f"{date_stamp}/{region}/{service}/aws4_request"\n    string_to_sign = "\\n".join(\n        (\n            "AWS4-HMAC-SHA256",\n            amz_date,\n            credential_scope,\n            hashlib.sha256(canonical_request.encode("utf-8")).hexdigest(),\n        )\n    )\n    k_date = _hmac_sha256(("AWS4" + creds.secret_access_key).encode("utf-8"), date_stamp)\n    k_region = _hmac_sha256(k_date, region)\n    k_service = _hmac_sha256(k_region, service)\n    k_signing = _hmac_sha256(k_service, "aws4_request")\n    signature = hmac.new(k_signing, string_to_sign.encode("utf-8"), hashlib.sha256).hexdigest()\n\n    out_headers = dict(hdrs)\n    out_headers["authorization"] = (\n        f"AWS4-HMAC-SHA256 Credential={creds.access_key_id}/{credential_scope}, "\n        f"SignedHeaders={signed_headers}, Signature={signature}"\n    )\n    request_target = canon_uri + (f"?{canonical_query}" if canonical_query else "")\n    return request_target, out_headers\n\n\ndef _strip_ns(tag: str) -> str:\n    if "}" in tag:\n        return tag.split("}", 1)[1]\n    return tag\n\n\ndef _xml_text(elem: ET.Element, child_name: str, default: str = "") -> str:\n    for child in elem:\n        if _strip_ns(child.tag) == child_name:\n            return child.text or default\n    return default\n\n\ndef _parse_s3_error_xml(raw: bytes) -> tuple[str, str]:\n    if not raw:\n        return "", ""\n    try:\n        root = ET.fromstring(raw)\n    except ET.ParseError:\n        return "", raw.decode("utf-8", errors="replace")[:300]\n    if _strip_ns(root.tag) != "Error":\n        return "", raw.decode("utf-8", errors="replace")[:300]\n    return _xml_text(root, "Code"), _xml_text(root, "Message")\n\n\n@dataclasses.dataclass(frozen=True)\nclass ObjectEntry:\n    key: str\n    size: int\n    etag: str\n    storage_class: str = "STANDARD"\n    last_modified: str = ""\n\n\n@dataclasses.dataclass(frozen=True)\nclass ListPage:\n    objects: list[ObjectEntry]\n    is_truncated: bool\n    next_continuation_token: str = ""\n    last_key: str = ""\n\n\n@dataclasses.dataclass(frozen=True)\nclass DownloadResult:\n    bytes_written: int\n    md5_hex: str\n    via_cdn_cache: bool\n\n\n@dataclasses.dataclass(frozen=True)\nclass DeleteBatchResult:\n    deleted: list[str]\n    errors: dict[str, str]\n\n\nclass R2UsageLedger:\n    """Thread-safe UTC-monthly R2 call-count and estimated USD cost tracker."""\n\n    def __init__(self, data_dir: Path, *, flush_interval: float = 30.0) -> None:\n        self.data_dir = Path(data_dir)\n        self.flush_interval = max(0.2, float(flush_interval))\n        self._lock = threading.Lock()\n        self._month = current_utc_month()\n        self._dirty = False\n        self._last_flush = time.monotonic()\n        self._counters = self._empty_counters()\n        self._load_month_locked(self._month)\n\n    @staticmethod\n    def _empty_counters() -> dict[str, int]:\n        return {\n            "class_a_ops": 0,\n            "ia_class_a_ops": 0,\n            "class_b_ops": 0,\n            "ia_class_b_ops": 0,\n            "ia_retrieval_bytes": 0,\n            "free_delete_ops": 0,\n            "deleted_objects": 0,\n            "cdn_hit_ops": 0,\n            "skipped_existing_ops": 0,\n            "zero_byte_ops": 0,\n            "bytes_transferred": 0,\n        }\n\n    def _path_for_month(self, month: str) -> Path:\n        return self.data_dir / f"r2-usage-{month}.json"\n\n    def _load_month_locked(self, month: str) -> None:\n        self._month = month\n        self._counters = self._empty_counters()\n        path = self._path_for_month(month)\n        if not path.is_file():\n            return\n        try:\n            loaded = json.loads(path.read_text(encoding="utf-8"))\n        except (OSError, ValueError):\n            return\n        for key in self._counters:\n            try:\n                self._counters[key] = max(0, int(loaded.get(key, 0)))\n            except (TypeError, ValueError):\n                pass\n\n    def _roll_month_if_needed_locked(self) -> None:\n        now_month = current_utc_month()\n        if now_month != self._month:\n            if self._dirty:\n                self._write_locked()\n            self._load_month_locked(now_month)\n\n    def _write_locked(self) -> None:\n        self.data_dir.mkdir(parents=True, exist_ok=True)\n        payload = self._snapshot_locked()\n        raw = json.dumps(payload, ensure_ascii=False, indent=2).encode("utf-8")\n        target = self._path_for_month(self._month)\n        if target.is_file():\n            with target.open("r+b") as fh:\n                fh.seek(0)\n                fh.write(raw)\n                fh.truncate()\n                fh.flush()\n        else:\n            tmp = target.with_name(f".{target.name}.{os.getpid()}.{threading.get_ident()}.tmp")\n            try:\n                tmp.write_bytes(raw)\n                os.replace(tmp, target)\n            except Exception:\n                with contextlib.suppress(OSError):\n                    tmp.unlink(missing_ok=True)\n                raise\n        self._dirty = False\n        self._last_flush = time.monotonic()\n\n    def _maybe_flush_locked(self) -> None:\n        if self._dirty and (time.monotonic() - self._last_flush) >= self.flush_interval:\n            try:\n                self._write_locked()\n            except OSError:\n                self._dirty = True\n                self._last_flush = time.monotonic()\n\n    def _snapshot_locked(self) -> dict[str, Any]:\n        c = dict(self._counters)\n        paid_a = max(0, c["class_a_ops"] - FREE_CLASS_A_MONTHLY)\n        paid_b = max(0, c["class_b_ops"] - FREE_CLASS_B_MONTHLY)\n        cost = (\n            (paid_a / 1_000_000.0) * PRICE_CLASS_A_PER_MILLION\n            + (c["ia_class_a_ops"] / 1_000_000.0) * PRICE_IA_CLASS_A_PER_MILLION\n            + (paid_b / 1_000_000.0) * PRICE_CLASS_B_PER_MILLION\n            + (c["ia_class_b_ops"] / 1_000_000.0) * PRICE_IA_CLASS_B_PER_MILLION\n            + (c["ia_retrieval_bytes"] / 1_000_000_000.0) * PRICE_IA_RETRIEVAL_PER_GB\n        )\n        return {\n            "month": self._month,\n            **c,\n            "free_class_a_limit": FREE_CLASS_A_MONTHLY,\n            "free_class_b_limit": FREE_CLASS_B_MONTHLY,\n            "free_class_a_remaining": max(0, FREE_CLASS_A_MONTHLY - c["class_a_ops"]),\n            "free_class_b_remaining": max(0, FREE_CLASS_B_MONTHLY - c["class_b_ops"]),\n            "paid_class_a_ops": paid_a,\n            "paid_class_b_ops": paid_b,\n            "estimated_cost_usd": round(cost, 6),\n        }\n\n    def snapshot(self) -> dict[str, Any]:\n        with self._lock:\n            self._roll_month_if_needed_locked()\n            return self._snapshot_locked()\n\n    def flush(self, *, ignore_errors: bool = False) -> dict[str, Any]:\n        with self._lock:\n            self._roll_month_if_needed_locked()\n            try:\n                self._write_locked()\n            except OSError:\n                if not ignore_errors:\n                    raise\n                self._dirty = True\n                self._last_flush = time.monotonic()\n            return self._snapshot_locked()\n\n    def can_spend_class_a(self, cost_mode: str, max_free_class_a: int = DEFAULT_SAFE_FREE_CLASS_A) -> tuple[bool, bool]:\n        """Return `(allowed, uses_paid_tier)` for a Class A operation (ListBuckets / ListObjectsV2)."""\n        with self._lock:\n            self._roll_month_if_needed_locked()\n            cap = min(FREE_CLASS_A_MONTHLY, max(0, int(max_free_class_a)))\n            cur = self._counters["class_a_ops"]\n            uses_paid = cur >= FREE_CLASS_A_MONTHLY\n            if cost_mode == "free-only":\n                return (cur < cap, cur >= cap)\n            return (True, uses_paid)\n\n    def can_spend_class_b(\n        self,\n        storage_class: str,\n        cost_mode: str,\n        max_free_class_b: int = DEFAULT_SAFE_FREE_CLASS_B,\n    ) -> tuple[bool, bool]:\n        """Return `(allowed, uses_paid_tier)` for a Class B GetObject operation."""\n        with self._lock:\n            self._roll_month_if_needed_locked()\n            sc = (storage_class or "STANDARD").upper()\n            if sc == "STANDARD_IA":\n                # STANDARD_IA Class B + retrieval GB have no free tier on R2.\n                return (cost_mode != "free-only", True)\n            cap = min(FREE_CLASS_B_MONTHLY, max(0, int(max_free_class_b)))\n            cur = self._counters["class_b_ops"]\n            uses_paid = cur >= FREE_CLASS_B_MONTHLY\n            if cost_mode == "free-only":\n                return (cur < cap, cur >= cap)\n            return (True, uses_paid)\n\n    def record_class_a(self, count: int = 1) -> None:\n        with self._lock:\n            self._roll_month_if_needed_locked()\n            self._counters["class_a_ops"] += max(0, int(count))\n            self._dirty = True\n            self._maybe_flush_locked()\n\n    def record_class_b(self, *, storage_class: str = "STANDARD", count: int = 1) -> None:\n        with self._lock:\n            self._roll_month_if_needed_locked()\n            n = max(0, int(count))\n            if (storage_class or "STANDARD").upper() == "STANDARD_IA":\n                self._counters["ia_class_b_ops"] += n\n            else:\n                self._counters["class_b_ops"] += n\n            self._dirty = True\n            self._maybe_flush_locked()\n\n    def record_transfer(self, bytes_count: int, *, storage_class: str = "STANDARD", counted_class_b: bool = False) -> None:\n        with self._lock:\n            self._roll_month_if_needed_locked()\n            b = max(0, int(bytes_count))\n            self._counters["bytes_transferred"] += b\n            if counted_class_b and (storage_class or "STANDARD").upper() == "STANDARD_IA":\n                self._counters["ia_retrieval_bytes"] += b\n            self._dirty = True\n            self._maybe_flush_locked()\n\n    def record_free(\n        self,\n        *,\n        delete_ops: int = 0,\n        deleted_objects: int = 0,\n        cdn_hits: int = 0,\n        skipped_existing: int = 0,\n        zero_byte: int = 0,\n        bytes_transferred: int = 0,\n    ) -> None:\n        with self._lock:\n            self._roll_month_if_needed_locked()\n            self._counters["free_delete_ops"] += max(0, int(delete_ops))\n            self._counters["deleted_objects"] += max(0, int(deleted_objects))\n            self._counters["cdn_hit_ops"] += max(0, int(cdn_hits))\n            self._counters["skipped_existing_ops"] += max(0, int(skipped_existing))\n            self._counters["zero_byte_ops"] += max(0, int(zero_byte))\n            self._counters["bytes_transferred"] += max(0, int(bytes_transferred))\n            self._dirty = True\n            self._maybe_flush_locked()\n\n    def calibrate(self, *, class_a_ops: int | None = None, class_b_ops: int | None = None, reset: bool = False) -> dict[str, Any]:\n        with self._lock:\n            self._roll_month_if_needed_locked()\n            if reset:\n                self._counters = self._empty_counters()\n            if class_a_ops is not None:\n                self._counters["class_a_ops"] = max(0, int(class_a_ops))\n            if class_b_ops is not None:\n                self._counters["class_b_ops"] = max(0, int(class_b_ops))\n            self._write_locked()\n            return self._snapshot_locked()\n\n\nclass R2Client:\n    """Thread-safe, connection-pooled pure-stdlib Cloudflare R2 S3 client."""\n\n    def __init__(\n        self,\n        creds: R2Credentials,\n        *,\n        ledger: R2UsageLedger | None = None,\n        timeout: float = 60.0,\n        max_retries: int = 5,\n    ) -> None:\n        self.creds = creds\n        self.ledger = ledger\n        self.timeout = float(timeout)\n        self.max_retries = max(1, int(max_retries))\n        endpoint = creds.resolve_endpoint_url()\n        parsed = urllib.parse.urlsplit(endpoint)\n        self.scheme = parsed.scheme.lower()\n        self.host = parsed.hostname or ""\n        self.port = parsed.port or (443 if self.scheme == "https" else 80)\n        self.netloc = parsed.netloc\n        self.base_path = parsed.path.rstrip("/")\n        self._ssl_ctx = ssl.create_default_context()\n        self._local = threading.local()\n        self._pool_lock = threading.Lock()\n        self._all_conns: dict[int, dict[tuple[str, str, int], http.client.HTTPConnection]] = {}\n        self._bucket_delete_mode: dict[str, tuple[str, int]] = {}\n\n    def _thread_pool(self) -> dict[tuple[str, str, int], http.client.HTTPConnection]:\n        pool = getattr(self._local, "pool", None)\n        if pool is None:\n            pool = {}\n            self._local.pool = pool\n            with self._pool_lock:\n                self._all_conns[threading.get_ident()] = pool\n        return pool\n\n    def _get_conn(\n        self,\n        scheme: str,\n        host: str,\n        port: int,\n        *,\n        timeout: float | None = None,\n    ) -> http.client.HTTPConnection:\n        pool = self._thread_pool()\n        key = (scheme, host, port)\n        eff_timeout = self.timeout if timeout is None else float(timeout)\n        conn = pool.get(key)\n        if conn is None:\n            if scheme == "https":\n                conn = http.client.HTTPSConnection(host, port, timeout=eff_timeout, context=self._ssl_ctx)\n            else:\n                conn = http.client.HTTPConnection(host, port, timeout=eff_timeout)\n            pool[key] = conn\n        else:\n            conn.timeout = eff_timeout\n            if conn.sock is not None:\n                with contextlib.suppress(OSError):\n                    conn.sock.settimeout(eff_timeout)\n        return conn\n\n    def _drop_conn(self, scheme: str, host: str, port: int) -> None:\n        pool = self._thread_pool()\n        conn = pool.pop((scheme, host, port), None)\n        if conn is not None:\n            with contextlib.suppress(Exception):\n                conn.close()\n\n    def close(self) -> None:\n        with self._pool_lock:\n            pools = list(self._all_conns.values())\n            self._all_conns.clear()\n        for pool in pools:\n            for conn in pool.values():\n                with contextlib.suppress(Exception):\n                    conn.close()\n            pool.clear()\n\n    def _retry_sleep(self, attempt: int, retry_after: str | None = None) -> None:\n        delay: float | None = None\n        if retry_after:\n            try:\n                delay = max(0.05, min(30.0, float(retry_after.strip())))\n            except ValueError:\n                delay = None\n        if delay is None:\n            delay = min(30.0, 0.25 * (2 ** attempt)) + random.uniform(0.0, 0.15)\n        time.sleep(delay)\n\n    def _signed_request(\n        self,\n        method: str,\n        rel_path: str,\n        *,\n        query_params: Sequence[tuple[str, str]] = (),\n        headers: dict[str, str] | None = None,\n        payload: bytes = b"",\n        timeout: float | None = None,\n        max_retries: int | None = None,\n    ) -> tuple[int, http.client.HTTPMessage, bytes]:\n        canon_uri = self.base_path + (rel_path if rel_path.startswith("/") else "/" + rel_path)\n        retries = self.max_retries if max_retries is None else max(1, int(max_retries))\n        last_err: Exception | None = None\n        for attempt in range(retries):\n            req_target, signed_hdrs = sign_s3_request(\n                self.creds,\n                method,\n                self.netloc,\n                canon_uri,\n                query_params=query_params,\n                headers=headers,\n                payload=payload,\n            )\n            try:\n                conn = self._get_conn(self.scheme, self.host, self.port, timeout=timeout)\n                conn.request(method, req_target, body=payload if payload else None, headers=signed_hdrs)\n                resp = conn.getresponse()\n                body = resp.read()\n                if resp.status in _RETRY_STATUSES and attempt + 1 < retries:\n                    ra = resp.getheader("Retry-After")\n                    self._drop_conn(self.scheme, self.host, self.port)\n                    self._retry_sleep(attempt, ra)\n                    continue\n                if resp.status >= 400:\n                    code, msg = _parse_s3_error_xml(body)\n                    raise R2HttpError(\n                        resp.status,\n                        msg or resp.reason or "S3 request failed",\n                        code=code,\n                        body=body.decode("utf-8", errors="replace")[:500],\n                    )\n                return resp.status, resp.headers, body\n            except R2HttpError:\n                raise\n            except (http.client.HTTPException, OSError, TimeoutError, ssl.SSLError) as err:\n                last_err = err\n                self._drop_conn(self.scheme, self.host, self.port)\n                if attempt + 1 >= retries:\n                    raise R2Error(f"R2 S3 request failed after {retries} attempts: {err}") from err\n                self._retry_sleep(attempt)\n        raise R2Error(f"R2 S3 request failed: {last_err}")\n\n    def list_buckets(self) -> list[dict[str, str]]:\n        """List all buckets in the R2 account (1 Class A op)."""\n        if self.ledger is not None:\n            self.ledger.record_class_a(1)\n        _, _, body = self._signed_request("GET", "/")\n        root = ET.fromstring(body)\n        buckets: list[dict[str, str]] = []\n        for elem in root.iter():\n            if _strip_ns(elem.tag) == "Bucket":\n                name = _xml_text(elem, "Name")\n                if name:\n                    buckets.append(\n                        {\n                            "name": name,\n                            "creation_date": _xml_text(elem, "CreationDate"),\n                        }\n                    )\n        return buckets\n\n    def list_objects_v2(\n        self,\n        bucket: str,\n        *,\n        prefix: str = "",\n        start_after: str = "",\n        continuation_token: str = "",\n        max_keys: int = 1000,\n    ) -> ListPage:\n        """Fetch a single flat ListObjectsV2 page (1 Class A op per up to 1,000 objects)."""\n        if self.ledger is not None:\n            self.ledger.record_class_a(1)\n        params: list[tuple[str, str]] = [\n            ("encoding-type", "url"),\n            ("list-type", "2"),\n            ("max-keys", str(max(1, min(1000, int(max_keys))))),\n        ]\n        if prefix:\n            params.append(("prefix", prefix))\n        if continuation_token:\n            params.append(("continuation-token", continuation_token))\n        elif start_after:\n            params.append(("start-after", start_after))\n\n        _, _, body = self._signed_request("GET", f"/{s3_quote(bucket)}", query_params=params)\n        root = ET.fromstring(body)\n        enc_type = _xml_text(root, "EncodingType").lower()\n        decode = urllib.parse.unquote_plus if enc_type == "url" else (lambda s: s)\n        is_truncated = _xml_text(root, "IsTruncated").lower() == "true"\n        next_token = _xml_text(root, "NextContinuationToken")\n\n        entries: list[ObjectEntry] = []\n        for child in root:\n            if _strip_ns(child.tag) != "Contents":\n                continue\n            raw_key = _xml_text(child, "Key")\n            key = decode(raw_key)\n            size = int(_xml_text(child, "Size", "0") or "0")\n            etag = _xml_text(child, "ETag").strip(\'"\').strip()\n            sc = (_xml_text(child, "StorageClass", "STANDARD") or "STANDARD").upper()\n            lm = _xml_text(child, "LastModified")\n            entries.append(\n                ObjectEntry(key=key, size=size, etag=etag, storage_class=sc, last_modified=lm)\n            )\n        last_key = entries[-1].key if entries else start_after\n        return ListPage(\n            objects=entries,\n            is_truncated=is_truncated,\n            next_continuation_token=next_token,\n            last_key=last_key,\n        )\n\n    def _stream_to_temp(\n        self,\n        conn: http.client.HTTPConnection,\n        method_url: str,\n        headers: dict[str, str],\n        dest_temp: Path,\n        *,\n        expected_size: int | None,\n        expected_etag: str,\n        chunk_size: int,\n    ) -> tuple[int, str, http.client.HTTPMessage]:\n        conn.request("GET", method_url, headers=headers)\n        resp = conn.getresponse()\n        if resp.status in _RETRY_STATUSES:\n            ra = resp.getheader("Retry-After")\n            resp.read()\n            raise _RetryableHttp(resp.status, ra)\n        if resp.status != 200:\n            body = resp.read(65536)\n            code, msg = _parse_s3_error_xml(body)\n            raise R2HttpError(\n                resp.status,\n                msg or resp.reason or "GET failed",\n                code=code,\n                body=body[:2048].decode("utf-8", errors="replace"),\n            )\n        md5 = hashlib.md5()\n        total = 0\n        with dest_temp.open("wb") as out:\n            while True:\n                chunk = resp.read(chunk_size)\n                if not chunk:\n                    break\n                out.write(chunk)\n                md5.update(chunk)\n                total += len(chunk)\n        md5_hex = md5.hexdigest()\n        if expected_size is not None and total < expected_size:\n            raise R2TruncatedReadError(f"Size mismatch: expected {expected_size}, got {total}")\n        if expected_size is not None and total != expected_size:\n            raise R2IntegrityError(f"Size mismatch: expected {expected_size}, got {total}")\n        etag = (expected_etag or "").strip(\'"\').strip()\n        if etag and _HEX32_RE.fullmatch(etag) and md5_hex.lower() != etag.lower():\n            raise R2IntegrityError(f"MD5 ETag mismatch: expected {etag.lower()}, got {md5_hex}")\n        return total, md5_hex, resp.headers\n\n    def _try_download_via_cdn(\n        self,\n        public_base_url: str,\n        key: str,\n        dest_temp: Path,\n        *,\n        expected_size: int | None,\n        expected_etag: str,\n        storage_class: str,\n        chunk_size: int,\n    ) -> DownloadResult | None:\n        base = public_base_url.rstrip("/")\n        parsed = urllib.parse.urlsplit(base)\n        scheme = parsed.scheme.lower()\n        host = parsed.hostname or ""\n        port = parsed.port or (443 if scheme == "https" else 80)\n        req_path = parsed.path.rstrip("/") + "/" + s3_quote_path(key).lstrip("/")\n        for attempt in range(2):\n            try:\n                conn = self._get_conn(scheme, host, port)\n                total, md5_hex, resp_headers = self._stream_to_temp(\n                    conn,\n                    req_path,\n                    {"Host": parsed.netloc, "Accept-Encoding": "identity"},\n                    dest_temp,\n                    expected_size=expected_size,\n                    expected_etag=expected_etag,\n                    chunk_size=chunk_size,\n                )\n                cf_cache = (resp_headers.get("CF-Cache-Status") or "").strip().upper()\n                via_hit = cf_cache in _CF_CACHE_HIT_STATUSES\n                if self.ledger is not None:\n                    if via_hit:\n                        self.ledger.record_free(cdn_hits=1, bytes_transferred=total)\n                    else:\n                        self.ledger.record_class_b(storage_class=storage_class, count=1)\n                        self.ledger.record_transfer(total, storage_class=storage_class, counted_class_b=True)\n                return DownloadResult(bytes_written=total, md5_hex=md5_hex, via_cdn_cache=via_hit)\n            except _RetryableHttp:\n                self._drop_conn(scheme, host, port)\n                if attempt == 0:\n                    time.sleep(0.15)\n                    continue\n                return None\n            except (R2HttpError, R2IntegrityError, http.client.HTTPException, OSError, TimeoutError, ssl.SSLError):\n                self._drop_conn(scheme, host, port)\n                with contextlib.suppress(OSError):\n                    dest_temp.unlink(missing_ok=True)\n                return None\n        return None\n\n    def download_object_to_file(\n        self,\n        bucket: str,\n        key: str,\n        dest_temp: Path,\n        *,\n        expected_size: int | None = None,\n        expected_etag: str = "",\n        storage_class: str = "STANDARD",\n        public_base_url: str | None = None,\n        chunk_size: int = 256 * 1024,\n    ) -> DownloadResult:\n        """Download an object cleanly with CDN cache-first optimization + SigV4 fallback and MD5 verification."""\n        if public_base_url:\n            cdn_res = self._try_download_via_cdn(\n                public_base_url,\n                key,\n                dest_temp,\n                expected_size=expected_size,\n                expected_etag=expected_etag,\n                storage_class=storage_class,\n                chunk_size=chunk_size,\n            )\n            if cdn_res is not None:\n                return cdn_res\n\n        rel_path = f"/{s3_quote(bucket)}/{s3_quote_path(key)}"\n        canon_uri = self.base_path + rel_path\n        last_err: Exception | None = None\n        for attempt in range(self.max_retries):\n            if self.ledger is not None:\n                self.ledger.record_class_b(storage_class=storage_class, count=1)\n            req_target, signed_hdrs = sign_s3_request(\n                self.creds,\n                "GET",\n                self.netloc,\n                canon_uri,\n            )\n            try:\n                conn = self._get_conn(self.scheme, self.host, self.port)\n                total, md5_hex, _ = self._stream_to_temp(\n                    conn,\n                    req_target,\n                    signed_hdrs,\n                    dest_temp,\n                    expected_size=expected_size,\n                    expected_etag=expected_etag,\n                    chunk_size=chunk_size,\n                )\n                if self.ledger is not None:\n                    self.ledger.record_transfer(total, storage_class=storage_class, counted_class_b=True)\n                return DownloadResult(bytes_written=total, md5_hex=md5_hex, via_cdn_cache=False)\n            except _RetryableHttp as retry_exc:\n                self._drop_conn(self.scheme, self.host, self.port)\n                last_err = R2HttpError(retry_exc.status, "Retryable upstream status")\n                if attempt + 1 >= self.max_retries:\n                    raise last_err from None\n                self._retry_sleep(attempt, retry_exc.retry_after)\n            except R2TruncatedReadError as err:\n                last_err = err\n                self._drop_conn(self.scheme, self.host, self.port)\n                with contextlib.suppress(OSError):\n                    dest_temp.unlink(missing_ok=True)\n                if attempt + 1 >= self.max_retries:\n                    raise\n                self._retry_sleep(attempt)\n            except R2IntegrityError:\n                self._drop_conn(self.scheme, self.host, self.port)\n                with contextlib.suppress(OSError):\n                    dest_temp.unlink(missing_ok=True)\n                raise\n            except R2HttpError:\n                self._drop_conn(self.scheme, self.host, self.port)\n                with contextlib.suppress(OSError):\n                    dest_temp.unlink(missing_ok=True)\n                raise\n            except (http.client.HTTPException, OSError, TimeoutError, ssl.SSLError) as err:\n                last_err = err\n                self._drop_conn(self.scheme, self.host, self.port)\n                with contextlib.suppress(OSError):\n                    dest_temp.unlink(missing_ok=True)\n                if attempt + 1 >= self.max_retries:\n                    raise R2Error(f"Download failed for {bucket}/{key}: {err}") from err\n                self._retry_sleep(attempt)\n        raise R2Error(f"Download failed for {bucket}/{key}: {last_err}")\n\n    def delete_object(\n        self,\n        bucket: str,\n        key: str,\n        *,\n        timeout: float | None = None,\n        max_retries: int | None = None,\n    ) -> None:\n        """Delete a single object (Free $0.00 operation on R2)."""\n        if self.ledger is not None:\n            self.ledger.record_free(delete_ops=1)\n        rel_path = f"/{s3_quote(bucket)}/{s3_quote_path(key)}"\n        try:\n            self._signed_request("DELETE", rel_path, timeout=timeout, max_retries=max_retries)\n        except R2HttpError as err:\n            if err.status == 404 or err.code == "NoSuchKey":\n                pass\n            else:\n                raise\n        if self.ledger is not None:\n            self.ledger.record_free(deleted_objects=1)\n\n    def _delete_xml_batch(\n        self,\n        bucket: str,\n        batch: Sequence[str],\n        *,\n        quiet: bool = True,\n        timeout: float | None = None,\n        max_retries: int | None = None,\n    ) -> tuple[list[str], dict[str, str]]:\n        """Execute one Free $0.00 S3 DeleteObjects XML POST request."""\n        if not batch:\n            return [], {}\n        quiet_tag = "true" if quiet else "false"\n        body_parts = [f"<Delete><Quiet>{quiet_tag}</Quiet>"]\n        for k in batch:\n            body_parts.append(f"<Object><Key>{xml_escape(k)}</Key></Object>")\n        body_parts.append("</Delete>")\n        xml_bytes = "".join(body_parts).encode("utf-8")\n        md5_b64 = base64.b64encode(hashlib.md5(xml_bytes).digest()).decode("ascii")\n        if self.ledger is not None:\n            self.ledger.record_free(delete_ops=1)\n        _, _, resp_body = self._signed_request(\n            "POST",\n            f"/{s3_quote(bucket)}",\n            query_params=[("delete", "")],\n            headers={\n                "content-md5": md5_b64,\n                "content-type": "application/xml",\n            },\n            payload=xml_bytes,\n            timeout=timeout,\n            max_retries=max_retries,\n        )\n        batch_deleted: list[str] = []\n        batch_errors: dict[str, str] = {}\n        try:\n            root = ET.fromstring(resp_body)\n            for child in root:\n                name = _strip_ns(child.tag)\n                if name == "Deleted":\n                    k = _xml_text(child, "Key")\n                    if k:\n                        batch_deleted.append(k)\n                elif name == "Error":\n                    k = _xml_text(child, "Key")\n                    code = _xml_text(child, "Code")\n                    msg = _xml_text(child, "Message")\n                    if k:\n                        batch_errors[k] = f"{code}: {msg}".strip(": ") or "Delete error"\n        except ET.ParseError as parse_err:\n            return [], {k: f"Invalid DeleteObjects response XML: {parse_err}" for k in batch}\n\n        if quiet and not batch_deleted:\n            batch_deleted = [k for k in batch if k not in batch_errors]\n        else:\n            accounted = set(batch_deleted) | set(batch_errors.keys())\n            if not batch_deleted and not batch_errors:\n                batch_deleted = list(batch)\n            elif len(accounted) < len(batch):\n                for k in batch:\n                    if k not in accounted:\n                        batch_errors[k] = "Missing from DeleteObjects result"\n        if self.ledger is not None and batch_deleted:\n            self.ledger.record_free(deleted_objects=len(batch_deleted))\n        return batch_deleted, batch_errors\n\n    def delete_objects(\n        self,\n        bucket: str,\n        keys: Sequence[str],\n        *,\n        batch_size: int = 250,\n        quiet: bool = True,\n        concurrency: int = 1,\n        log: Callable[[str], None] | None = None,\n    ) -> DeleteBatchResult:\n        """Delete keys via Free $0.00 R2 DeleteObjects with adaptive fallback to smaller batches / single DELETE."""\n        deleted: list[str] = []\n        errors: dict[str, str] = {}\n        xml_safe: list[str] = []\n        for k in keys:\n            if _XML_ILLEGAL_CTRL_RE.search(k):\n                try:\n                    self.delete_object(bucket, k, timeout=min(self.timeout, 25.0), max_retries=2)\n                    deleted.append(k)\n                except Exception as err:\n                    errors[k] = str(err)\n            else:\n                xml_safe.append(k)\n\n        if not xml_safe:\n            return DeleteBatchResult(deleted=deleted, errors=errors)\n\n        req_batch = max(1, min(1000, int(batch_size)))\n        conc = max(1, int(concurrency))\n\n        def _is_non_retryable_http(err: Exception) -> bool:\n            return isinstance(err, R2HttpError) and err.status < 500 and err.status not in (408, 429)\n\n        def _delete_via_single(sub_keys: Sequence[str]) -> tuple[list[str], dict[str, str]]:\n            if not sub_keys:\n                return [], {}\n            probe_timeout = min(self.timeout, 20.0)\n            self.delete_object(bucket, sub_keys[0], timeout=probe_timeout, max_retries=2)\n            ok_keys: list[str] = [sub_keys[0]]\n            err_map: dict[str, str] = {}\n            if len(sub_keys) == 1:\n                return ok_keys, err_map\n\n            rest = sub_keys[1:]\n            max_w = min(len(rest), max(8, conc * 2))\n            last_r2_err: R2Error | None = None\n\n            def _del_one(one_key: str) -> tuple[str, bool, str, R2Error | None]:\n                try:\n                    self.delete_object(bucket, one_key, timeout=probe_timeout, max_retries=2)\n                    return one_key, True, "", None\n                except R2HttpError as http_err:\n                    if _is_non_retryable_http(http_err):\n                        return one_key, False, str(http_err), None\n                    return one_key, False, str(http_err), http_err\n                except R2Error as r2_err:\n                    return one_key, False, str(r2_err), r2_err\n\n            with concurrent.futures.ThreadPoolExecutor(max_workers=max_w) as pool:\n                for one_key, ok, msg, r2_err in pool.map(_del_one, rest):\n                    if ok:\n                        ok_keys.append(one_key)\n                    else:\n                        err_map[one_key] = msg\n                        if r2_err is not None:\n                            last_r2_err = r2_err\n            if len(ok_keys) == 1 and len(err_map) == len(rest) and last_r2_err is not None:\n                raise last_r2_err\n            return ok_keys, err_map\n\n        def _process_chunk(chunk: list[str]) -> tuple[list[str], dict[str, str]]:\n            if not chunk:\n                return [], {}\n            with self._pool_lock:\n                cur_mode = self._bucket_delete_mode.get(bucket, ("batch", req_batch))[0]\n            if cur_mode == "single":\n                return _delete_via_single(chunk)\n\n            try:\n                res_ok = self._delete_xml_batch(\n                    bucket,\n                    chunk,\n                    quiet=quiet,\n                    timeout=min(self.timeout, 35.0),\n                    max_retries=2,\n                )\n                with self._pool_lock:\n                    if bucket not in self._bucket_delete_mode:\n                        self._bucket_delete_mode[bucket] = ("batch", req_batch)\n                return res_ok\n            except R2Error as err:\n                if _is_non_retryable_http(err):\n                    raise\n                d_list: list[str] = []\n                e_dict: dict[str, str] = {}\n                remaining_keys: Sequence[str] = chunk\n                if len(chunk) > 15:\n                    with self._pool_lock:\n                        self._bucket_delete_mode[bucket] = ("batch", 15)\n                    if log is not None:\n                        log(f"R2 DeleteObjects on {bucket} timed out at batch={len(chunk)}; adapting to 15-key batches ({err})")\n                    fallback_from: int | None = None\n                    for idx in range(0, len(chunk), 15):\n                        sub = chunk[idx : idx + 15]\n                        try:\n                            sd, se = self._delete_xml_batch(\n                                bucket,\n                                sub,\n                                quiet=quiet,\n                                timeout=min(self.timeout, 25.0),\n                                max_retries=2,\n                            )\n                            d_list.extend(sd)\n                            e_dict.update(se)\n                        except R2Error as err2:\n                            if _is_non_retryable_http(err2):\n                                raise\n                            fallback_from = idx\n                            break\n                    if fallback_from is None:\n                        return d_list, e_dict\n                    remaining_keys = chunk[fallback_from:]\n\n                if log is not None:\n                    log(f"R2 POST ?delete on {bucket} failed at <=15 keys; probing single DELETE fallback")\n                d_rest, e_rest = _delete_via_single(remaining_keys)\n                with self._pool_lock:\n                    self._bucket_delete_mode[bucket] = ("single", 1)\n                return d_list + d_rest, {**e_dict, **e_rest}\n\n        with self._pool_lock:\n            calibrated = bucket in self._bucket_delete_mode\n            mode, eff_batch = self._bucket_delete_mode.get(bucket, ("batch", req_batch))\n\n        if mode == "single":\n            d0, e0 = _delete_via_single(xml_safe)\n            deleted.extend(d0)\n            errors.update(e0)\n            return DeleteBatchResult(deleted=deleted, errors=errors)\n\n        eff_batch = max(1, min(1000, int(eff_batch)))\n        if not calibrated:\n            first_chunk = xml_safe[:eff_batch]\n            d_first, e_first = _process_chunk(first_chunk)\n            deleted.extend(d_first)\n            errors.update(e_first)\n\n            rest_keys = xml_safe[len(first_chunk) :]\n            if not rest_keys:\n                return DeleteBatchResult(deleted=deleted, errors=errors)\n\n            with self._pool_lock:\n                mode, new_batch = self._bucket_delete_mode.get(bucket, ("batch", req_batch))\n            if mode == "single":\n                d_rest, e_rest = _delete_via_single(rest_keys)\n                deleted.extend(d_rest)\n                errors.update(e_rest)\n                return DeleteBatchResult(deleted=deleted, errors=errors)\n            eff_batch = max(1, min(1000, int(new_batch)))\n        else:\n            rest_keys = xml_safe\n\n        rem_chunks = [rest_keys[i : i + eff_batch] for i in range(0, len(rest_keys), eff_batch)]\n        if conc <= 1 or len(rem_chunks) <= 1:\n            for ch in rem_chunks:\n                dc, ec = _process_chunk(ch)\n                deleted.extend(dc)\n                errors.update(ec)\n        else:\n            with concurrent.futures.ThreadPoolExecutor(max_workers=min(conc, len(rem_chunks))) as pool:\n                for dc, ec in pool.map(_process_chunk, rem_chunks):\n                    deleted.extend(dc)\n                    errors.update(ec)\n\n        return DeleteBatchResult(deleted=deleted, errors=errors)\n\n\nclass _RetryableHttp(Exception):\n    def __init__(self, status: int, retry_after: str | None = None) -> None:\n        self.status = status\n        self.retry_after = retry_after\n\n\n# --------------------------------------------------------------------------- safe Drive path mapping\n\n\ndef sanitize_segment(segment: str) -> str:\n    """Convert an arbitrary S3 key path component into a safe, contained POSIX segment."""\n    clean = segment.replace("\\x00", "%00")\n    if clean == "":\n        return "__r2_empty__"\n    if clean == ".":\n        return "__r2_dot__"\n    if clean == "..":\n        return "__r2_dotdot__"\n    return clean\n\n\ndef resolve_drive_output_root(\n    drive_mount: Path,\n    output_dir: str | None,\n    *,\n    default_subdir: str = "r2-backup",\n    fallback_root: Path | None = None,\n) -> Path:\n    """Resolve and strictly contain an output directory inside MyDrive (or local fallback)."""\n    mount = drive_mount if drive_mount.is_dir() else (fallback_root or drive_mount)\n    mount_resolved = mount.resolve()\n    raw = (output_dir or "").strip()\n    if raw.startswith("drive://"):\n        raw = raw[len("drive://") :]\n    elif raw.startswith("/content/drive/MyDrive/"):\n        raw = raw[len("/content/drive/MyDrive/") :]\n    elif raw == "/content/drive/MyDrive":\n        raw = ""\n    elif raw.startswith("MyDrive/"):\n        raw = raw[len("MyDrive/") :]\n    elif raw == "MyDrive":\n        raw = ""\n    raw = raw.lstrip("/")\n    if not raw:\n        raw = default_subdir\n    if "\\x00" in raw:\n        raise ValueError("输出目录路径包含非法空字节")\n    candidate = (mount_resolved / raw).resolve()\n    if not candidate.is_relative_to(mount_resolved):\n        raise ValueError(f"输出目录超出允许根目录 {mount_resolved}: {output_dir!r}")\n    return candidate\n\n\ndef map_object_to_drive_relpath(\n    bucket: str,\n    key: str,\n    *,\n    size: int,\n    include_bucket: bool = True,\n) -> tuple[str, bool]:\n    """Map `(bucket, key, size)` to `(relative_posix_path, is_dir_marker)` safely."""\n    safe_bucket = sanitize_segment(bucket.strip("/"))\n    if key == "":\n        parts = ["__r2_empty__"]\n        is_dir = False\n    elif key.endswith("/"):\n        trimmed = key[:-1]\n        raw_parts = trimmed.split("/") if trimmed else ["__r2_empty__"]\n        parts = [sanitize_segment(p) for p in raw_parts]\n        if int(size) == 0:\n            is_dir = True\n        else:\n            parts.append("__r2_slash_object__")\n            is_dir = False\n    else:\n        parts = [sanitize_segment(p) for p in key.split("/")]\n        is_dir = False\n\n    if include_bucket:\n        parts = [safe_bucket, *parts]\n    return "/".join(parts), is_dir\n', encoding="utf-8")
(R2_SRC / 'r2_migrate_engine.py').write_text('"""Resumable SQLite WAL migration engine for Cloudflare R2 -> Google Drive."""\nfrom __future__ import annotations\n\nfrom collections import OrderedDict, deque\nimport concurrent.futures\nimport contextlib\nimport dataclasses\nimport errno\nimport hashlib\nimport io\nimport json\nimport os\nfrom pathlib import Path\nimport queue\nimport re\nimport secrets\nimport shutil\nimport sqlite3\nimport sys\nimport tarfile\nimport threading\nimport time\nfrom typing import Any, Callable\nimport urllib.error\nimport urllib.parse\nimport urllib.request\nimport uuid\n\nfrom r2_s3 import (\n    DEFAULT_SAFE_FREE_CLASS_A,\n    DEFAULT_SAFE_FREE_CLASS_B,\n    DeleteBatchResult,\n    R2Client,\n    R2Credentials,\n    R2Error,\n    R2HttpError,\n    R2UsageLedger,\n    map_object_to_drive_relpath,\n    parse_public_urls,\n    resolve_drive_output_root,\n    sanitize_segment,\n)\n\nCOST_MODES = ("free-first", "free-only", "unlimited")\nSTORAGE_MODES = ("files", "tar")\nTERMINAL_JOB_STATUSES = {\n    "completed",\n    "cancelled",\n    "failed",\n    "paused",\n    "paused_budget",\n    "paused_quota",\n    "paused_credentials",\n    "interrupted",\n}\nRESUMABLE_JOB_STATUSES = {\n    "queued",\n    "running",\n    "paused",\n    "paused_budget",\n    "paused_quota",\n    "paused_credentials",\n    "interrupted",\n    "failed",\n}\nDEFAULT_WORKERS = 48\nDEFAULT_WRITER_WORKERS = 16\nDEFAULT_TAR_SHARD_MB = 512\nDEFAULT_SPOOL_BYTES = 4 * 1024 * 1024 * 1024\nDEFAULT_CHECKPOINT_INTERVAL = 300.0\nTAR_DEFAULT_CHECKPOINT_INTERVAL = 900.0\n_MIN_DRIVEFS_WORK_FREE_BYTES = 6 * 1024 * 1024 * 1024\nDEFAULT_PAGE_SIZE = 1000\nMAX_LISTED_BUFFER = 250_000\n_LARGE_FILE_FSYNC_BYTES = 16 * 1024 * 1024\n_PREFETCH_MEM_MAX_FILE = 8 * 1024 * 1024\n_PREFETCH_BUDGET_BYTES = 256 * 1024 * 1024\n_DIR_ENTRY_CAP = 250_000\n\n\ndef _atomic_write_json(path: Path, payload: dict[str, Any], *, mode: int | None = None) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    raw = json.dumps(payload, ensure_ascii=False, indent=2).encode("utf-8")\n    if mode is None and path.is_file():\n        with path.open("r+b") as fh:\n            fh.seek(0)\n            fh.write(raw)\n            fh.truncate()\n            fh.flush()\n        return\n    tmp = path.with_name(f".{path.name}.{os.getpid()}.{threading.get_ident()}.tmp")\n    try:\n        tmp.write_bytes(raw)\n        if mode is not None:\n            os.chmod(tmp, mode)\n        os.replace(tmp, path)\n        if mode is not None:\n            with contextlib.suppress(OSError):\n                os.chmod(path, mode)\n    except Exception:\n        with contextlib.suppress(OSError):\n            tmp.unlink(missing_ok=True)\n        raise\n\n\nclass ByteBudgetSemaphore:\n    """Bounded byte-budget allocator for NVMe spool staging that never deadlocks on oversized files."""\n\n    def __init__(self, limit_bytes: int) -> None:\n        self.limit = max(64 * 1024 * 1024, int(limit_bytes))\n        self._used = 0\n        self._cond = threading.Condition()\n\n    def acquire(self, size_bytes: int, stop_event: threading.Event | None = None) -> int:\n        need = min(self.limit, max(1, int(size_bytes)))\n        with self._cond:\n            while self._used + need > self.limit:\n                if stop_event is not None and stop_event.is_set():\n                    raise _JobStopped("stopped")\n                self._cond.wait(timeout=0.5)\n            self._used += need\n            return need\n\n    def release(self, amount: int) -> None:\n        with self._cond:\n            self._used = max(0, self._used - max(0, int(amount)))\n            self._cond.notify_all()\n\n\nclass _JobStopped(Exception):\n    pass\n\n\nclass _BudgetPaused(Exception):\n    def __init__(self, message: str) -> None:\n        super().__init__(message)\n        self.message = message\n\n\nclass _QuotaPaused(Exception):\n    def __init__(self, message: str) -> None:\n        super().__init__(message)\n        self.message = message\n\n\ndef _detect_fuse_drive_mydrive(path: Path) -> Path | None:\n    """Return resolved `<mount>/MyDrive` if `path` resides on a `fuse.drive` mount, else None."""\n    try:\n        resolved = Path(path).resolve()\n    except OSError:\n        resolved = Path(path)\n    try:\n        mounts_text = Path("/proc/mounts").read_text(encoding="utf-8", errors="replace")\n    except OSError:\n        return None\n    for line in mounts_text.splitlines():\n        parts = line.split()\n        if len(parts) < 3:\n            continue\n        mnt_point = parts[1].replace("\\\\040", " ")\n        fstype = parts[2]\n        if fstype != "fuse.drive":\n            continue\n        for candidate_name in ("MyDrive", "My Drive"):\n            mydrive = Path(mnt_point) / candidate_name\n            try:\n                mydrive_res = mydrive.resolve()\n            except OSError:\n                mydrive_res = mydrive\n            if resolved == mydrive_res or resolved.is_relative_to(mydrive_res):\n                return mydrive_res\n    return None\n\n\ndef _gdrive_escape_q(val: str) -> str:\n    return val.replace("\\\\", "\\\\\\\\").replace("\'", "\\\\\'")\n\n\nclass _DriveCloudSyncVerifier:\n    """Verifies whether files on a `fuse.drive` mount have completed upload to Google Drive Cloud."""\n\n    _TOKEN_TTL_SKEW = 60.0\n    _LIST_CACHE_TTL = 30.0\n    _FILE_CACHE_TTL = 30.0\n\n    def __init__(self, dest_root: Path, log: Callable[[str], None] | None = None) -> None:\n        self.dest_root = Path(dest_root)\n        self._log = log or (lambda _msg: None)\n        self._mydrive_root = _detect_fuse_drive_mydrive(self.dest_root)\n        self._lock = threading.Lock()\n        self._access_token: str = ""\n        self._token_expires_mono: float = 0.0\n        self._folder_ids: dict[str, str] = {"": "root"}\n        self._dir_cache: dict[str, tuple[float, dict[str, list[tuple[int, bool]]]]] = {}\n        self._file_cache: dict[tuple[Path, int], tuple[float, str]] = {}\n\n    @property\n    def is_fuse_drive(self) -> bool:\n        return self._mydrive_root is not None\n\n    def invalidate_cache(self) -> None:\n        with self._lock:\n            self._dir_cache.clear()\n            self._file_cache.clear()\n\n    def _get_access_token_locked(self) -> str:\n        now = time.monotonic()\n        if self._access_token and now < self._token_expires_mono:\n            return self._access_token\n        candidates = ["127.0.0.1:8012"]\n        for env_key in ("TBE_EPHEM_CREDS_ADDR", "TBE_CREDS_ADDR"):\n            addr = (os.environ.get(env_key) or "").strip()\n            if addr and addr not in candidates:\n                candidates.append(addr)\n        last_err: Exception | None = None\n        for host_port in candidates:\n            url = f"http://{host_port}/computeMetadata/v1/instance/service-accounts/default/token"\n            req = urllib.request.Request(url, headers={"Metadata-Flavor": "Google"})\n            try:\n                with urllib.request.urlopen(req, timeout=5.0) as resp:\n                    data = json.loads(resp.read().decode("utf-8"))\n                tok = str(data.get("access_token") or "")\n                if not tok:\n                    raise RuntimeError("empty access_token")\n                expires_in = max(120, int(data.get("expires_in") or 3599))\n                self._access_token = tok\n                self._token_expires_mono = time.monotonic() + max(30.0, expires_in - self._TOKEN_TTL_SKEW)\n                return tok\n            except Exception as err:\n                last_err = err\n        raise RuntimeError(f"Drive token shim unreachable: {last_err}")\n\n    def _gdrive_list(self, query: str, fields: str) -> list[dict[str, Any]]:\n        items: list[dict[str, Any]] = []\n        page_token: str | None = None\n        while True:\n            with self._lock:\n                tok = self._get_access_token_locked()\n            params = {\n                "q": query,\n                "fields": f"nextPageToken,{fields}",\n                "pageSize": "1000",\n            }\n            if page_token:\n                params["pageToken"] = page_token\n            url = "https://www.googleapis.com/drive/v3/files?" + urllib.parse.urlencode(params)\n            req = urllib.request.Request(url, headers={"Authorization": f"Bearer {tok}"})\n            try:\n                with urllib.request.urlopen(req, timeout=20.0) as resp:\n                    data = json.loads(resp.read().decode("utf-8"))\n            except urllib.error.HTTPError as err:\n                if err.code == 401:\n                    with self._lock:\n                        self._access_token = ""\n                        self._token_expires_mono = 0.0\n                        tok = self._get_access_token_locked()\n                    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {tok}"})\n                    with urllib.request.urlopen(req, timeout=20.0) as resp:\n                        data = json.loads(resp.read().decode("utf-8"))\n                else:\n                    raise\n            items.extend(data.get("files") or [])\n            page_token = data.get("nextPageToken")\n            if not page_token:\n                break\n        return items\n\n    def _resolve_folder_id(self, parent_parts: tuple[str, ...]) -> str | None:\n        cur_rel = ""\n        cur_id = "root"\n        for seg in parent_parts:\n            nxt_rel = f"{cur_rel}/{seg}" if cur_rel else seg\n            with self._lock:\n                cached = self._folder_ids.get(nxt_rel)\n            if cached is not None:\n                cur_id = cached\n                cur_rel = nxt_rel\n                continue\n            q = (\n                f"\'{_gdrive_escape_q(cur_id)}\' in parents and trashed = false "\n                "and mimeType = \'application/vnd.google-apps.folder\' "\n                f"and name = \'{_gdrive_escape_q(seg)}\'"\n            )\n            matches = self._gdrive_list(q, "files(id,name)")\n            if not matches:\n                return None\n            fid = str(matches[0]["id"])\n            with self._lock:\n                self._folder_ids[nxt_rel] = fid\n            cur_id = fid\n            cur_rel = nxt_rel\n        return cur_id\n\n    def _cloud_dir_listing(self, parent_parts: tuple[str, ...]) -> dict[str, list[tuple[int, bool]]] | None:\n        rel_dir = "/".join(parent_parts)\n        now = time.monotonic()\n        with self._lock:\n            cached = self._dir_cache.get(rel_dir)\n            if cached is not None and (now - cached[0]) < self._LIST_CACHE_TTL:\n                return cached[1]\n        folder_id = self._resolve_folder_id(parent_parts)\n        if folder_id is None:\n            return None\n        q = f"\'{_gdrive_escape_q(folder_id)}\' in parents and trashed = false"\n        files = self._gdrive_list(q, "files(name,size,md5Checksum)")\n        listing: dict[str, list[tuple[int, bool]]] = {}\n        for item in files:\n            fname = str(item.get("name") or "")\n            if not fname:\n                continue\n            try:\n                sz = int(item.get("size") or 0)\n            except (TypeError, ValueError):\n                sz = 0\n            has_md5 = bool(item.get("md5Checksum"))\n            listing.setdefault(fname, []).append((sz, has_md5))\n        with self._lock:\n            self._dir_cache[rel_dir] = (time.monotonic(), listing)\n        return listing\n\n    def _fallback_check(self, full_path: Path, expected_size: int) -> str:\n        try:\n            raw = os.getxattr(full_path, "user.drive.md5")  # type: ignore[attr-defined]\n            val = raw.decode("utf-8", errors="ignore").strip()\n            if val or expected_size == 0:\n                return "synced"\n            return "pending_sync"\n        except OSError as err:\n            if err.errno == getattr(errno, "ENODATA", 61):\n                return "pending_sync"\n            if err.errno not in (errno.ENOTSUP, getattr(errno, "EOPNOTSUPP", errno.ENOTSUP), errno.ENOSYS):\n                return "pending_sync"\n        except AttributeError:\n            pass\n        with contextlib.suppress(OSError):\n            for log_p in Path("/tmp").glob("drive*.ERROR*"):\n                try:\n                    sz = log_p.stat().st_size\n                    with log_p.open("rb") as lf:\n                        if sz > 512 * 1024:\n                            lf.seek(sz - 512 * 1024)\n                        tail = lf.read().decode("utf-8", errors="replace")\n                    if "userRateLimitExceeded" in tail:\n                        return "pending_sync"\n                except OSError:\n                    continue\n        return "synced"\n\n    def verify_file(self, full_path: Path, expected_size: int) -> str:\n        """Return `\'synced\'`, `\'pending_sync\'`, or `\'missing\'`."""\n        expected = int(expected_size)\n        try:\n            if not full_path.is_file() or full_path.stat().st_size != expected:\n                return "missing"\n        except OSError:\n            return "missing"\n\n        if self._mydrive_root is None:\n            return "synced"\n\n        now = time.monotonic()\n        key = (full_path, expected)\n        with self._lock:\n            cached = self._file_cache.get(key)\n            if cached is not None and (now - cached[0]) < self._FILE_CACHE_TTL:\n                return cached[1]\n\n        try:\n            rel = full_path.resolve().relative_to(self._mydrive_root)\n            parent_parts = rel.parts[:-1]\n            filename = rel.name\n            listing = self._cloud_dir_listing(parent_parts)\n            if listing is None:\n                res = "pending_sync"\n            else:\n                entries = listing.get(filename) or []\n                if any(sz == expected and (expected == 0 or has_md5) for sz, has_md5 in entries):\n                    res = "synced"\n                else:\n                    res = "pending_sync"\n        except Exception as err:\n            self._log(f"Drive Cloud verify fallback for {full_path.name}: {err}")\n            res = self._fallback_check(full_path, expected)\n\n        with self._lock:\n            self._file_cache[key] = (time.monotonic(), res)\n        return res\n\n\nclass DriveDirCache:\n    """Thread-safe bounded directory & readdir cache with S3 file-vs-directory POSIX collision repair."""\n\n    def __init__(\n        self,\n        dest_root: Path,\n        *,\n        max_entries: int = 50_000,\n        max_dir_entries: int = 8_000,\n    ) -> None:\n        self.dest_root = Path(dest_root)\n        self.max_entries = max(1000, int(max_entries))\n        self.max_dir_entries = max(256, int(max_dir_entries))\n        self.lock = threading.Lock()\n        self._known: OrderedDict[str, None] = OrderedDict()\n        # rel_dir -> {entry_name: -1 (dir) | None (file, size unknown) | int >= 0 (file size)}\n        self._dir_entries: OrderedDict[str, dict[str, int | None]] = OrderedDict()\n        self._dir_total_items = 0\n        self._scanning: set[str] = set()\n\n    def _remember_locked(self, rel_dir: str) -> None:\n        if rel_dir in self._known:\n            self._known.move_to_end(rel_dir)\n            return\n        self._known[rel_dir] = None\n        if len(self._known) > self.max_entries:\n            self._known.popitem(last=False)\n\n    def _invalidate_locked(self, rel_dir: str) -> None:\n        self._known.pop(rel_dir, None)\n        old = self._dir_entries.pop(rel_dir, None)\n        if old is not None:\n            self._dir_total_items = max(0, self._dir_total_items - len(old))\n\n    def _set_dir_entries_locked(self, rel_dir: str, mapping: dict[str, int | None]) -> None:\n        prev = self._dir_entries.pop(rel_dir, None)\n        if prev is not None:\n            self._dir_total_items = max(0, self._dir_total_items - len(prev))\n        self._dir_entries[rel_dir] = mapping\n        self._dir_total_items += len(mapping)\n        while (len(self._dir_entries) > self.max_dir_entries or self._dir_total_items > _DIR_ENTRY_CAP) and len(self._dir_entries) > 1:\n            _, evicted = self._dir_entries.popitem(last=False)\n            self._dir_total_items = max(0, self._dir_total_items - len(evicted))\n\n    def _record_entry_locked(self, parent_rel: str, name: str, val: int | None) -> None:\n        entries = self._dir_entries.get(parent_rel)\n        if entries is not None:\n            if name not in entries:\n                self._dir_total_items += 1\n            entries[name] = val\n            self._dir_entries.move_to_end(parent_rel)\n\n    def ensure_dir_locked(\n        self,\n        rel_dir: str,\n        on_relocate: Callable[[str, str], None] | None = None,\n    ) -> Path:\n        """Create directory `dest_root / rel_dir`, relocating any colliding file into `dir/__r2_object__`."""\n        clean = rel_dir.strip("/")\n        if "" not in self._known:\n            self.dest_root.mkdir(parents=True, exist_ok=True)\n            self._remember_locked("")\n        if not clean:\n            return self.dest_root\n        if clean in self._known:\n            self._known.move_to_end(clean)\n            return self.dest_root / clean\n\n        cur = self.dest_root\n        parts = [p for p in clean.split("/") if p]\n        built: list[str] = []\n        parent_rel = ""\n        for seg in parts:\n            built.append(seg)\n            cur_rel = "/".join(built)\n            cur = cur / seg\n            if cur_rel in self._known:\n                parent_rel = cur_rel\n                continue\n            try:\n                os.mkdir(cur)\n                # Newly created folder did not exist on disk -> known empty (0 scandir/stat calls later).\n                self._set_dir_entries_locked(cur_rel, {})\n                self._record_entry_locked(parent_rel, seg, -1)\n            except FileExistsError:\n                if not cur.is_dir():\n                    # Regular file already exists where a directory is needed (S3 key `a/b` + `a/b/c`).\n                    tmp_reloc = cur.with_name(f".{cur.name}.ctreloc.{uuid.uuid4().hex[:8]}")\n                    os.replace(cur, tmp_reloc)\n                    cur.mkdir(parents=True, exist_ok=True)\n                    target_inside = cur / "__r2_object__"\n                    os.replace(tmp_reloc, target_inside)\n                    self._invalidate_locked(parent_rel)\n                    reloc_sz: int | None = None\n                    with contextlib.suppress(OSError):\n                        reloc_sz = target_inside.stat().st_size\n                    self._set_dir_entries_locked(cur_rel, {"__r2_object__": reloc_sz})\n                    if on_relocate is not None:\n                        on_relocate(cur_rel, f"{cur_rel}/__r2_object__")\n                else:\n                    self._record_entry_locked(parent_rel, seg, -1)\n            self._remember_locked(cur_rel)\n            parent_rel = cur_rel\n        return cur\n\n    def ensure_parent_dir(\n        self,\n        rel_path: str,\n        on_relocate: Callable[[str, str], None] | None = None,\n    ) -> str:\n        """Ensure parent directory exists under short `self.lock`; returns stripped `rel_path`."""\n        actual_rel = rel_path.strip("/")\n        parent_rel, _, _ = actual_rel.rpartition("/")\n        with self.lock:\n            self.ensure_dir_locked(parent_rel, on_relocate=on_relocate)\n        return actual_rel\n\n    def _scan_dir_if_needed(self, parent_rel: str) -> dict[str, int | None] | None:\n        with self.lock:\n            entries = self._dir_entries.get(parent_rel)\n            if entries is not None:\n                self._dir_entries.move_to_end(parent_rel)\n                return entries\n        target_dir = (self.dest_root / parent_rel) if parent_rel else self.dest_root\n        scanned: dict[str, int | None] = {}\n        try:\n            with os.scandir(target_dir) as it:\n                for entry in it:\n                    name = entry.name\n                    if name.startswith(".") and (".ctpart." in name or ".ctreloc." in name):\n                        continue\n                    try:\n                        is_d = entry.is_dir(follow_symlinks=False)\n                    except OSError:\n                        is_d = False\n                    scanned[name] = -1 if is_d else None\n        except OSError:\n            return None\n        with self.lock:\n            existing = self._dir_entries.get(parent_rel)\n            if existing is not None:\n                self._dir_entries.move_to_end(parent_rel)\n                return existing\n            self._remember_locked(parent_rel)\n            self._set_dir_entries_locked(parent_rel, scanned)\n            return scanned\n\n    def preexisting_file_size(self, rel_path: str) -> tuple[str, int | None]:\n        """Fast Drive existence check via bounded readdir cache; handles file-vs-dir collisions."""\n        clean = rel_path.strip("/")\n        for _ in range(2):\n            parent_rel, _, name = clean.rpartition("/")\n            with self.lock:\n                if clean in self._known:\n                    clean = f"{clean}/__r2_object__"\n                    continue\n                cached = self._dir_entries.get(parent_rel)\n                if cached is not None:\n                    self._dir_entries.move_to_end(parent_rel)\n                    val = cached.get(name, -2)\n                    if val == -2:\n                        return clean, None\n                    if val == -1:\n                        clean = f"{clean}/__r2_object__"\n                        continue\n                    if isinstance(val, int) and val >= 0:\n                        return clean, val\n\n            scanned = self._scan_dir_if_needed(parent_rel)\n            if scanned is None:\n                return clean, None\n            val = scanned.get(name, -2)\n            if val == -2:\n                return clean, None\n            if val == -1:\n                clean = f"{clean}/__r2_object__"\n                continue\n            if isinstance(val, int) and val >= 0:\n                return clean, val\n            # File exists in directory listing with unqueried size -> single stat() outside lock.\n            try:\n                sz = (self.dest_root / clean).stat().st_size\n            except OSError:\n                return clean, None\n            with self.lock:\n                self._record_entry_locked(parent_rel, name, int(sz))\n            return clean, int(sz)\n        return clean, None\n\n    def write_file_direct(\n        self,\n        rel_path: str,\n        src_path: Path | None,\n        expected_size: int,\n        on_relocate: Callable[[str, str], None] | None = None,\n    ) -> str:\n        """Write file directly to DriveFS without holding `self.lock` across FUSE I/O."""\n        actual_rel = rel_path.strip("/")\n        for attempt in range(2):\n            parent_rel, _, name = actual_rel.rpartition("/")\n            with self.lock:\n                self.ensure_dir_locked(parent_rel, on_relocate=on_relocate)\n            dest = self.dest_root / actual_rel\n            try:\n                with dest.open("wb") as dst_f:\n                    if expected_size > 0 and src_path is not None:\n                        with src_path.open("rb") as src_f:\n                            shutil.copyfileobj(src_f, dst_f, length=1024 * 1024)\n                    if expected_size >= _LARGE_FILE_FSYNC_BYTES:\n                        dst_f.flush()\n                        with contextlib.suppress(OSError):\n                            os.fsync(dst_f.fileno())\n                actual_size = dest.stat().st_size\n                if actual_size != int(expected_size):\n                    with contextlib.suppress(OSError):\n                        dest.unlink(missing_ok=True)\n                    raise OSError(f"Drive verify failed for {dest}: expected {expected_size}B, got {actual_size}B")\n                with self.lock:\n                    self._record_entry_locked(parent_rel, name, int(expected_size))\n                return actual_rel\n            except (IsADirectoryError, FileExistsError):\n                if attempt == 0 and dest.is_dir():\n                    actual_rel = f"{actual_rel}/__r2_object__"\n                    with self.lock:\n                        self._remember_locked(rel_path.strip("/"))\n                    continue\n                raise\n            except OSError as exc:\n                if attempt == 0 and dest.is_dir():\n                    actual_rel = f"{actual_rel}/__r2_object__"\n                    with self.lock:\n                        self._remember_locked(rel_path.strip("/"))\n                    continue\n                if attempt == 0 and exc.errno in (2, 20):  # ENOENT / ENOTDIR (stale dir cache or collision)\n                    with self.lock:\n                        self._invalidate_locked(parent_rel)\n                    continue\n                with contextlib.suppress(OSError):\n                    if dest.is_file():\n                        dest.unlink(missing_ok=True)\n                raise\n        raise OSError(f"Drive write failed for {rel_path}")\n\n\ndef _open_state_db(db_path: Path) -> sqlite3.Connection:\n    db_path.parent.mkdir(parents=True, exist_ok=True)\n    conn = sqlite3.connect(str(db_path), timeout=60.0, check_same_thread=False)\n    conn.execute("PRAGMA journal_mode=WAL")\n    conn.execute("PRAGMA synchronous=NORMAL")\n    conn.execute("PRAGMA temp_store=MEMORY")\n    conn.execute("PRAGMA busy_timeout=60000")\n    conn.executescript(\n        """\n        CREATE TABLE IF NOT EXISTS job_meta (\n            k TEXT PRIMARY KEY,\n            v TEXT NOT NULL\n        );\n        CREATE TABLE IF NOT EXISTS buckets (\n            bucket TEXT PRIMARY KEY,\n            last_key TEXT NOT NULL DEFAULT \'\',\n            listing_done INTEGER NOT NULL DEFAULT 0,\n            objects_listed INTEGER NOT NULL DEFAULT 0,\n            bytes_listed INTEGER NOT NULL DEFAULT 0\n        );\n        CREATE TABLE IF NOT EXISTS objects (\n            bucket TEXT NOT NULL,\n            key TEXT NOT NULL,\n            size INTEGER NOT NULL,\n            etag TEXT NOT NULL,\n            storage_class TEXT NOT NULL,\n            rel_path TEXT NOT NULL,\n            is_dir INTEGER NOT NULL DEFAULT 0,\n            status TEXT NOT NULL,\n            error TEXT NOT NULL DEFAULT \'\',\n            updated_at REAL NOT NULL,\n            PRIMARY KEY (bucket, key)\n        );\n        CREATE INDEX IF NOT EXISTS idx_objects_status ON objects(status, bucket);\n        CREATE INDEX IF NOT EXISTS idx_objects_status_bk ON objects(status, bucket, key);\n        CREATE INDEX IF NOT EXISTS idx_objects_rel ON objects(rel_path);\n        CREATE TABLE IF NOT EXISTS tar_shards (\n            shard_id INTEGER PRIMARY KEY AUTOINCREMENT,\n            bucket TEXT NOT NULL,\n            shard_name TEXT NOT NULL UNIQUE,\n            rel_tar_path TEXT NOT NULL,\n            file_count INTEGER NOT NULL,\n            payload_bytes INTEGER NOT NULL,\n            tar_bytes INTEGER NOT NULL,\n            created_at REAL NOT NULL\n        );\n        CREATE TABLE IF NOT EXISTS archived_files (\n            rel_path TEXT PRIMARY KEY,\n            bucket TEXT NOT NULL DEFAULT \'\',\n            key TEXT NOT NULL DEFAULT \'\',\n            size INTEGER NOT NULL,\n            shard_name TEXT NOT NULL,\n            source TEXT NOT NULL,\n            loose_cleaned INTEGER NOT NULL DEFAULT 0,\n            archived_at REAL NOT NULL\n        );\n        CREATE INDEX IF NOT EXISTS idx_archived_shard ON archived_files(shard_name);\n        CREATE INDEX IF NOT EXISTS idx_archived_uncleaned ON archived_files(loose_cleaned) WHERE loose_cleaned = 0;\n        """\n    )\n    conn.commit()\n    return conn\n\n\ndef _backup_sqlite_to_drive(\n    src_conn: sqlite3.Connection,\n    local_backup_tmp: Path,\n    drive_checkpoint: Path,\n    *,\n    db_path: Path | None = None,\n) -> None:\n    """Snapshot local NVMe SQLite WAL DB -> local file (or direct WAL checkpoint) -> in-place/atomic sync on Drive."""\n    drive_checkpoint.parent.mkdir(parents=True, exist_ok=True)\n    with contextlib.suppress(OSError):\n        for stale in drive_checkpoint.parent.iterdir():\n            if stale.is_file() and stale.name.startswith(f".{drive_checkpoint.name}.") and stale.name.endswith(".tmp"):\n                with contextlib.suppress(OSError):\n                    stale.unlink(missing_ok=True)\n\n    local_backup_tmp.parent.mkdir(parents=True, exist_ok=True)\n    with contextlib.suppress(OSError):\n        local_backup_tmp.unlink(missing_ok=True)\n\n    use_direct_db = False\n    if db_path is not None and db_path.is_file():\n        with contextlib.suppress(OSError):\n            db_sz = db_path.stat().st_size\n            free_sz = shutil.disk_usage(local_backup_tmp.parent).free\n            if free_sz < db_sz + 256 * 1024 * 1024:\n                use_direct_db = True\n\n    src_file: Path\n    if use_direct_db and db_path is not None:\n        src_conn.commit()\n        src_conn.execute("PRAGMA wal_checkpoint(PASSIVE)")\n        src_file = db_path\n    else:\n        dst = sqlite3.connect(str(local_backup_tmp))\n        try:\n            src_conn.backup(dst)\n            dst.commit()\n        finally:\n            dst.close()\n        src_file = local_backup_tmp\n\n    drive_tmp: Path | None = None\n    try:\n        if drive_checkpoint.is_file():\n            with src_file.open("rb") as src_f, drive_checkpoint.open("r+b") as dst_f:\n                dst_f.seek(0)\n                shutil.copyfileobj(src_f, dst_f, length=4 * 1024 * 1024)\n                dst_f.truncate()\n                dst_f.flush()\n                with contextlib.suppress(OSError):\n                    os.fsync(dst_f.fileno())\n        else:\n            drive_tmp = drive_checkpoint.with_name(f".{drive_checkpoint.name}.{os.getpid()}.{uuid.uuid4().hex[:8]}.tmp")\n            with src_file.open("rb") as src_f, drive_tmp.open("wb") as dst_f:\n                shutil.copyfileobj(src_f, dst_f, length=4 * 1024 * 1024)\n                dst_f.flush()\n                with contextlib.suppress(OSError):\n                    os.fsync(dst_f.fileno())\n            os.replace(drive_tmp, drive_checkpoint)\n            drive_tmp = None\n    finally:\n        if drive_tmp is not None:\n            with contextlib.suppress(OSError):\n                drive_tmp.unlink(missing_ok=True)\n        with contextlib.suppress(OSError):\n            local_backup_tmp.unlink(missing_ok=True)\n\n\ndef _restore_sqlite_from_drive(drive_checkpoint: Path, local_db: Path) -> bool:\n    if not drive_checkpoint.is_file():\n        return False\n    local_db.parent.mkdir(parents=True, exist_ok=True)\n    tmp = local_db.with_name(f".{local_db.name}.restore.{uuid.uuid4().hex[:8]}.tmp")\n    try:\n        with drive_checkpoint.open("rb") as src_f, tmp.open("wb") as dst_f:\n            shutil.copyfileobj(src_f, dst_f, length=1024 * 1024)\n            dst_f.flush()\n        for suffix in ("-wal", "-shm"):\n            with contextlib.suppress(OSError):\n                Path(str(local_db) + suffix).unlink(missing_ok=True)\n        os.replace(tmp, local_db)\n        return True\n    finally:\n        with contextlib.suppress(OSError):\n            tmp.unlink(missing_ok=True)\n\n\nclass CredentialStore:\n    """Handles R2 credential resolution across request overrides, env vars, and 0600 Drive storage."""\n\n    def __init__(self, data_dir: Path) -> None:\n        self.data_dir = Path(data_dir)\n        self.drive_creds_path = self.data_dir / "r2-credentials.json"\n        self.legacy_shared_path = Path("/content/drive/MyDrive/colab-llm/r2_credentials.json")\n        self._lock = threading.Lock()\n        self._memory: R2Credentials | None = None\n\n    def set_credentials(\n        self,\n        creds: R2Credentials,\n        *,\n        persist: bool = True,\n    ) -> dict[str, Any]:\n        creds.validate()\n        with self._lock:\n            self._memory = creds\n            if persist:\n                _atomic_write_json(self.drive_creds_path, creds.to_dict(), mode=0o600)\n                return creds.redacted_summary(source="drive")\n            return creds.redacted_summary(source="memory")\n\n    def clear(self) -> None:\n        with self._lock:\n            self._memory = None\n            with contextlib.suppress(OSError):\n                self.drive_creds_path.unlink(missing_ok=True)\n\n    def resolve(self, override: dict[str, Any] | R2Credentials | None = None) -> tuple[R2Credentials | None, str]:\n        if isinstance(override, R2Credentials):\n            override.validate()\n            return override, "request"\n        if isinstance(override, dict) and any(\n            str(override.get(k) or "").strip()\n            for k in ("access_key_id", "secret_access_key", "account_id", "endpoint_url")\n        ):\n            creds = R2Credentials.from_dict(override)\n            # Fill partial override from saved/env defaults if user supplied only secret or endpoint.\n            fallback, _ = self.resolve(None)\n            if fallback is not None:\n                if not creds.account_id and not creds.endpoint_url:\n                    creds.account_id = fallback.account_id\n                    creds.endpoint_url = fallback.endpoint_url\n                    creds.jurisdiction = creds.jurisdiction or fallback.jurisdiction\n                if not creds.access_key_id:\n                    creds.access_key_id = fallback.access_key_id\n                if not creds.secret_access_key:\n                    creds.secret_access_key = fallback.secret_access_key\n            creds.validate()\n            return creds, "request"\n\n        with self._lock:\n            if self._memory is not None:\n                return self._memory, "memory"\n\n        env_ak = os.environ.get("R2_ACCESS_KEY_ID", "").strip()\n        env_sk = os.environ.get("R2_SECRET_ACCESS_KEY", "").strip()\n        env_ep = os.environ.get("R2_ENDPOINT_URL", "").strip()\n        env_acct = os.environ.get("R2_ACCOUNT_ID", "").strip()\n        env_jur = os.environ.get("R2_JURISDICTION", "").strip()\n        if env_ak and env_sk and (env_ep or env_acct):\n            creds = R2Credentials(\n                access_key_id=env_ak,\n                secret_access_key=env_sk,\n                account_id=env_acct,\n                endpoint_url=env_ep,\n                jurisdiction=env_jur,\n            )\n            creds.validate()\n            return creds, "env"\n\n        for src_name, path in (("drive", self.drive_creds_path), ("shared_drive", self.legacy_shared_path)):\n            if path.is_file():\n                try:\n                    loaded = json.loads(path.read_text(encoding="utf-8"))\n                    creds = R2Credentials.from_dict(loaded)\n                    creds.validate()\n                    return creds, src_name\n                except (OSError, ValueError):\n                    continue\n        return None, "none"\n\n    def summary(self) -> dict[str, Any]:\n        creds, source = self.resolve(None)\n        if creds is None:\n            return {"configured": False, "source": "none"}\n        return creds.redacted_summary(source=source)\n\n\n@dataclasses.dataclass\nclass _RuntimeJob:\n    job_id: str\n    config: dict[str, Any]\n    db_path: Path\n    drive_job_dir: Path\n    spool_dir: Path\n    dest_root: Path\n    status: str = "queued"\n    message: str = ""\n    paid_fallback_used: bool = False\n    created_at: float = dataclasses.field(default_factory=time.time)\n    started_at: float | None = None\n    finished_at: float | None = None\n    session_started_at: float | None = None\n    session_finished_at: float | None = None\n    updated_at: float = dataclasses.field(default_factory=time.time)\n    lock: threading.Lock = dataclasses.field(default_factory=threading.Lock)\n    stop_event: threading.Event = dataclasses.field(default_factory=threading.Event)\n    pause_requested: bool = False\n    cancel_requested: bool = False\n    worker_thread: threading.Thread | None = None\n    db_lock: threading.Lock = dataclasses.field(default_factory=threading.Lock)\n    conn: sqlite3.Connection | None = None\n    recent_errors: deque[dict[str, str]] = dataclasses.field(default_factory=lambda: deque(maxlen=50))\n    status_counts: dict[str, dict[str, int]] = dataclasses.field(default_factory=dict)\n    bytes_done_session: int = 0\n    files_done_session: int = 0\n    tar_shards_count: int = 0\n    tar_bytes_total: int = 0\n    consolidated_files: int = 0\n    consolidated_bytes: int = 0\n    last_db_refresh_mono: float = 0.0\n    buckets_snapshot: list[dict[str, Any]] = dataclasses.field(default_factory=list)\n\n\n_SHARD_SEQ_RE = re.compile(r"-part-(\\d+)\\.tar$")\n\n\ndef _is_temp_artifact_name(name: str) -> bool:\n    """Return True only for r2-migrate internal staging/temp files, never user dotfiles."""\n    return name.startswith(".") and (".ctpart." in name or ".ctreloc." in name)\n\n\n@dataclasses.dataclass\nclass _ArchiveTask:\n    bucket: str\n    key: str\n    rel_path: str\n    size: int\n    is_dir: bool\n    source: str  # "r2_download" | "drive_existing"\n    shard_bucket: str = ""\n    payload_bytes: bytes | None = None\n    source_path: Path | None = None\n    unlink_source_after_pack: bool = False\n    held_budget: int = 0\n    release_fn: Callable[[int], None] | None = None\n    loose_path: Path | None = None\n\n\n@dataclasses.dataclass\nclass _ArchivedEntry:\n    bucket: str\n    key: str\n    orig_rel_path: str\n    archive_path: str\n    size: int\n    is_dir: bool\n    source: str\n    loose_path: Path | None = None\n\n\n@dataclasses.dataclass\nclass _StagingShard:\n    bucket: str\n    safe_bucket: str\n    shard_name: str\n    rel_tar_path: str\n    stage_tar_path: Path\n    raw_fh: io.BufferedWriter\n    tar: tarfile.TarFile\n    entries: list[_ArchivedEntry] = dataclasses.field(default_factory=list)\n    seen_paths: set[str] = dataclasses.field(default_factory=set)\n    payload_bytes: int = 0\n    file_count: int = 0\n    created_mono: float = dataclasses.field(default_factory=time.monotonic)\n\n\n@dataclasses.dataclass\nclass _FlushOpenMarker:\n    wait_event: threading.Event\n\n\n@dataclasses.dataclass\nclass _StopPackerMarker:\n    pass\n\n\nclass _TarArchivePipeline:\n    """Packs R2 downloads & existing Drive files into uncompressed PAX tar shards on NVMe,\n    atomically flushes them to Drive with manifests, commits SQLite, then cleans loose files."""\n\n    def __init__(\n        self,\n        *,\n        job: _RuntimeJob,\n        conn: sqlite3.Connection,\n        tar_shard_mb: float,\n        on_committed: Callable[\n            [list[_ArchivedEntry], int, dict[str, tuple[int, int]], int, int, int, int], None\n        ],\n        on_failed: Callable[[list[_ArchivedEntry], str], None],\n        on_quota_pause: Callable[[str], None] | None = None,\n        log: Callable[[str], None],\n    ) -> None:\n        self.job = job\n        self.conn = conn\n        self.archives_dir = job.dest_root / "archives"\n        self.tar_shard_bytes = max(1024 * 1024, int(float(tar_shard_mb) * 1024 * 1024))\n        self.max_members_per_shard = 50_000\n        self._on_committed = on_committed\n        self._on_failed = on_failed\n        self._on_quota_pause = on_quota_pause\n        self._log = log\n\n        self.archives_dir.mkdir(parents=True, exist_ok=True)\n        self._ensure_restore_readme()\n        self._loose_unlink_pool = concurrent.futures.ThreadPoolExecutor(\n            max_workers=24, thread_name_prefix=f"r2-unlink-{job.job_id}"\n        )\n        self.reconcile_drive_manifests()\n        self.reconcile_missing_drive_shards()\n        self.heal_uncleaned_loose_files()\n\n        self._seq_lock = threading.Lock()\n        self._next_seq = self._init_next_seq()\n        self._open_shards: dict[str, _StagingShard] = {}\n        self._known_dirs: set[str] = set()\n        self._archived_paths_mem: set[str] = set()\n\n        self._inflight_lock = threading.Lock()\n        self._inflight_tasks = 0\n        self._closed = False\n\n        self._pack_q: queue.Queue[Any] = queue.Queue(maxsize=256)\n        self._drive_flush_q: queue.Queue[tuple[_StagingShard, int] | None] = queue.Queue(maxsize=2)\n        self._flush_threads = [\n            threading.Thread(\n                target=self._flush_worker_loop,\n                name=f"r2-tar-flush-{job.job_id}-{i}",\n                daemon=True,\n            )\n            for i in range(2)\n        ]\n        for th in self._flush_threads:\n            th.start()\n        self._packer_thread = threading.Thread(\n            target=self._packer_loop,\n            name=f"r2-tar-pack-{job.job_id}",\n            daemon=True,\n        )\n        self._packer_thread.start()\n\n    def _ensure_restore_readme(self) -> None:\n        readme = self.archives_dir / "README-RESTORE.txt"\n        if readme.is_file():\n            return\n        content = (\n            "Cloudflare R2 -> Google Drive TAR Shards Archive\\n"\n            "===============================================\\n\\n"\n            "Format: Uncompressed POSIX PAX (.tar) shards + JSONL manifests (*.tar.manifest.jsonl).\\n\\n"\n            "Restore all shards under this directory preserving relative bucket/key paths:\\n"\n            "  mkdir -p ./restored\\n"\n            "  find . -name \'*.tar\' -print0 | xargs -0 -n1 -I{} tar -xf \'{}\' -C ./restored\\n\\n"\n            "Notes:\\n"\n            "- If an S3 key collided with a directory of the same path (e.g. \'a\' and \'a/b\'),\\n"\n            "  the file object is stored as \'a/__r2_object__\'.\\n"\n            "- Each \'*.tar.manifest.jsonl\' file lists every archived object with its bucket,\\n"\n            "  key, archive_path, size, source, and shard byte size for integrity audits.\\n"\n        )\n        with contextlib.suppress(OSError):\n            tmp = readme.with_name(f".{readme.name}.ctpart.{os.getpid()}.{uuid.uuid4().hex[:6]}")\n            tmp.write_text(content, encoding="utf-8")\n            os.replace(tmp, readme)\n\n    def _init_next_seq(self) -> dict[str, int]:\n        seqs: dict[str, int] = {}\n        with self.job.db_lock:\n            rows = self.conn.execute("SELECT bucket, shard_name FROM tar_shards").fetchall()\n        for bucket, shard_name in rows:\n            m = _SHARD_SEQ_RE.search(str(shard_name))\n            if m:\n                n = int(m.group(1))\n                seqs[str(bucket)] = max(seqs.get(str(bucket), 1), n + 1)\n        if self.archives_dir.is_dir():\n            for bucket_dir in self.archives_dir.iterdir():\n                if not bucket_dir.is_dir():\n                    continue\n                for p in bucket_dir.iterdir():\n                    m = _SHARD_SEQ_RE.search(p.name)\n                    if m:\n                        n = int(m.group(1))\n                        # Prefix before -part- is safe_bucket; preserve highest seq across matching dirs too.\n                        b_guess = bucket_dir.name\n                        seqs[b_guess] = max(seqs.get(b_guess, 1), n + 1)\n        return seqs\n\n    def _alloc_seq(self, bucket: str, safe_bucket: str) -> int:\n        with self._seq_lock:\n            cur = max(self._next_seq.get(bucket, 1), self._next_seq.get(safe_bucket, 1))\n            self._next_seq[bucket] = cur + 1\n            self._next_seq[safe_bucket] = cur + 1\n            return cur\n\n    def pending_count(self) -> int:\n        with self._inflight_lock:\n            return self._inflight_tasks\n\n    def has_open_shards(self) -> bool:\n        with self._inflight_lock:\n            return self._inflight_tasks > 0 or bool(self._open_shards)\n\n    def find_archived(self, rel_path: str, expected_size: int | None = None) -> tuple[str, int, str, str] | None:\n        clean = rel_path.strip("/") or "__r2_root__"\n        cands = (clean, f"{clean}/__r2_object__")\n        with self.job.db_lock:\n            rows = self.conn.execute(\n                "SELECT rel_path, size, bucket, key FROM archived_files WHERE rel_path IN (?, ?)",\n                cands,\n            ).fetchall()\n        for rpath, sz, b, k in rows:\n            if expected_size is None or int(sz) == int(expected_size):\n                return str(rpath), int(sz), str(b), str(k)\n        return None\n\n    def backfill_archived_bk(self, archived_rel_path: str, bucket: str, key: str) -> None:\n        with self.job.db_lock:\n            self.conn.execute(\n                "UPDATE archived_files SET bucket=?, key=? WHERE rel_path=? AND bucket=\'\'",\n                (bucket, key, archived_rel_path),\n            )\n            self.conn.commit()\n\n    def submit(self, task: _ArchiveTask) -> None:\n        with self._inflight_lock:\n            self._inflight_tasks += 1\n        self._pack_q.put(task)\n\n    def flush_open_shards(self, *, wait: bool = True) -> None:\n        ev = threading.Event()\n        self._pack_q.put(_FlushOpenMarker(ev))\n        if wait:\n            ev.wait()\n\n    def close(self, *, wait: bool = True) -> None:\n        if self._closed:\n            return\n        self._closed = True\n        self._pack_q.put(_StopPackerMarker())\n        if wait:\n            self._packer_thread.join()\n            for _ in self._flush_threads:\n                self._drive_flush_q.put(None)\n            for th in self._flush_threads:\n                th.join()\n            self._loose_unlink_pool.shutdown(wait=True)\n\n    def _dec_inflight(self, count: int) -> None:\n        if count <= 0:\n            return\n        with self._inflight_lock:\n            self._inflight_tasks = max(0, self._inflight_tasks - count)\n\n    def _open_new_shard(self, bucket: str) -> _StagingShard:\n        safe_bucket = sanitize_segment(bucket)\n        while True:\n            seq = self._alloc_seq(bucket, safe_bucket)\n            shard_name = f"{safe_bucket}-part-{seq:05d}.tar"\n            rel_tar_path = f"archives/{safe_bucket}/{shard_name}"\n            dest_tar_path = self.job.dest_root / rel_tar_path\n            if not dest_tar_path.exists():\n                break\n        stage_tar_path = self.job.spool_dir / f".stage.{shard_name}.{uuid.uuid4().hex[:8]}.part"\n        stage_tar_path.parent.mkdir(parents=True, exist_ok=True)\n        raw_fh = open(stage_tar_path, "wb", buffering=4 * 1024 * 1024)\n        tar = tarfile.open(fileobj=raw_fh, mode="w", format=tarfile.PAX_FORMAT, encoding="utf-8")\n        shard = _StagingShard(\n            bucket=bucket,\n            safe_bucket=safe_bucket,\n            shard_name=shard_name,\n            rel_tar_path=rel_tar_path,\n            stage_tar_path=stage_tar_path,\n            raw_fh=raw_fh,\n            tar=tar,\n        )\n        self._open_shards[bucket] = shard\n        return shard\n\n    def _seal_and_enqueue(self, bucket: str) -> None:\n        shard = self._open_shards.pop(bucket, None)\n        if shard is None:\n            return\n        try:\n            shard.tar.close()\n            shard.raw_fh.flush()\n            shard.raw_fh.close()\n            if not shard.entries:\n                with contextlib.suppress(OSError):\n                    shard.stage_tar_path.unlink(missing_ok=True)\n                return\n            tar_bytes = shard.stage_tar_path.stat().st_size\n            self._drive_flush_q.put((shard, int(tar_bytes)))\n        except Exception as exc:\n            with contextlib.suppress(Exception):\n                shard.raw_fh.close()\n            with contextlib.suppress(OSError):\n                shard.stage_tar_path.unlink(missing_ok=True)\n            self._dec_inflight(len(shard.entries))\n            if (isinstance(exc, OSError) and exc.errno == errno.ENOSPC) or "No space left on device" in str(exc):\n                if self._on_quota_pause is not None:\n                    self._on_quota_pause(\n                        "Google Drive 挂载或本地缓存空间不足（可能已触发单账号 750 GB / 24h 上传限流 userRateLimitExceeded）；"\n                        "任务已安全暂停，待云端后台上传释放缓存后执行 resume 继续"\n                    )\n                    self._on_failed(shard.entries, "")\n                    return\n            self._on_failed(shard.entries, f"TAR shard seal failed: {exc}")\n\n    def _resolve_batch_paths(self, tasks: list[_ArchiveTask]) -> list[str]:\n        """Resolve S3 file-vs-directory collisions in batch with one SQLite lock acquisition."""\n        cleaned = [t.rel_path.strip("/") or "__r2_root__" for t in tasks]\n        for c, t in zip(cleaned, tasks):\n            if "/" in c:\n                cur = c.rsplit("/", 1)[0]\n                while cur:\n                    if cur in self._known_dirs:\n                        break\n                    self._known_dirs.add(cur)\n                    if "/" not in cur:\n                        break\n                    cur = cur.rsplit("/", 1)[0]\n            if t.is_dir:\n                self._known_dirs.add(c)\n\n        db_check_idx = [i for i, (c, t) in enumerate(zip(cleaned, tasks)) if not t.is_dir and c not in self._known_dirs]\n        db_has_children: set[str] = set()\n        if db_check_idx:\n            with self.job.db_lock:\n                for i in db_check_idx:\n                    c = cleaned[i]\n                    if self.conn.execute(\n                        "SELECT 1 FROM objects WHERE rel_path >= ? AND rel_path < ? LIMIT 1",\n                        (f"{c}/", f"{c}0"),\n                    ).fetchone() is not None:\n                        db_has_children.add(c)\n\n        resolved: list[str] = []\n        for c, t in zip(cleaned, tasks):\n            if not t.is_dir and (c in self._known_dirs or c in db_has_children):\n                self._known_dirs.add(c)\n                resolved.append(f"{c}/__r2_object__")\n            else:\n                resolved.append(c)\n        return resolved\n\n    def _packer_loop(self) -> None:\n        while True:\n            item = self._pack_q.get()\n            if isinstance(item, _StopPackerMarker):\n                for b in list(self._open_shards.keys()):\n                    self._seal_and_enqueue(b)\n                return\n            if isinstance(item, _FlushOpenMarker):\n                for b in list(self._open_shards.keys()):\n                    self._seal_and_enqueue(b)\n                # Wait until queued shards are flushed to Drive & committed.\n                self._drive_flush_q.join()\n                item.wait_event.set()\n                continue\n\n            batch: list[_ArchiveTask] = [item]\n            deferred_ctrl: Any = None\n            while len(batch) < 64:\n                try:\n                    nxt = self._pack_q.get_nowait()\n                except queue.Empty:\n                    break\n                if isinstance(nxt, (_StopPackerMarker, _FlushOpenMarker)):\n                    deferred_ctrl = nxt\n                    break\n                batch.append(nxt)\n\n            resolved_paths = self._resolve_batch_paths(batch)\n            for task, arch_path in zip(batch, resolved_paths):\n                self._pack_single_task(task, arch_path)\n\n            if isinstance(deferred_ctrl, _FlushOpenMarker):\n                for b in list(self._open_shards.keys()):\n                    self._seal_and_enqueue(b)\n                self._drive_flush_q.join()\n                deferred_ctrl.wait_event.set()\n            elif isinstance(deferred_ctrl, _StopPackerMarker):\n                for b in list(self._open_shards.keys()):\n                    self._seal_and_enqueue(b)\n                return\n\n    def _pack_single_task(self, task: _ArchiveTask, archive_path: str) -> None:\n        shard_bucket = task.shard_bucket or task.bucket or "drive-existing"\n        shard: _StagingShard | None = None\n        try:\n            shard = self._open_shards.get(shard_bucket)\n            if shard is None:\n                # Cap total open shards across different buckets to keep NVMe staging bounded.\n                if len(self._open_shards) >= 2:\n                    oldest_bucket = min(self._open_shards, key=lambda bk: self._open_shards[bk].created_mono)\n                    self._seal_and_enqueue(oldest_bucket)\n                shard = self._open_new_shard(shard_bucket)\n\n            member_name = archive_path\n            if task.is_dir and (member_name in shard.seen_paths or member_name in self._archived_paths_mem):\n                # Regular file already took this exact member name in this shard; skip conflicting dir entry.\n                self._dec_inflight(1)\n                self._on_committed(\n                    [\n                        _ArchivedEntry(\n                            bucket=task.bucket,\n                            key=task.key,\n                            orig_rel_path=task.rel_path,\n                            archive_path=archive_path,\n                            size=0,\n                            is_dir=True,\n                            source=task.source,\n                            loose_path=task.loose_path,\n                        )\n                    ],\n                    0,\n                    {},\n                    0,\n                    0,\n                    0,\n                    0,\n                )\n                return\n\n            now_mtime = int(time.time())\n            ti = tarfile.TarInfo(name=member_name)\n            ti.mtime = now_mtime\n            if task.is_dir:\n                ti.type = tarfile.DIRTYPE\n                ti.mode = 0o755\n                ti.size = 0\n                shard.tar.addfile(ti)\n            else:\n                ti.type = tarfile.REGTYPE\n                ti.mode = 0o644\n                ti.size = int(task.size)\n                if task.payload_bytes is not None:\n                    with io.BytesIO(task.payload_bytes) as bio:\n                        shard.tar.addfile(ti, fileobj=bio)\n                elif task.source_path is not None:\n                    with task.source_path.open("rb") as src_f:\n                        shard.tar.addfile(ti, fileobj=src_f)\n                else:\n                    with io.BytesIO(b"") as bio:\n                        shard.tar.addfile(ti, fileobj=bio)\n\n            shard.seen_paths.add(member_name)\n            if not task.is_dir:\n                self._archived_paths_mem.add(member_name)\n            shard.payload_bytes += int(task.size)\n            shard.file_count += 1\n            shard.entries.append(\n                _ArchivedEntry(\n                    bucket=task.bucket,\n                    key=task.key,\n                    orig_rel_path=task.rel_path,\n                    archive_path=archive_path,\n                    size=int(task.size),\n                    is_dir=bool(task.is_dir),\n                    source=task.source,\n                    loose_path=task.loose_path,\n                )\n            )\n            if shard.payload_bytes >= self.tar_shard_bytes or shard.file_count >= self.max_members_per_shard:\n                self._seal_and_enqueue(shard_bucket)\n        except Exception as exc:\n            err_entry = _ArchivedEntry(\n                bucket=task.bucket,\n                key=task.key,\n                orig_rel_path=task.rel_path,\n                archive_path=archive_path,\n                size=int(task.size),\n                is_dir=bool(task.is_dir),\n                source=task.source,\n                loose_path=task.loose_path,\n            )\n            self._dec_inflight(1)\n            self._on_failed([err_entry], f"TAR pack failed: {exc}")\n        finally:\n            if task.unlink_source_after_pack and task.source_path is not None:\n                with contextlib.suppress(OSError):\n                    task.source_path.unlink(missing_ok=True)\n            if task.held_budget and task.release_fn is not None:\n                task.release_fn(task.held_budget)\n\n    def _flush_worker_loop(self) -> None:\n        while True:\n            item = self._drive_flush_q.get()\n            if item is None:\n                self._drive_flush_q.task_done()\n                return\n            shard, expected_tar_bytes = item\n            try:\n                self._flush_shard_to_drive(shard, expected_tar_bytes)\n            finally:\n                self._drive_flush_q.task_done()\n\n    def _flush_shard_to_drive(self, shard: _StagingShard, expected_tar_bytes: int) -> None:\n        dest_tar_path = self.job.dest_root / shard.rel_tar_path\n        manifest_path = dest_tar_path.with_name(f"{dest_tar_path.name}.manifest.jsonl")\n        entries = shard.entries\n        last_err: Exception | None = None\n        for attempt in range(3):\n            tar_stage: Path | None = None\n            man_stage: Path | None = None\n            try:\n                dest_tar_path.parent.mkdir(parents=True, exist_ok=True)\n                tar_stage = dest_tar_path.with_name(f".{dest_tar_path.name}.ctpart.{os.getpid()}.{uuid.uuid4().hex[:6]}")\n                man_stage = manifest_path.with_name(f".{manifest_path.name}.ctpart.{os.getpid()}.{uuid.uuid4().hex[:6]}")\n                with shard.stage_tar_path.open("rb") as src_f, tar_stage.open("wb") as dst_f:\n                    shutil.copyfileobj(src_f, dst_f, length=8 * 1024 * 1024)\n                    dst_f.flush()\n                    with contextlib.suppress(OSError):\n                        os.fsync(dst_f.fileno())\n                if tar_stage.stat().st_size != expected_tar_bytes:\n                    raise OSError(\n                        f"Drive TAR stage size mismatch: expected {expected_tar_bytes}B, got {tar_stage.stat().st_size}B"\n                    )\n\n                with man_stage.open("w", encoding="utf-8") as mf:\n                    for e in entries:\n                        mf.write(\n                            json.dumps(\n                                {\n                                    "bucket": e.bucket,\n                                    "key": e.key,\n                                    "rel_path": e.orig_rel_path,\n                                    "archive_path": e.archive_path,\n                                    "size": e.size,\n                                    "is_dir": e.is_dir,\n                                    "source": e.source,\n                                },\n                                ensure_ascii=False,\n                            )\n                            + "\\n"\n                        )\n                    mf.write(\n                        json.dumps(\n                            {\n                                "__shard_meta__": True,\n                                "bucket": shard.bucket,\n                                "shard_name": shard.shard_name,\n                                "rel_tar_path": shard.rel_tar_path,\n                                "file_count": len(entries),\n                                "payload_bytes": shard.payload_bytes,\n                                "tar_bytes": expected_tar_bytes,\n                            },\n                            ensure_ascii=False,\n                        )\n                        + "\\n"\n                    )\n                    mf.flush()\n                    with contextlib.suppress(OSError):\n                        os.fsync(mf.fileno())\n\n                os.replace(man_stage, manifest_path)\n                man_stage = None\n                os.replace(tar_stage, dest_tar_path)\n                tar_stage = None\n                actual_sz = dest_tar_path.stat().st_size\n                if actual_sz != expected_tar_bytes:\n                    raise OSError(f"Drive TAR verify failed: expected {expected_tar_bytes}B, got {actual_sz}B")\n                last_err = None\n                break\n            except Exception as exc:\n                last_err = exc\n                for tmp in (tar_stage, man_stage):\n                    if tmp is not None:\n                        with contextlib.suppress(OSError):\n                            tmp.unlink(missing_ok=True)\n                time.sleep(1.0 * (attempt + 1))\n\n        with contextlib.suppress(OSError):\n            shard.stage_tar_path.unlink(missing_ok=True)\n\n        if last_err is not None:\n            self._dec_inflight(len(entries))\n            if (isinstance(last_err, OSError) and last_err.errno == errno.ENOSPC) or "No space left on device" in str(last_err):\n                if self._on_quota_pause is not None:\n                    self._on_quota_pause(\n                        "Google Drive 挂载或本地缓存空间不足（可能已触发单账号 750 GB / 24h 上传限流 userRateLimitExceeded）；"\n                        "任务已安全暂停，待云端后台上传释放缓存后执行 resume 继续"\n                    )\n                    self._on_failed(entries, "")\n                    return\n            self._on_failed(entries, f"Drive TAR flush failed: {last_err}")\n            return\n\n        # Commit shard + archived_files + objects in a single SQLite transaction BEFORE unlinking loose files.\n        deltas, cons_cnt, cons_bytes, sess_cnt, sess_bytes = self._commit_shard_to_db(\n            shard, expected_tar_bytes\n        )\n        self._dec_inflight(len(entries))\n        self._on_committed(\n            entries,\n            expected_tar_bytes,\n            deltas,\n            cons_cnt,\n            cons_bytes,\n            sess_cnt,\n            sess_bytes,\n        )\n\n        # Unlink consolidated loose source files ONLY after Drive flush & SQLite commit succeeded!\n        loose_files = [e.loose_path for e in entries if e.source == "drive_existing" and not e.is_dir and e.loose_path is not None]\n        if loose_files:\n            def _unlink_one(p: Path) -> None:\n                with contextlib.suppress(OSError):\n                    p.unlink(missing_ok=True)\n\n            list(self._loose_unlink_pool.map(_unlink_one, loose_files))\n        if any(e.source == "drive_existing" for e in entries):\n            with self.job.db_lock:\n                self.conn.execute("UPDATE archived_files SET loose_cleaned = 1 WHERE shard_name = ?", (shard.shard_name,))\n                self.conn.commit()\n\n    def _commit_shard_to_db(\n        self, shard: _StagingShard, expected_tar_bytes: int\n    ) -> tuple[dict[str, tuple[int, int]], int, int, int, int]:\n        entries = shard.entries\n        now_ts = time.time()\n        deltas: dict[str, tuple[int, int]] = {}\n        cons_cnt = 0\n        cons_bytes = 0\n        sess_cnt = 0\n        sess_bytes = 0\n\n        by_bucket: dict[str, list[_ArchivedEntry]] = {}\n        for e in entries:\n            if e.bucket and e.key:\n                by_bucket.setdefault(e.bucket, []).append(e)\n            if e.source == "drive_existing":\n                cons_cnt += 1\n                cons_bytes += int(e.size)\n            sess_cnt += 1\n            sess_bytes += int(e.size)\n\n        with self.job.db_lock:\n            self.conn.execute(\n                """\n                INSERT OR IGNORE INTO tar_shards\n                (bucket, shard_name, rel_tar_path, file_count, payload_bytes, tar_bytes, created_at)\n                VALUES (?, ?, ?, ?, ?, ?, ?)\n                """,\n                (\n                    shard.bucket,\n                    shard.shard_name,\n                    shard.rel_tar_path,\n                    len(entries),\n                    shard.payload_bytes,\n                    expected_tar_bytes,\n                    now_ts,\n                ),\n            )\n            self.conn.executemany(\n                """\n                INSERT OR REPLACE INTO archived_files\n                (rel_path, bucket, key, size, shard_name, source, loose_cleaned, archived_at)\n                VALUES (?, ?, ?, ?, ?, ?, ?, ?)\n                """,\n                [\n                    (\n                        e.archive_path,\n                        e.bucket,\n                        e.key,\n                        int(e.size),\n                        shard.shard_name,\n                        e.source,\n                        0 if e.source == "drive_existing" else 1,\n                        now_ts,\n                    )\n                    for e in entries\n                ],\n            )\n            obj_updates: list[tuple[str, str, float, str, str]] = []\n            rel_only_updates: list[tuple[str, float, str, str]] = []\n            for b, b_entries in by_bucket.items():\n                existing_state: dict[str, tuple[str, int, str]] = {}\n                for idx in range(0, len(b_entries), 400):\n                    chunk_keys = [x.key for x in b_entries[idx : idx + 400]]\n                    q_marks = ",".join("?" for _ in chunk_keys)\n                    rows = self.conn.execute(\n                        f"SELECT key, status, size, rel_path FROM objects WHERE bucket = ? AND key IN ({q_marks})",\n                        [b, *chunk_keys],\n                    ).fetchall()\n                    for r_key, r_st, r_sz, r_rel in rows:\n                        existing_state[str(r_key)] = (str(r_st), int(r_sz), str(r_rel))\n\n                for e in b_entries:\n                    cur = existing_state.get(e.key)\n                    if cur is None:\n                        continue\n                    cur_st, cur_sz, cur_rel = cur\n                    if cur_st in ("pending", "ia_deferred", "failed"):\n                        obj_updates.append((e.archive_path, "copied", now_ts, b, e.key))\n                        dc, db = deltas.get(cur_st, (0, 0))\n                        deltas[cur_st] = (dc - 1, db - cur_sz)\n                        cc, cb = deltas.get("copied", (0, 0))\n                        deltas["copied"] = (cc + 1, cb + cur_sz)\n                    elif e.archive_path != cur_rel:\n                        # Preserve copied/deleted status, update rel_path if relocated.\n                        rel_only_updates.append((e.archive_path, now_ts, b, e.key))\n            if obj_updates:\n                self.conn.executemany(\n                    "UPDATE objects SET rel_path=?, status=?, error=\'\', updated_at=? WHERE bucket=? AND key=?",\n                    obj_updates,\n                )\n            if rel_only_updates:\n                self.conn.executemany(\n                    "UPDATE objects SET rel_path=?, updated_at=? WHERE bucket=? AND key=?",\n                    rel_only_updates,\n                )\n            self.conn.commit()\n        return deltas, cons_cnt, cons_bytes, sess_cnt, sess_bytes\n\n    def reconcile_drive_manifests(self) -> int:\n        """Recover shards written to Drive before SQLite checkpoint was persisted."""\n        if not self.archives_dir.is_dir():\n            return 0\n        for p in list(self.archives_dir.rglob("*")):\n            if p.is_file() and _is_temp_artifact_name(p.name):\n                with contextlib.suppress(OSError):\n                    p.unlink(missing_ok=True)\n        with self.job.db_lock:\n            known_shards = {\n                str(r[0]) for r in self.conn.execute("SELECT shard_name FROM tar_shards").fetchall()\n            }\n        replayed = 0\n        for man_path in sorted(self.archives_dir.rglob("*.tar.manifest.jsonl")):\n            if _is_temp_artifact_name(man_path.name):\n                continue\n            try:\n                lines = [ln for ln in man_path.read_text(encoding="utf-8").splitlines() if ln.strip()]\n                if not lines:\n                    continue\n                meta = json.loads(lines[-1])\n                if not isinstance(meta, dict) or not meta.get("__shard_meta__"):\n                    continue\n                shard_name = str(meta.get("shard_name") or "")\n                if not shard_name or shard_name in known_shards:\n                    continue\n                rel_tar_path = str(meta["rel_tar_path"])\n                tar_path = self.job.dest_root / rel_tar_path\n                expected_tar_bytes = int(meta["tar_bytes"])\n                if not tar_path.is_file() or tar_path.stat().st_size != expected_tar_bytes:\n                    continue\n                items = [json.loads(ln) for ln in lines[:-1]]\n                shard_bucket = str(meta.get("bucket") or "drive-existing")\n                entries = [\n                    _ArchivedEntry(\n                        bucket=str(it.get("bucket") or ""),\n                        key=str(it.get("key") or ""),\n                        orig_rel_path=str(it.get("rel_path") or it.get("archive_path") or ""),\n                        archive_path=str(it.get("archive_path") or ""),\n                        size=int(it.get("size") or 0),\n                        is_dir=bool(it.get("is_dir", False)),\n                        source=str(it.get("source") or "r2_download"),\n                    )\n                    for it in items\n                    if isinstance(it, dict)\n                ]\n                dummy = _StagingShard(\n                    bucket=shard_bucket,\n                    safe_bucket=sanitize_segment(shard_bucket),\n                    shard_name=shard_name,\n                    rel_tar_path=rel_tar_path,\n                    stage_tar_path=tar_path,\n                    raw_fh=None,  # type: ignore[arg-type]\n                    tar=None,  # type: ignore[arg-type]\n                    entries=entries,\n                    payload_bytes=int(meta.get("payload_bytes") or 0),\n                    file_count=len(entries),\n                )\n                self._commit_shard_to_db(dummy, expected_tar_bytes)\n                known_shards.add(shard_name)\n                replayed += 1\n            except Exception as exc:\n                self._log(f"job {self.job.job_id}: manifest reconcile skipped {man_path.name}: {exc}")\n        if replayed:\n            self._log(f"job {self.job.job_id}: reconciled {replayed} TAR shards from Drive manifests")\n        return replayed\n\n    def reconcile_missing_drive_shards(self) -> int:\n        """Reset objects from shards recorded in SQLite whose Drive .tar file is missing or size-mismatched."""\n        with self.job.db_lock:\n            rows = self.conn.execute("SELECT shard_name, rel_tar_path, tar_bytes FROM tar_shards").fetchall()\n        missing: list[tuple[str, Path]] = []\n        for sname, rel_tar, expected_sz in rows:\n            tar_path = self.job.dest_root / str(rel_tar)\n            exists_ok = False\n            with contextlib.suppress(OSError):\n                exists_ok = tar_path.is_file() and tar_path.stat().st_size == int(expected_sz)\n            if not exists_ok:\n                missing.append((str(sname), tar_path))\n        if not missing:\n            return 0\n        reset_objs = 0\n        for sname, tar_path in missing:\n            with contextlib.suppress(OSError):\n                tar_path.unlink(missing_ok=True)\n            with contextlib.suppress(OSError):\n                tar_path.with_name(f"{tar_path.name}.manifest.jsonl").unlink(missing_ok=True)\n            with self.job.db_lock:\n                cur = self.conn.execute(\n                    """\n                    UPDATE objects SET status=\'pending\', error=\'\'\n                    WHERE status=\'copied\'\n                      AND (bucket, key) IN (\n                          SELECT bucket, key FROM archived_files\n                          WHERE shard_name=? AND bucket != \'\' AND key != \'\'\n                      )\n                    """,\n                    (sname,),\n                )\n                reset_objs += max(0, int(cur.rowcount or 0))\n                self.conn.execute("DELETE FROM archived_files WHERE shard_name=?", (sname,))\n                self.conn.execute("DELETE FROM tar_shards WHERE shard_name=?", (sname,))\n                self.conn.commit()\n        self._log(\n            f"job {self.job.job_id}: reset {len(missing)} missing/incomplete TAR shards ({reset_objs} objects back to pending)"\n        )\n        return len(missing)\n\n    def heal_uncleaned_loose_files(self) -> None:\n        """Finish or roll back shards whose loose Drive source cleanup was interrupted by crash."""\n        with self.job.db_lock:\n            rows = self.conn.execute(\n                """\n                SELECT a.rel_path, a.is_dir, a.shard_name, t.rel_tar_path, t.tar_bytes\n                FROM (\n                    SELECT af.rel_path, COALESCE(o.is_dir, 0) AS is_dir, af.shard_name\n                    FROM archived_files af\n                    LEFT JOIN objects o ON af.bucket = o.bucket AND af.key = o.key\n                    WHERE af.loose_cleaned = 0\n                ) a\n                JOIN tar_shards t ON a.shard_name = t.shard_name\n                """\n            ).fetchall()\n        if not rows:\n            with self.job.db_lock:\n                orphan_rows = self.conn.execute(\n                    "SELECT COUNT(*) FROM archived_files WHERE loose_cleaned = 0"\n                ).fetchone()[0]\n                if orphan_rows:\n                    self.conn.execute(\n                        "DELETE FROM archived_files WHERE loose_cleaned = 0 AND shard_name NOT IN (SELECT shard_name FROM tar_shards)"\n                    )\n                    self.conn.commit()\n            return\n\n        by_shard: dict[str, tuple[str, int, list[tuple[str, bool]]]] = {}\n        for rpath, is_dir_val, sname, rel_tar, tbytes in rows:\n            entry = by_shard.setdefault(str(sname), (str(rel_tar), int(tbytes), []))\n            entry[2].append((str(rpath), bool(is_dir_val)))\n\n        for sname, (rel_tar, expected_sz, items) in by_shard.items():\n            tar_path = self.job.dest_root / rel_tar\n            verified = False\n            with contextlib.suppress(OSError):\n                verified = tar_path.is_file() and tar_path.stat().st_size == expected_sz\n            if verified:\n                files_to_unlink: list[Path] = []\n                for rp, is_d in items:\n                    if is_d:\n                        continue\n                    files_to_unlink.append(self.job.dest_root / rp)\n                    if rp.endswith("/__r2_object__"):\n                        files_to_unlink.append(self.job.dest_root / rp[: -len("/__r2_object__")])\n                if files_to_unlink:\n                    def _rm(p: Path) -> None:\n                        with contextlib.suppress(OSError):\n                            if p.is_file():\n                                p.unlink(missing_ok=True)\n\n                    list(self._loose_unlink_pool.map(_rm, files_to_unlink))\n                with self.job.db_lock:\n                    self.conn.execute("UPDATE archived_files SET loose_cleaned = 1 WHERE shard_name = ?", (sname,))\n                    self.conn.commit()\n            else:\n                with self.job.db_lock:\n                    self.conn.execute("DELETE FROM archived_files WHERE shard_name = ?", (sname,))\n                    self.conn.execute("DELETE FROM tar_shards WHERE shard_name = ?", (sname,))\n                    self.conn.commit()\n                self._log(f"job {self.job.job_id}: dropped unverified shard {sname}; loose files preserved")\n\n\nclass MigrationManager:\n    """Coordinates resumable multi-bucket R2 -> Google Drive migration jobs."""\n\n    def __init__(\n        self,\n        *,\n        data_dir: Path,\n        work_dir: Path,\n        drive_mount: Path = Path("/content/drive/MyDrive"),\n        default_workers: int = DEFAULT_WORKERS,\n        default_writer_workers: int = DEFAULT_WRITER_WORKERS,\n        spool_bytes_cap: int = DEFAULT_SPOOL_BYTES,\n        checkpoint_interval: float = DEFAULT_CHECKPOINT_INTERVAL,\n        page_size: int = DEFAULT_PAGE_SIZE,\n        client_timeout: float = 60.0,\n        log: Callable[[str], None] | None = None,\n    ) -> None:\n        self.data_dir = Path(data_dir)\n        self.work_dir = Path(work_dir)\n        self.drive_mount = Path(drive_mount)\n        self.jobs_drive_dir = self.data_dir / "jobs"\n        self.state_dir = self.work_dir / "state"\n        self.spool_root = self.work_dir / "spool"\n        self.jobs_drive_dir.mkdir(parents=True, exist_ok=True)\n        self.state_dir.mkdir(parents=True, exist_ok=True)\n        self.spool_root.mkdir(parents=True, exist_ok=True)\n\n        self.default_workers = max(1, int(default_workers))\n        self.default_writer_workers = max(1, int(default_writer_workers))\n        self.spool_bytes_cap = max(64 * 1024 * 1024, int(spool_bytes_cap))\n        self.checkpoint_interval = max(0.5, float(checkpoint_interval))\n        self.page_size = max(1, min(1000, int(page_size)))\n        self.client_timeout = float(client_timeout)\n        self._log = log or (lambda msg: print(f"[r2-migrate] {msg}", file=sys.stderr, flush=True))\n\n        self.ledger = R2UsageLedger(self.data_dir)\n        self.cred_store = CredentialStore(self.data_dir)\n        self._mgr_lock = threading.Lock()\n        self._jobs: dict[str, _RuntimeJob] = {}\n        self._shutdown = False\n        self._load_existing_jobs()\n\n    def _compute_scope_hash(self, config: dict[str, Any]) -> str:\n        canonical = json.dumps(\n            {\n                "endpoint_scope": config["endpoint_scope"],\n                "buckets": sorted(config.get("buckets") or []),\n                "exclude_buckets": sorted(config.get("exclude_buckets") or []),\n                "prefix": config.get("prefix") or "",\n                "dest_root": config["dest_root"],\n                "flatten_single_bucket": bool(config.get("flatten_single_bucket", False)),\n            },\n            sort_keys=True,\n        )\n        return hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:24]\n\n    def _job_json_path(self, job: _RuntimeJob) -> Path:\n        return job.drive_job_dir / "job.json"\n\n    def _job_checkpoint_path(self, job: _RuntimeJob) -> Path:\n        return job.drive_job_dir / "checkpoint.sqlite3"\n\n    def _ensure_job_db_open(self, job: _RuntimeJob) -> sqlite3.Connection:\n        with job.db_lock:\n            if job.conn is not None:\n                return job.conn\n            if not job.db_path.is_file():\n                _restore_sqlite_from_drive(self._job_checkpoint_path(job), job.db_path)\n            conn = _open_state_db(job.db_path)\n            job.conn = conn\n            self._refresh_counts_locked(job, conn)\n            self._refresh_buckets_locked(job, conn)\n            return conn\n\n    def _adjust_status_counts(self, job: _RuntimeJob, deltas: dict[str, tuple[int, int]]) -> None:\n        if not deltas:\n            return\n        with job.lock:\n            for st_key, (dc, db) in deltas.items():\n                if dc == 0 and db == 0:\n                    continue\n                cur = job.status_counts.setdefault(st_key, {"count": 0, "bytes": 0})\n                cur["count"] = max(0, int(cur["count"]) + int(dc))\n                cur["bytes"] = max(0, int(cur["bytes"]) + int(db))\n\n    def _refresh_counts_locked(self, job: _RuntimeJob, conn: sqlite3.Connection) -> None:\n        counts: dict[str, dict[str, int]] = {}\n        for status, cnt, total_sz in conn.execute(\n            "SELECT status, COUNT(*), COALESCE(SUM(size), 0) FROM objects GROUP BY status"\n        ).fetchall():\n            counts[str(status)] = {"count": int(cnt), "bytes": int(total_sz)}\n        shard_row = conn.execute("SELECT COUNT(*), COALESCE(SUM(tar_bytes), 0) FROM tar_shards").fetchone()\n        cons_row = conn.execute(\n            "SELECT COUNT(*), COALESCE(SUM(size), 0) FROM archived_files WHERE source = \'drive_existing\'"\n        ).fetchone()\n        now_m = time.monotonic()\n        with job.lock:\n            job.status_counts = counts\n            job.tar_shards_count = int(shard_row[0]) if shard_row else 0\n            job.tar_bytes_total = int(shard_row[1]) if shard_row else 0\n            job.consolidated_files = int(cons_row[0]) if cons_row else 0\n            job.consolidated_bytes = int(cons_row[1]) if cons_row else 0\n            job.last_db_refresh_mono = now_m\n\n    def _refresh_buckets_locked(self, job: _RuntimeJob, conn: sqlite3.Connection) -> None:\n        b_rows = conn.execute(\n            "SELECT bucket, last_key, listing_done, objects_listed, bytes_listed FROM buckets ORDER BY bucket"\n        ).fetchall()\n        by_bucket: dict[str, dict[str, int]] = {}\n        for status_val, b_name, cnt in conn.execute(\n            "SELECT status, bucket, COUNT(*) FROM objects GROUP BY status, bucket"\n        ).fetchall():\n            by_bucket.setdefault(str(b_name), {})[str(status_val)] = int(cnt)\n        snap = []\n        for r in b_rows:\n            b_name = str(r[0])\n            st_map = by_bucket.get(b_name, {})\n            copied_c = st_map.get("copied", 0)\n            deleted_c = st_map.get("deleted", 0)\n            snap.append(\n                {\n                    "bucket": b_name,\n                    "last_key": r[1],\n                    "listing_done": bool(r[2]),\n                    "objects_listed": int(r[3]),\n                    "bytes_listed": int(r[4]),\n                    "objects_migrated": copied_c + deleted_c,\n                    "objects_deleted": deleted_c,\n                    "objects_failed": st_map.get("failed", 0),\n                    "objects_ia_deferred": st_map.get("ia_deferred", 0),\n                }\n            )\n        with job.lock:\n            job.buckets_snapshot = snap\n\n    def _snapshot_job(self, job: _RuntimeJob) -> dict[str, Any]:\n        with job.lock:\n            counts = {\n                k: dict(v)\n                for k, v in job.status_counts.items()\n            }\n            pending = counts.get("pending", {}).get("count", 0)\n            ia_deferred = counts.get("ia_deferred", {}).get("count", 0)\n            copied = counts.get("copied", {}).get("count", 0)\n            deleted = counts.get("deleted", {}).get("count", 0)\n            failed = counts.get("failed", {}).get("count", 0)\n            listed = pending + ia_deferred + copied + deleted + failed\n            migrated = copied + deleted\n            bytes_listed = sum(v.get("bytes", 0) for v in counts.values())\n            bytes_migrated = counts.get("copied", {}).get("bytes", 0) + counts.get("deleted", {}).get("bytes", 0)\n\n            now = time.time()\n            elapsed = max(0.001, (job.finished_at or now) - (job.started_at or job.created_at))\n            if job.session_started_at is not None:\n                active_elapsed = max(0.001, (job.session_finished_at or now) - job.session_started_at)\n            elif job.started_at:\n                active_elapsed = max(0.001, (job.finished_at or now) - job.started_at)\n            else:\n                active_elapsed = elapsed\n            files_per_sec = round(job.files_done_session / active_elapsed, 2) if job.files_done_session else 0.0\n            mb_per_sec = round((job.bytes_done_session / (1024 * 1024)) / active_elapsed, 2) if job.bytes_done_session else 0.0\n\n            cfg_safe = {k: v for k, v in job.config.items() if "secret" not in k.lower()}\n            return {\n                "id": job.job_id,\n                "status": job.status,\n                "message": job.message,\n                "paid_fallback_used": job.paid_fallback_used,\n                "resumed_from_checkpoint": bool(job.config.get("resumed_from_checkpoint", False)),\n                "config": cfg_safe,\n                "created_at": job.created_at,\n                "started_at": job.started_at,\n                "finished_at": job.finished_at,\n                "updated_at": job.updated_at,\n                "progress": {\n                    "objects_listed": listed,\n                    "objects_migrated": migrated,\n                    "objects_copied_pending_delete": copied,\n                    "objects_deleted": deleted,\n                    "objects_failed": failed,\n                    "objects_pending": pending,\n                    "objects_ia_deferred": ia_deferred,\n                    "bytes_listed": bytes_listed,\n                    "bytes_migrated": bytes_migrated,\n                    "files_per_sec": files_per_sec,\n                    "mb_per_sec": mb_per_sec,\n                    "storage_mode": str(job.config.get("storage_mode") or "files"),\n                    "tar_shards": job.tar_shards_count,\n                    "tar_bytes": job.tar_bytes_total,\n                    "consolidated_existing_files": job.consolidated_files,\n                    "consolidated_existing_bytes": job.consolidated_bytes,\n                },\n                "buckets": list(job.buckets_snapshot),\n                "recent_errors": list(job.recent_errors),\n                "checkpoint_path": str(self._job_checkpoint_path(job)),\n                "usage": self.ledger.snapshot(),\n            }\n\n    def _persist_job_summary(self, job: _RuntimeJob) -> None:\n        snap = self._snapshot_job(job)\n        # Keep stored job.json concise without volatile full usage duplicate if desired; including usage is nice for inspection.\n        _atomic_write_json(self._job_json_path(job), snap)\n\n    def _flush_checkpoint(self, job: _RuntimeJob, conn: sqlite3.Connection) -> None:\n        with job.db_lock:\n            self._refresh_counts_locked(job, conn)\n            self._refresh_buckets_locked(job, conn)\n            local_tmp = self.state_dir / f".{job.job_id}.backup.tmp"\n            try:\n                _backup_sqlite_to_drive(conn, local_tmp, self._job_checkpoint_path(job), db_path=job.db_path)\n            except OSError as exc:\n                self._log(f"job {job.job_id}: checkpoint backup to Drive deferred: {exc}")\n        with job.lock:\n            job.updated_at = time.time()\n        try:\n            self._persist_job_summary(job)\n        except OSError as exc:\n            self._log(f"job {job.job_id}: job.json summary persist deferred: {exc}")\n\n    def _load_existing_jobs(self) -> None:\n        if not self.jobs_drive_dir.is_dir():\n            return\n        for job_dir in sorted(self.jobs_drive_dir.iterdir()):\n            if not job_dir.is_dir():\n                continue\n            jpath = job_dir / "job.json"\n            if not jpath.is_file():\n                continue\n            try:\n                data = json.loads(jpath.read_text(encoding="utf-8"))\n                cfg = dict(data.get("config") or {})\n                job_id = str(data.get("id") or job_dir.name)\n                dest_root = Path(str(cfg.get("dest_root") or (self.drive_mount / "r2-backup")))\n                status = str(data.get("status") or "interrupted")\n                if status in ("queued", "running"):\n                    status = "interrupted"\n                prog = data.get("progress") or {}\n                job = _RuntimeJob(\n                    job_id=job_id,\n                    config=cfg,\n                    db_path=self.state_dir / f"{job_id}.sqlite3",\n                    drive_job_dir=job_dir,\n                    spool_dir=self.spool_root / job_id,\n                    dest_root=dest_root,\n                    status=status,\n                    message=str(data.get("message") or ""),\n                    paid_fallback_used=bool(data.get("paid_fallback_used", False)),\n                    created_at=float(data.get("created_at") or time.time()),\n                    started_at=data.get("started_at"),\n                    finished_at=data.get("finished_at"),\n                    updated_at=float(data.get("updated_at") or time.time()),\n                    tar_shards_count=int(prog.get("tar_shards", 0) or 0),\n                    tar_bytes_total=int(prog.get("tar_bytes", 0) or 0),\n                    consolidated_files=int(prog.get("consolidated_existing_files", 0) or 0),\n                    consolidated_bytes=int(prog.get("consolidated_existing_bytes", 0) or 0),\n                    buckets_snapshot=list(data.get("buckets") or []),\n                )\n                job.status_counts = {\n                    "pending": {"count": int(prog.get("objects_pending", 0)), "bytes": 0},\n                    "ia_deferred": {"count": int(prog.get("objects_ia_deferred", 0)), "bytes": 0},\n                    "copied": {"count": int(prog.get("objects_copied_pending_delete", 0)), "bytes": 0},\n                    "deleted": {"count": int(prog.get("objects_deleted", 0)), "bytes": 0},\n                    "failed": {"count": int(prog.get("objects_failed", 0)), "bytes": 0},\n                }\n                for err_item in data.get("recent_errors") or []:\n                    job.recent_errors.append(dict(err_item))\n                self._jobs[job_id] = job\n                if job.db_path.is_file():\n                    with contextlib.suppress(sqlite3.Error, OSError):\n                        self._ensure_job_db_open(job)\n            except (OSError, ValueError, TypeError):\n                continue\n\n    def recover_on_startup(self) -> list[str]:\n        """Auto-resume jobs that were running when a previous Colab VM died if credentials exist."""\n        resumed: list[str] = []\n        creds, _ = self.cred_store.resolve(None)\n        with self._mgr_lock:\n            candidates = [j for j in self._jobs.values() if j.status == "interrupted"]\n        for job in candidates:\n            if creds is not None and not self._shutdown:\n                self._log(f"Restoring interrupted migration job {job.job_id} from Drive checkpoint")\n                self.resume_job(job.job_id, credentials=creds)\n                resumed.append(job.job_id)\n            else:\n                with job.lock:\n                    job.status = "paused_credentials"\n                    job.message = f"Colab VM restarted; supply R2 credentials and call resume {job.job_id} to continue from checkpoint"\n                    job.updated_at = time.time()\n                self._persist_job_summary(job)\n        return resumed\n\n    def active_jobs_count(self) -> int:\n        with self._mgr_lock:\n            return sum(1 for j in self._jobs.values() if j.status in ("queued", "running"))\n\n    def list_jobs(self) -> list[dict[str, Any]]:\n        with self._mgr_lock:\n            jobs = list(self._jobs.values())\n        jobs.sort(key=lambda j: j.created_at, reverse=True)\n        return [self._snapshot_job(j) for j in jobs]\n\n    def get_job(self, job_id: str) -> dict[str, Any]:\n        job = self._require_job(job_id)\n        if (\n            job.conn is not None\n            and job.status not in ("queued", "running")\n            and (time.monotonic() - job.last_db_refresh_mono) > 5.0\n        ):\n            with job.db_lock:\n                self._refresh_counts_locked(job, job.conn)\n                self._refresh_buckets_locked(job, job.conn)\n        return self._snapshot_job(job)\n\n    def _require_job(self, job_id: str) -> _RuntimeJob:\n        with self._mgr_lock:\n            job = self._jobs.get(job_id)\n        if job is None:\n            raise KeyError(f"未找到迁移任务: {job_id}")\n        return job\n\n    def list_buckets(self, credentials: dict[str, Any] | R2Credentials | None = None) -> list[dict[str, str]]:\n        creds, _ = self.cred_store.resolve(credentials)\n        if creds is None:\n            raise ValueError("缺少 R2 凭据；请先通过 auth 保存或传入 credentials")\n        client = R2Client(creds, ledger=self.ledger, timeout=self.client_timeout)\n        try:\n            return client.list_buckets()\n        finally:\n            client.close()\n            self.ledger.flush(ignore_errors=True)\n\n    def create_or_resume_job(\n        self,\n        *,\n        buckets: list[str] | None = None,\n        exclude_buckets: list[str] | None = None,\n        prefix: str = "",\n        output_dir: str | None = None,\n        flatten_single_bucket: bool = False,\n        delete_source: bool = False,\n        cost_mode: str = "free-first",\n        public_urls: dict[str, str] | list[str] | None = None,\n        max_free_class_a: int = DEFAULT_SAFE_FREE_CLASS_A,\n        max_free_class_b: int = DEFAULT_SAFE_FREE_CLASS_B,\n        workers: int | None = None,\n        writer_workers: int | None = None,\n        overwrite: bool = False,\n        storage_mode: str | None = None,\n        tar_shard_mb: int | float | None = None,\n        consolidate_existing: bool | None = None,\n        new_job: bool = False,\n        credentials: dict[str, Any] | R2Credentials | None = None,\n    ) -> dict[str, Any]:\n        if cost_mode not in COST_MODES:\n            raise ValueError(f"无效 cost_mode: {cost_mode!r} (可选: {\', \'.join(COST_MODES)})")\n        if storage_mode is not None and storage_mode not in STORAGE_MODES:\n            raise ValueError(f"无效 storage_mode: {storage_mode!r} (可选: {\', \'.join(STORAGE_MODES)})")\n        if tar_shard_mb is not None and float(tar_shard_mb) <= 0:\n            raise ValueError(f"tar_shard_mb 必须大于 0: {tar_shard_mb!r}")\n        creds, _ = self.cred_store.resolve(credentials)\n        if creds is None:\n            raise ValueError("缺少 R2 凭据；请先运行 `colabllm r2-migrate auth` 或在请求中提供凭据")\n        public_map = parse_public_urls(public_urls)\n        fallback_outputs = self.data_dir / "outputs"\n        dest_root = resolve_drive_output_root(\n            self.drive_mount,\n            output_dir,\n            default_subdir="r2-backup",\n            fallback_root=fallback_outputs,\n        )\n        cfg = {\n            "endpoint_scope": creds.scope_key(),\n            "buckets": [b.strip() for b in (buckets or []) if str(b).strip()],\n            "exclude_buckets": [b.strip() for b in (exclude_buckets or []) if str(b).strip()],\n            "prefix": str(prefix or ""),\n            "output_dir": output_dir or "r2-backup",\n            "dest_root": str(dest_root),\n            "flatten_single_bucket": bool(flatten_single_bucket),\n            "delete_source": bool(delete_source),\n            "cost_mode": cost_mode,\n            "public_urls": public_map,\n            "max_free_class_a": max(0, int(max_free_class_a)),\n            "max_free_class_b": max(0, int(max_free_class_b)),\n            "workers": max(1, int(workers or self.default_workers)),\n            "writer_workers": max(1, int(writer_workers or self.default_writer_workers)),\n            "overwrite": bool(overwrite),\n            "storage_mode": storage_mode or "files",\n            "tar_shard_mb": max(1.0, float(tar_shard_mb if tar_shard_mb is not None else DEFAULT_TAR_SHARD_MB)),\n            "consolidate_existing": True if consolidate_existing is None else bool(consolidate_existing),\n            "resumed_from_checkpoint": False,\n        }\n        scope_hash = self._compute_scope_hash(cfg)\n        cfg["scope_hash"] = scope_hash\n\n        if not new_job:\n            with self._mgr_lock:\n                existing = [\n                    j\n                    for j in self._jobs.values()\n                    if j.config.get("scope_hash") == scope_hash and j.status in RESUMABLE_JOB_STATUSES\n                ]\n            if existing:\n                existing.sort(key=lambda j: j.created_at, reverse=True)\n                target = existing[0]\n                if target.status in ("queued", "running") and target.worker_thread and target.worker_thread.is_alive():\n                    return self._snapshot_job(target)\n                return self.resume_job(\n                    target.job_id,\n                    cost_mode=cost_mode,\n                    delete_source=delete_source,\n                    workers=workers,\n                    writer_workers=writer_workers,\n                    max_free_class_a=max_free_class_a,\n                    max_free_class_b=max_free_class_b,\n                    overwrite=overwrite,\n                    public_urls=public_map if public_urls is not None else None,\n                    storage_mode=storage_mode,\n                    tar_shard_mb=tar_shard_mb,\n                    consolidate_existing=consolidate_existing,\n                    credentials=creds,\n                )\n\n        job_id = "r2m_" + secrets.token_hex(6)\n        drive_job_dir = self.jobs_drive_dir / job_id\n        drive_job_dir.mkdir(parents=True, exist_ok=True)\n        job = _RuntimeJob(\n            job_id=job_id,\n            config=cfg,\n            db_path=self.state_dir / f"{job_id}.sqlite3",\n            drive_job_dir=drive_job_dir,\n            spool_dir=self.spool_root / job_id,\n            dest_root=dest_root,\n            status="queued",\n            message="已创建任务，准备同步",\n        )\n        with self._mgr_lock:\n            self._jobs[job_id] = job\n        self._persist_job_summary(job)\n        self._launch_job_thread(job, creds)\n        return self._snapshot_job(job)\n\n    def resume_job(\n        self,\n        job_id: str,\n        *,\n        cost_mode: str | None = None,\n        delete_source: bool | None = None,\n        workers: int | None = None,\n        writer_workers: int | None = None,\n        max_free_class_a: int | None = None,\n        max_free_class_b: int | None = None,\n        overwrite: bool | None = None,\n        public_urls: dict[str, str] | list[str] | None = None,\n        storage_mode: str | None = None,\n        tar_shard_mb: int | float | None = None,\n        consolidate_existing: bool | None = None,\n        credentials: dict[str, Any] | R2Credentials | None = None,\n    ) -> dict[str, Any]:\n        job = self._require_job(job_id)\n        if cost_mode is not None and cost_mode not in COST_MODES:\n            raise ValueError(f"无效 cost_mode: {cost_mode!r}")\n        if storage_mode is not None and storage_mode not in STORAGE_MODES:\n            raise ValueError(f"无效 storage_mode: {storage_mode!r}")\n        if tar_shard_mb is not None and float(tar_shard_mb) <= 0:\n            raise ValueError(f"tar_shard_mb 必须大于 0: {tar_shard_mb!r}")\n        creds, _ = self.cred_store.resolve(credentials)\n        if creds is None:\n            raise ValueError("缺少 R2 凭据；请先保存或传入凭据后再恢复任务")\n        public_map = parse_public_urls(public_urls) if public_urls is not None else None\n\n        with job.lock:\n            prev_th = job.worker_thread\n            prev_running = job.status in ("queued", "running")\n            if prev_th is not None and prev_th.is_alive() and prev_running:\n                return self._snapshot_job(job)\n        if prev_th is not None and prev_th.is_alive() and prev_th is not threading.current_thread():\n            prev_th.join(timeout=10.0)\n        with job.lock:\n            if job.worker_thread is not None and job.worker_thread.is_alive() and job.status in ("queued", "running"):\n                return self._snapshot_job(job)\n            job.stop_event.clear()\n            job.pause_requested = False\n            job.cancel_requested = False\n            if cost_mode is not None:\n                job.config["cost_mode"] = cost_mode\n            if delete_source is not None:\n                job.config["delete_source"] = bool(delete_source)\n            if workers is not None:\n                job.config["workers"] = max(1, int(workers))\n            if writer_workers is not None:\n                job.config["writer_workers"] = max(1, int(writer_workers))\n            if max_free_class_a is not None:\n                job.config["max_free_class_a"] = max(0, int(max_free_class_a))\n            if max_free_class_b is not None:\n                job.config["max_free_class_b"] = max(0, int(max_free_class_b))\n            if overwrite is not None:\n                job.config["overwrite"] = bool(overwrite)\n            if public_map is not None:\n                job.config["public_urls"] = public_map\n            if storage_mode is not None:\n                job.config["storage_mode"] = storage_mode\n            if tar_shard_mb is not None:\n                job.config["tar_shard_mb"] = max(1.0, float(tar_shard_mb))\n            if consolidate_existing is not None:\n                job.config["consolidate_existing"] = bool(consolidate_existing)\n            job.config["resumed_from_checkpoint"] = True\n            job.status = "queued"\n            job.finished_at = None\n            job.session_finished_at = None\n            job.message = "从检查点恢复任务"\n            job.updated_at = time.time()\n\n        conn = self._ensure_job_db_open(job)\n        with job.db_lock:\n            # Retry previously failed objects on explicit resume.\n            conn.execute("UPDATE objects SET status=\'pending\', error=\'\' WHERE status=\'failed\'")\n            if job.config.get("cost_mode") == "unlimited":\n                conn.execute("UPDATE objects SET status=\'pending\' WHERE status=\'ia_deferred\'")\n            conn.commit()\n            self._refresh_counts_locked(job, conn)\n        self._persist_job_summary(job)\n        self._launch_job_thread(job, creds)\n        return self._snapshot_job(job)\n\n    def pause_job(self, job_id: str, *, wait: bool = True) -> dict[str, Any]:\n        job = self._require_job(job_id)\n        with job.lock:\n            thread = job.worker_thread\n            if job.status in ("queued", "running"):\n                job.pause_requested = True\n                job.stop_event.set()\n                job.message = "正在安全暂停并保存检查点..."\n        if wait and thread is not None and thread.is_alive():\n            thread.join(timeout=60.0)\n        return self.get_job(job_id)\n\n    def cancel_job(self, job_id: str, *, wait: bool = True) -> dict[str, Any]:\n        job = self._require_job(job_id)\n        with job.lock:\n            thread = job.worker_thread\n            if job.status in ("queued", "running"):\n                job.cancel_requested = True\n                job.stop_event.set()\n                job.message = "正在取消任务并保存检查点..."\n            elif job.status not in ("completed",):\n                job.status = "cancelled"\n                job.finished_at = time.time()\n                job.session_finished_at = job.finished_at\n                job.message = "任务已取消"\n        if wait and thread is not None and thread.is_alive():\n            thread.join(timeout=60.0)\n        if job.status != "running":\n            self._persist_job_summary(job)\n        return self.get_job(job_id)\n\n    def _reconcile_missing_tar_shards(self, job: _RuntimeJob, conn: sqlite3.Connection) -> int:\n        """Reset objects belonging to tar_shards missing/incomplete on Drive back to pending before cleanup."""\n        with job.db_lock:\n            rows = conn.execute("SELECT shard_name, rel_tar_path, tar_bytes FROM tar_shards").fetchall()\n        if not rows:\n            return 0\n        missing: list[tuple[str, Path]] = []\n        for sname, rel_tar, expected_sz in rows:\n            tar_path = job.dest_root / str(rel_tar)\n            exists_ok = False\n            with contextlib.suppress(OSError):\n                exists_ok = tar_path.is_file() and tar_path.stat().st_size == int(expected_sz)\n            if not exists_ok:\n                missing.append((str(sname), tar_path))\n        if not missing:\n            return 0\n\n        reset_objs = 0\n        for sname, tar_path in missing:\n            with contextlib.suppress(OSError):\n                tar_path.unlink(missing_ok=True)\n            with contextlib.suppress(OSError):\n                tar_path.with_name(f"{tar_path.name}.manifest.jsonl").unlink(missing_ok=True)\n            with job.db_lock:\n                cur = conn.execute(\n                    """\n                    UPDATE objects SET status=\'pending\', error=\'\'\n                    WHERE status IN (\'copied\', \'failed\')\n                      AND (bucket, key) IN (\n                          SELECT bucket, key FROM archived_files\n                          WHERE shard_name=? AND bucket != \'\' AND key != \'\'\n                      )\n                    """,\n                    (sname,),\n                )\n                reset_objs += max(0, int(cur.rowcount or 0))\n                conn.execute("DELETE FROM archived_files WHERE shard_name=?", (sname,))\n                conn.execute("DELETE FROM tar_shards WHERE shard_name=?", (sname,))\n                conn.commit()\n        with job.db_lock:\n            shard_row = conn.execute("SELECT COUNT(*), COALESCE(SUM(tar_bytes), 0) FROM tar_shards").fetchone()\n            self._refresh_counts_locked(job, conn)\n            self._refresh_buckets_locked(job, conn)\n        with job.lock:\n            job.tar_shards_count = int(shard_row[0] or 0) if shard_row else 0\n            job.tar_bytes_total = int(shard_row[1] or 0) if shard_row else 0\n        self._log(\n            f"job {job.job_id}: reconciled {len(missing)} missing/incomplete TAR shards ({reset_objs} objects back to pending)"\n        )\n        return len(missing)\n\n    def cleanup_job(\n        self,\n        job_id: str,\n        *,\n        credentials: dict[str, Any] | R2Credentials | None = None,\n        background: bool = False,\n    ) -> dict[str, Any]:\n        """Verify Drive copies (including Cloud sync on fuse.drive) and batch-delete R2 originals (Free DeleteObjects)."""\n        job = self._require_job(job_id)\n        with job.lock:\n            prev_th = job.worker_thread\n            prev_running = job.status in ("queued", "running")\n        if prev_th is not None and prev_th.is_alive():\n            if prev_running:\n                raise ValueError("任务仍在运行，请先暂停或等待完成后再执行 cleanup")\n            if prev_th is not threading.current_thread():\n                prev_th.join(timeout=10.0)\n        with job.lock:\n            if (job.worker_thread is not None and job.worker_thread.is_alive()) or job.status in ("queued", "running"):\n                raise ValueError("任务仍在运行，请先暂停或等待完成后再执行 cleanup")\n            prev_status = job.status\n            job.stop_event.clear()\n            job.pause_requested = False\n            job.cancel_requested = False\n            job.status = "running"\n            job.finished_at = None\n            job.files_done_session = 0\n            job.bytes_done_session = 0\n            job.session_started_at = time.time()\n            job.session_finished_at = None\n            job.message = "正在核验 Drive（含云端同步状态）并批量清理 R2 原文件..."\n            job.updated_at = time.time()\n        creds, _ = self.cred_store.resolve(credentials)\n        if creds is None:\n            with job.lock:\n                job.status = prev_status\n            raise ValueError("缺少 R2 凭据；请提供或先保存凭据")\n\n        conn = self._ensure_job_db_open(job)\n        with contextlib.suppress(OSError):\n            self._persist_job_summary(job)\n\n        if background:\n            th = threading.Thread(\n                target=self._run_cleanup_wrapper,\n                args=(job, conn, creds, prev_status),\n                name=f"r2-cleanup-{job.job_id}",\n                daemon=True,\n            )\n            with job.lock:\n                job.worker_thread = th\n            th.start()\n            return self._snapshot_job(job)\n\n        return self._run_cleanup_wrapper(job, conn, creds, prev_status)\n\n    def _run_cleanup_wrapper(\n        self,\n        job: _RuntimeJob,\n        conn: sqlite3.Connection,\n        creds: R2Credentials,\n        prev_status: str,\n    ) -> dict[str, Any]:\n        client = R2Client(creds, ledger=self.ledger, timeout=self.client_timeout)\n        deleted_total = 0\n        failed_total = 0\n        deferred_total = 0\n        verifier = _DriveCloudSyncVerifier(job.dest_root, log=self._log)\n        after_keys: dict[str, str] = {}\n        skip_buckets: set[str] = set()\n        transient_bucket_errors: dict[str, str] = {}\n        last_log_mono = time.monotonic()\n        try:\n            if job.config.get("storage_mode") == "tar":\n                self._reconcile_missing_tar_shards(job, conn)\n            while not job.stop_event.is_set() and not self._shutdown:\n                d_cnt, f_cnt, s_cnt = self._delete_copied_chunk(\n                    job,\n                    conn,\n                    client,\n                    batch_limit=10000,\n                    after_keys=after_keys,\n                    single_pass=True,\n                    cloud_verifier=verifier,\n                    delete_concurrency=32,\n                    skip_buckets=skip_buckets,\n                    transient_bucket_errors=transient_bucket_errors,\n                )\n                deleted_total += d_cnt\n                failed_total += f_cnt\n                deferred_total += s_cnt\n                now_m = time.monotonic()\n                if (now_m - last_log_mono) >= 30.0:\n                    with job.db_lock:\n                        self._refresh_buckets_locked(job, conn)\n                    with job.lock:\n                        job.message = (\n                            f"正在核验 Drive 并清理 R2 原文件：已删 {deleted_total} 个"\n                            + (f"，等待同步 {deferred_total} 个" if deferred_total else "")\n                            + (f"，失败 {failed_total} 个" if failed_total else "")\n                        )\n                        job.updated_at = time.time()\n                    self._log(\n                        f"job {job.job_id} cleanup progress: deleted={deleted_total} deferred_sync={deferred_total} failed={failed_total}"\n                    )\n                    last_log_mono = now_m\n                if d_cnt == 0 and f_cnt == 0 and s_cnt == 0:\n                    break\n\n            with job.db_lock:\n                self._refresh_counts_locked(job, conn)\n                self._refresh_buckets_locked(job, conn)\n            counts = job.status_counts\n            rem_pending = counts.get("pending", {}).get("count", 0)\n            rem_ia = counts.get("ia_deferred", {}).get("count", 0)\n            rem_copied = counts.get("copied", {}).get("count", 0)\n            rem_failed = counts.get("failed", {}).get("count", 0)\n\n            now_done = time.time()\n            with job.lock:\n                if job.cancel_requested:\n                    job.status = "cancelled"\n                    job.message = f"已取消清理任务（本次已清理 {deleted_total} 个 R2 原文件）"\n                elif job.pause_requested or self._shutdown:\n                    job.status = "paused_quota" if prev_status == "paused_quota" else "paused"\n                    job.message = f"已暂停清理任务（本次已清理 {deleted_total} 个 R2 原文件）"\n                elif failed_total > 0 or (transient_bucket_errors and deleted_total == 0):\n                    job.status = "failed"\n                    err_note = ""\n                    if transient_bucket_errors:\n                        err_note = "；R2 暂态超时桶: " + ", ".join(\n                            f"{b} ({e})" for b, e in sorted(transient_bucket_errors.items())\n                        )\n                    job.message = (\n                        f"已核验并清理 {deleted_total} 个 R2 原文件；"\n                        f"{failed_total or rem_failed} 个核验或删除失败"\n                        + (f"，{deferred_total} 个正等待 Drive 云端同步" if deferred_total else "")\n                        + (f"，另有 {rem_pending} 个待迁移" if rem_pending else "")\n                        + err_note\n                    )\n                elif deferred_total > 0 or rem_copied > 0:\n                    job.status = "paused_quota"\n                    extra_parts: list[str] = []\n                    if rem_pending or rem_ia:\n                        extra_parts.append(f"另有 {rem_pending + rem_ia} 个待迁移")\n                    if rem_failed:\n                        extra_parts.append(f"{rem_failed} 个失败待重试")\n                    if transient_bucket_errors:\n                        extra_parts.append(\n                            "部分桶 R2 删源暂时超时待重试: " + ", ".join(sorted(transient_bucket_errors.keys()))\n                        )\n                    extra_suffix = ("（" + "；".join(extra_parts) + "）") if extra_parts else ""\n                    job.message = (\n                        f"已核验并清理 {deleted_total} 个 R2 原文件；"\n                        f"剩余 {rem_copied} 个已拷贝对象正等待 Google Drive 云端同步 (750 GB/24h 限流)"\n                        f"{extra_suffix}，同步完成后请再次执行 cleanup / resume {job.job_id}"\n                    )\n                elif rem_pending > 0 or rem_ia > 0:\n                    job.status = (\n                        prev_status\n                        if prev_status in ("paused", "paused_budget", "paused_quota", "interrupted")\n                        else "paused"\n                    )\n                    job.message = (\n                        f"已核验并清理 {deleted_total} 个 R2 原文件；"\n                        f"仍有 {rem_pending + rem_ia} 个对象待迁移"\n                        + (f"（另有 {rem_failed} 个失败待重试）" if rem_failed else "")\n                    )\n                elif rem_failed > 0:\n                    job.status = "failed"\n                    job.message = f"已核验并清理 {deleted_total} 个 R2 原文件；仍有 {rem_failed} 个失败对象待重试"\n                else:\n                    job.status = "completed"\n                    job.message = f"已核验 Drive 并清理 {deleted_total} 个 R2 原文件"\n                job.finished_at = now_done\n                job.session_finished_at = now_done\n            self._flush_checkpoint(job, conn)\n            return self.get_job(job.job_id)\n        except Exception as err:\n            self._log(f"job {job.job_id} cleanup error: {err}")\n            now_done = time.time()\n            with job.lock:\n                job.recent_errors.append({"key": "", "error": str(err)})\n                job.status = "failed"\n                job.message = f"R2 清理失败（已删 {deleted_total} 个）: {err}"\n                job.finished_at = now_done\n                job.session_finished_at = now_done\n            with contextlib.suppress(Exception):\n                self._flush_checkpoint(job, conn)\n            return self.get_job(job.job_id)\n        finally:\n            client.close()\n            self.ledger.flush(ignore_errors=True)\n\n    def shutdown(self) -> None:\n        self._shutdown = True\n        with self._mgr_lock:\n            jobs = list(self._jobs.values())\n        for job in jobs:\n            with job.lock:\n                if job.status in ("queued", "running"):\n                    job.pause_requested = True\n                    job.stop_event.set()\n        for job in jobs:\n            th = job.worker_thread\n            if th is not None and th.is_alive():\n                th.join(timeout=60.0)\n            with job.db_lock:\n                if job.conn is not None:\n                    with contextlib.suppress(Exception):\n                        job.conn.execute("PRAGMA wal_checkpoint(TRUNCATE)")\n                    with contextlib.suppress(Exception):\n                        job.conn.close()\n                    job.conn = None\n        self.ledger.flush(ignore_errors=True)\n\n    def _launch_job_thread(self, job: _RuntimeJob, creds: R2Credentials) -> None:\n        th = threading.Thread(\n            target=self._run_job_wrapper,\n            args=(job, creds),\n            name=f"r2-job-{job.job_id}",\n            daemon=True,\n        )\n        with job.lock:\n            job.worker_thread = th\n        th.start()\n\n    def _run_job_wrapper(self, job: _RuntimeJob, creds: R2Credentials) -> None:\n        client = R2Client(creds, ledger=self.ledger, timeout=self.client_timeout)\n        conn: sqlite3.Connection | None = None\n        try:\n            conn = self._ensure_job_db_open(job)\n            self._run_job(job, conn, client)\n        except _BudgetPaused as pause_err:\n            self._log(f"job {job.job_id} paused on free-tier cap: {pause_err.message}")\n            now_wall = time.time()\n            with job.lock:\n                job.status = "paused_budget"\n                job.message = pause_err.message\n                job.finished_at = now_wall\n                job.session_finished_at = now_wall\n            if conn is not None:\n                with contextlib.suppress(Exception):\n                    self._flush_checkpoint(job, conn)\n            else:\n                with contextlib.suppress(OSError):\n                    self._persist_job_summary(job)\n        except _QuotaPaused as quota_err:\n            self._log(f"job {job.job_id} paused on Drive quota/cache guard: {quota_err.message}")\n            now_wall = time.time()\n            with job.lock:\n                job.status = "paused_quota"\n                job.message = quota_err.message\n                job.finished_at = now_wall\n                job.session_finished_at = now_wall\n            if conn is not None:\n                with contextlib.suppress(Exception):\n                    self._flush_checkpoint(job, conn)\n            else:\n                with contextlib.suppress(OSError):\n                    self._persist_job_summary(job)\n        except Exception as err:\n            self._log(f"job {job.job_id} error: {err}")\n            now_wall = time.time()\n            with job.lock:\n                job.recent_errors.append({"key": "", "error": str(err)})\n                job.status = "failed"\n                job.message = f"任务失败: {err}"\n                job.finished_at = now_wall\n                job.session_finished_at = now_wall\n            if conn is not None:\n                with contextlib.suppress(Exception):\n                    self._flush_checkpoint(job, conn)\n            else:\n                with contextlib.suppress(OSError):\n                    self._persist_job_summary(job)\n        finally:\n            client.close()\n            self.ledger.flush(ignore_errors=True)\n            with contextlib.suppress(OSError):\n                if job.spool_dir.is_dir() and not any(job.spool_dir.iterdir()):\n                    job.spool_dir.rmdir()\n\n    def _consolidate_existing_drive_files(\n        self,\n        job: _RuntimeJob,\n        conn: sqlite3.Connection,\n        tar_pipeline: _TarArchivePipeline,\n        *,\n        include_bucket: bool,\n        spool_budget: ByteBudgetSemaphore,\n        is_stopping: Callable[[], bool],\n        on_archived_transition: Callable[[dict[str, tuple[int, int]], int, int], None],\n    ) -> None:\n        """Scan loose files already stored under `dest_root`, pack verified files into TAR shards,\n        unlink corrupt partial downloads, and prune empty directories only after Drive flush + DB commit."""\n        dest_root = job.dest_root\n        if not dest_root.is_dir():\n            return\n        archives_name = "archives"\n        skip_roots: list[Path] = []\n        for extra in (self.data_dir, self.work_dir):\n            with contextlib.suppress(OSError):\n                skip_roots.append(extra.resolve())\n\n        def _is_skipped_dir(p: Path) -> bool:\n            if p.parent == dest_root and p.name == archives_name:\n                return True\n            with contextlib.suppress(OSError):\n                rp = p.resolve()\n                for sr in skip_roots:\n                    if rp == sr or sr in rp.parents:\n                        return True\n                    # Do not recurse into data_dir/work_dir if placed inside dest_root.\n                    if rp == sr or rp.is_relative_to(sr):\n                        return True\n            return False\n\n        # Quick check: skip thread pools completely when no loose items exist outside archives/.\n        has_non_archive = False\n        try:\n            with os.scandir(dest_root) as it:\n                for entry in it:\n                    if entry.is_symlink():\n                        continue\n                    p = Path(entry.path)\n                    if entry.is_dir(follow_symlinks=False):\n                        if not _is_skipped_dir(p):\n                            has_non_archive = True\n                            break\n                    elif entry.is_file(follow_symlinks=False):\n                        if _is_temp_artifact_name(entry.name):\n                            with contextlib.suppress(OSError):\n                                p.unlink(missing_ok=True)\n                        else:\n                            has_non_archive = True\n                            break\n        except OSError:\n            return\n        if not has_non_archive:\n            return\n\n        with job.lock:\n            job.message = "正在整理 Drive 已有散列文件并打包为 TAR 分卷..."\n            job.updated_at = time.time()\n\n        with job.db_lock:\n            bucket_names = [str(r[0]) for r in conn.execute("SELECT bucket FROM buckets ORDER BY bucket").fetchall()]\n        single_bucket = bucket_names[0] if len(bucket_names) == 1 else ""\n\n        prefetch_budget = ByteBudgetSemaphore(_PREFETCH_BUDGET_BYTES)\n        shard_verify_cache: dict[str, bool] = {}\n        shard_verify_lock = threading.Lock()\n        last_ckpt_m = time.monotonic()\n\n        def _verify_archived_shard(shard_name: str, rel_tar_path: str, tar_bytes: int) -> bool:\n            with shard_verify_lock:\n                cached = shard_verify_cache.get(shard_name)\n            if cached is not None:\n                return cached\n            ok = False\n            with contextlib.suppress(OSError):\n                tp = dest_root / rel_tar_path\n                ok = tp.is_file() and tp.stat().st_size == int(tar_bytes)\n            with shard_verify_lock:\n                shard_verify_cache[shard_name] = ok\n            return ok\n\n        def _derive_shard_bucket(rel_path: str, db_bucket: str) -> str:\n            if db_bucket:\n                return db_bucket\n            if not include_bucket and single_bucket:\n                return single_bucket\n            if "/" in rel_path:\n                first = rel_path.split("/", 1)[0]\n                if first and first != "__r2_root__":\n                    return first\n            return single_bucket or "drive-existing"\n\n        def _prefetch_one(\n            rel_path: str,\n            full_path: Path,\n            db_row: tuple[str, str, int, int, str] | None,\n            arch_row: tuple[int, str, str, int] | None,\n        ) -> None:\n            if is_stopping():\n                return\n            try:\n                st_res = full_path.stat()\n                actual_size = int(st_res.st_size)\n            except OSError as exc:\n                with job.lock:\n                    job.recent_errors.append({"key": rel_path, "error": f"Drive 整理 stat 失败: {exc}"})\n                return\n\n            # Case 1: Already stored in an existing verified .tar shard on Drive.\n            if arch_row is not None:\n                arch_size, shard_name, rel_tar_path, tar_bytes = arch_row\n                if actual_size == int(arch_size) and _verify_archived_shard(shard_name, rel_tar_path, tar_bytes):\n                    if db_row is not None:\n                        db_b, db_k, db_sz, _, db_st = db_row\n                        if db_st in ("pending", "ia_deferred", "failed"):\n                            now_ts = time.time()\n                            with job.db_lock:\n                                conn.execute(\n                                    "UPDATE objects SET status=\'copied\', error=\'\', updated_at=? WHERE bucket=? AND key=? AND status IN (\'pending\',\'ia_deferred\',\'failed\')",\n                                    (now_ts, db_b, db_k),\n                                )\n                                conn.commit()\n                            deltas = {\n                                db_st: (-1, -int(db_sz)),\n                                "copied": (1, int(db_sz)),\n                            }\n                            on_archived_transition(deltas, 1, actual_size)\n                            self.ledger.record_free(skipped_existing=1, bytes_transferred=actual_size)\n                    with contextlib.suppress(OSError):\n                        full_path.unlink(missing_ok=True)\n                    return\n\n            # Case 2: Incomplete / corrupted partial loose file waiting for retry/pending download.\n            if db_row is not None:\n                db_b, db_k, db_sz, db_is_dir, db_st = db_row\n                if not db_is_dir and db_st in ("pending", "ia_deferred", "failed") and actual_size != int(db_sz):\n                    with contextlib.suppress(OSError):\n                        full_path.unlink(missing_ok=True)\n                    return\n                if db_st in ("pending", "ia_deferred", "failed"):\n                    if actual_size == 0:\n                        self.ledger.record_free(zero_byte=1)\n                    else:\n                        self.ledger.record_free(skipped_existing=1, bytes_transferred=actual_size)\n            else:\n                db_b, db_k = "", ""\n\n            shard_bucket = _derive_shard_bucket(rel_path, db_b)\n            held = 0\n            spool_tmp: Path | None = None\n            submitted = False\n            try:\n                if actual_size == 0:\n                    tar_pipeline.submit(\n                        _ArchiveTask(\n                            bucket=db_b,\n                            key=db_k,\n                            shard_bucket=shard_bucket,\n                            rel_path=rel_path,\n                            size=0,\n                            is_dir=False,\n                            source="drive_existing",\n                            payload_bytes=b"",\n                            loose_path=full_path,\n                        )\n                    )\n                    submitted = True\n                elif actual_size <= _PREFETCH_MEM_MAX_FILE:\n                    held = prefetch_budget.acquire(actual_size, stop_event=job.stop_event)\n                    data = full_path.read_bytes()\n                    if len(data) != actual_size:\n                        raise OSError(\n                            f"Drive pre-pack read mismatch for {full_path}: expected {actual_size}B, read {len(data)}B"\n                        )\n                    tar_pipeline.submit(\n                        _ArchiveTask(\n                            bucket=db_b,\n                            key=db_k,\n                            shard_bucket=shard_bucket,\n                            rel_path=rel_path,\n                            size=actual_size,\n                            is_dir=False,\n                            source="drive_existing",\n                            payload_bytes=data,\n                            held_budget=held,\n                            release_fn=prefetch_budget.release,\n                            loose_path=full_path,\n                        )\n                    )\n                    submitted = True\n                else:\n                    held = spool_budget.acquire(actual_size, stop_event=job.stop_event)\n                    spool_tmp = job.spool_dir / f"cons.{uuid.uuid4().hex}.spool"\n                    copied = 0\n                    with full_path.open("rb") as src_f, spool_tmp.open("wb") as dst_f:\n                        while True:\n                            buf = src_f.read(4 * 1024 * 1024)\n                            if not buf:\n                                break\n                            dst_f.write(buf)\n                            copied += len(buf)\n                    if copied != actual_size:\n                        raise OSError(\n                            f"Drive stage copy mismatch for {full_path}: expected {actual_size}B, copied {copied}B"\n                        )\n                    tar_pipeline.submit(\n                        _ArchiveTask(\n                            bucket=db_b,\n                            key=db_k,\n                            shard_bucket=shard_bucket,\n                            rel_path=rel_path,\n                            size=actual_size,\n                            is_dir=False,\n                            source="drive_existing",\n                            source_path=spool_tmp,\n                            unlink_source_after_pack=True,\n                            held_budget=held,\n                            release_fn=spool_budget.release,\n                            loose_path=full_path,\n                        )\n                    )\n                    submitted = True\n            except _JobStopped:\n                pass\n            except Exception as exc:\n                with job.lock:\n                    job.recent_errors.append({"key": rel_path, "error": f"Drive 整理读取失败: {exc}"})\n            finally:\n                if not submitted:\n                    if spool_tmp is not None:\n                        with contextlib.suppress(OSError):\n                            spool_tmp.unlink(missing_ok=True)\n                    if held:\n                        if spool_tmp is not None:\n                            spool_budget.release(held)\n                        else:\n                            prefetch_budget.release(held)\n\n        def _flush_batch(\n            file_batch: list[tuple[str, Path]],\n            dir_batch: list[tuple[str, Path]],\n            prefetch_pool: concurrent.futures.ThreadPoolExecutor,\n            futs: list[concurrent.futures.Future[None]],\n        ) -> None:\n            nonlocal last_ckpt_m\n            if dir_batch:\n                dir_rels = [r for r, _ in dir_batch]\n                with job.db_lock:\n                    q_marks = ",".join("?" for _ in dir_rels)\n                    dir_rows = conn.execute(\n                        f"SELECT rel_path, bucket, key FROM objects WHERE is_dir = 1 AND rel_path IN ({q_marks})",\n                        dir_rels,\n                    ).fetchall()\n                    arch_dirs = {\n                        str(r[0])\n                        for r in conn.execute(\n                            f"SELECT rel_path FROM archived_files WHERE rel_path IN ({q_marks})",\n                            dir_rels,\n                        ).fetchall()\n                    }\n                dir_db_map = {str(r[0]): (str(r[1]), str(r[2])) for r in dir_rows}\n                dir_path_map = dict(dir_batch)\n                for r_dir, (db_b, db_k) in dir_db_map.items():\n                    if r_dir in arch_dirs:\n                        continue\n                    tar_pipeline.submit(\n                        _ArchiveTask(\n                            bucket=db_b,\n                            key=db_k,\n                            shard_bucket=_derive_shard_bucket(r_dir, db_b),\n                            rel_path=r_dir,\n                            size=0,\n                            is_dir=True,\n                            source="drive_existing",\n                            loose_path=dir_path_map[r_dir],\n                        )\n                    )\n                dir_batch.clear()\n\n            if file_batch:\n                rels = [r for r, _ in file_batch]\n                obj_map: dict[str, tuple[str, str, int, int, str]] = {}\n                arch_map: dict[str, tuple[int, str, str, int]] = {}\n                with job.db_lock:\n                    q_marks = ",".join("?" for _ in rels)\n                    for rp, b, k, sz, is_d, st in conn.execute(\n                        f"SELECT rel_path, bucket, key, size, is_dir, status FROM objects WHERE rel_path IN ({q_marks})",\n                        rels,\n                    ).fetchall():\n                        obj_map[str(rp)] = (str(b), str(k), int(sz), int(is_d), str(st))\n                    for rp, asz, sname, rtar, tbytes in conn.execute(\n                        f"""\n                        SELECT a.rel_path, a.size, a.shard_name, t.rel_tar_path, t.tar_bytes\n                        FROM archived_files a\n                        JOIN tar_shards t ON a.shard_name = t.shard_name\n                        WHERE a.rel_path IN ({q_marks})\n                        """,\n                        rels,\n                    ).fetchall():\n                        arch_map[str(rp)] = (int(asz), str(sname), str(rtar), int(tbytes))\n\n                for rpath, fpath in file_batch:\n                    if is_stopping():\n                        break\n                    futs.append(\n                        prefetch_pool.submit(\n                            _prefetch_one,\n                            rpath,\n                            fpath,\n                            obj_map.get(rpath),\n                            arch_map.get(rpath),\n                        )\n                    )\n                    if len(futs) >= 256:\n                        done, not_done = concurrent.futures.wait(\n                            futs, return_when=concurrent.futures.FIRST_COMPLETED\n                        )\n                        for df in done:\n                            with contextlib.suppress(Exception):\n                                df.result()\n                        futs[:] = list(not_done)\n                file_batch.clear()\n\n            now_m = time.monotonic()\n            if (now_m - last_ckpt_m) >= self.checkpoint_interval:\n                self._flush_checkpoint(job, conn)\n                last_ckpt_m = time.monotonic()\n                with job.lock:\n                    job.message = (\n                        f"正在打包 Drive 已有文件: {job.consolidated_files} 个 ({job.tar_shards_count} 个 TAR 分卷)"\n                    )\n\n        visited_dirs: list[Path] = []\n        file_batch: list[tuple[str, Path]] = []\n        dir_batch: list[tuple[str, Path]] = []\n        pending_futs: list[concurrent.futures.Future[None]] = []\n\n        with concurrent.futures.ThreadPoolExecutor(\n            max_workers=24, thread_name_prefix=f"r2-cons-{job.job_id}"\n        ) as prefetch_pool:\n            dir_stack: list[Path] = [dest_root]\n            while dir_stack and not is_stopping():\n                cur_dir = dir_stack.pop()\n                try:\n                    with os.scandir(cur_dir) as it:\n                        entries = list(it)\n                except OSError:\n                    continue\n                for entry in entries:\n                    if is_stopping():\n                        break\n                    if entry.is_symlink():\n                        continue\n                    ep = Path(entry.path)\n                    if entry.is_dir(follow_symlinks=False):\n                        if _is_skipped_dir(ep):\n                            continue\n                        visited_dirs.append(ep)\n                        rel_d = str(ep.relative_to(dest_root)).replace("\\\\", "/")\n                        dir_batch.append((rel_d, ep))\n                        dir_stack.append(ep)\n                    elif entry.is_file(follow_symlinks=False):\n                        if _is_temp_artifact_name(entry.name):\n                            with contextlib.suppress(OSError):\n                                ep.unlink(missing_ok=True)\n                            continue\n                        rel_f = str(ep.relative_to(dest_root)).replace("\\\\", "/")\n                        file_batch.append((rel_f, ep))\n                    if len(file_batch) >= 400 or len(dir_batch) >= 400:\n                        _flush_batch(file_batch, dir_batch, prefetch_pool, pending_futs)\n            _flush_batch(file_batch, dir_batch, prefetch_pool, pending_futs)\n            for df in pending_futs:\n                with contextlib.suppress(Exception):\n                    df.result()\n\n        # Seal & flush consolidation shards to Drive and finish loose-file unlinks before pruning empty dirs.\n        tar_pipeline.flush_open_shards(wait=True)\n        for d in sorted(visited_dirs, key=lambda p: len(p.parts), reverse=True):\n            with contextlib.suppress(OSError):\n                d.rmdir()\n\n    def _run_job(self, job: _RuntimeJob, conn: sqlite3.Connection, client: R2Client) -> None:\n        cfg = job.config\n        cost_mode = str(cfg["cost_mode"])\n        max_free_a = int(cfg["max_free_class_a"])\n        max_free_b = int(cfg["max_free_class_b"])\n        workers = int(cfg["workers"])\n        writer_workers = int(cfg["writer_workers"])\n        overwrite = bool(cfg["overwrite"])\n        delete_source = bool(cfg["delete_source"])\n        storage_mode = str(cfg.get("storage_mode") or "files")\n        tar_shard_mb = float(cfg.get("tar_shard_mb") or DEFAULT_TAR_SHARD_MB)\n        consolidate_existing = bool(cfg.get("consolidate_existing", True))\n        prefix = str(cfg.get("prefix") or "")\n        public_urls: dict[str, str] = dict(cfg.get("public_urls") or {})\n\n        now_wall = time.time()\n        with job.lock:\n            job.status = "running"\n            job.started_at = job.started_at or now_wall\n            job.finished_at = None\n            job.session_started_at = now_wall\n            job.session_finished_at = None\n            job.files_done_session = 0\n            job.bytes_done_session = 0\n            job.message = "正在运行"\n            job.updated_at = now_wall\n        job.spool_dir.mkdir(parents=True, exist_ok=True)\n        # Clear any orphan NVMe temp files left by a killed worker before restarting.\n        for stale in list(job.spool_dir.iterdir()):\n            if stale.is_file():\n                with contextlib.suppress(OSError):\n                    stale.unlink(missing_ok=True)\n        job.dest_root.mkdir(parents=True, exist_ok=True)\n\n        # Step A: discover buckets if the buckets table is still empty.\n        with job.db_lock:\n            existing_buckets = [r[0] for r in conn.execute("SELECT bucket FROM buckets ORDER BY bucket").fetchall()]\n            flatten_meta = conn.execute("SELECT v FROM job_meta WHERE k=\'include_bucket\'").fetchone()\n\n        if not existing_buckets:\n            requested = list(cfg.get("buckets") or [])\n            excluded = set(cfg.get("exclude_buckets") or [])\n            if requested:\n                selected = [b for b in requested if b not in excluded]\n            else:\n                allowed, paid_a = self.ledger.can_spend_class_a(cost_mode, max_free_a)\n                if not allowed:\n                    raise _BudgetPaused(f"免费 Class A 配额上限已达 ({max_free_a})，ListBuckets 已暂停")\n                if paid_a:\n                    with job.lock:\n                        job.paid_fallback_used = True\n                discovered = [b["name"] for b in client.list_buckets()]\n                selected = [b for b in discovered if b not in excluded]\n            if not selected:\n                raise ValueError("未匹配到任何可迁移的 R2 Bucket")\n            if cfg.get("flatten_single_bucket") and len(selected) != 1:\n                raise ValueError(f"--flatten-single-bucket 仅支持单个桶，当前匹配 {len(selected)} 个: {selected}")\n            include_bucket = not bool(cfg.get("flatten_single_bucket") and len(selected) == 1)\n            with job.db_lock:\n                conn.executemany("INSERT OR IGNORE INTO buckets(bucket) VALUES (?)", [(b,) for b in selected])\n                conn.execute(\n                    "INSERT OR REPLACE INTO job_meta(k, v) VALUES (\'include_bucket\', ?)",\n                    ("1" if include_bucket else "0",),\n                )\n                conn.commit()\n                self._refresh_counts_locked(job, conn)\n                self._refresh_buckets_locked(job, conn)\n        else:\n            include_bucket = (flatten_meta[0] != "0") if flatten_meta else not bool(\n                cfg.get("flatten_single_bucket") and len(existing_buckets) == 1\n            )\n            with job.db_lock:\n                self._refresh_counts_locked(job, conn)\n                self._refresh_buckets_locked(job, conn)\n\n        self._persist_job_summary(job)\n\n        dir_cache = DriveDirCache(job.dest_root)\n        spool_budget = ByteBudgetSemaphore(self.spool_bytes_cap)\n        is_fuse_drive = _detect_fuse_drive_mydrive(job.dest_root) is not None\n        work_cond = threading.Condition()\n        budget_pause_msg: list[str] = []\n        quota_pause_msg: list[str] = []\n        lister_error: list[str] = []\n        internal_stop = threading.Event()\n        listing_finished = threading.Event()\n        delete_wake = threading.Event()\n        stop_deleter = threading.Event()\n\n        dispatched_keys: set[tuple[str, str]] = set()\n        active_dl: set[tuple[str, str]] = set()\n        in_flight_lock = threading.Lock()\n        in_flight_writers = [0]\n        outcomes_q: queue.Queue[tuple[str, str, str, str, str, int, str]] = queue.Queue()\n        relocate_q: queue.Queue[tuple[str, str]] = queue.Queue()\n        relocated_paths: dict[str, str] = {}\n        write_q: queue.Queue[tuple[str, str, str, int, Path, int] | None] = queue.Queue(maxsize=max(16, workers * 2))\n\n        def trigger_budget_pause(msg: str) -> None:\n            if not budget_pause_msg:\n                budget_pause_msg.append(msg)\n            internal_stop.set()\n            job.stop_event.set()\n            with work_cond:\n                work_cond.notify_all()\n\n        def trigger_quota_pause(msg: str) -> None:\n            if not quota_pause_msg:\n                quota_pause_msg.append(msg)\n            internal_stop.set()\n            job.stop_event.set()\n            with work_cond:\n                work_cond.notify_all()\n\n        def is_stopping() -> bool:\n            return job.stop_event.is_set() or internal_stop.is_set()\n\n        def record_relocate(old_rel: str, new_rel: str) -> None:\n            relocate_q.put((old_rel, new_rel))\n\n        # --- Lister thread ---\n        def lister_loop() -> None:\n            try:\n                while not is_stopping():\n                    with job.db_lock:\n                        row = conn.execute(\n                            "SELECT bucket, last_key FROM buckets WHERE listing_done = 0 ORDER BY bucket LIMIT 1"\n                        ).fetchone()\n                    if row is None:\n                        return\n                    bucket, cursor_key = str(row[0]), str(row[1] or "")\n                    while not is_stopping():\n                        buffered = (\n                            job.status_counts.get("pending", {}).get("count", 0)\n                            + job.status_counts.get("ia_deferred", {}).get("count", 0)\n                        )\n                        if buffered >= MAX_LISTED_BUFFER:\n                            time.sleep(0.5)\n                            continue\n\n                        allowed, paid_a = self.ledger.can_spend_class_a(cost_mode, max_free_a)\n                        if not allowed:\n                            trigger_budget_pause(\n                                f"已达到 free-only Class A 上限 ({max_free_a:,})，列表在 {bucket}/{cursor_key} 处安全暂停"\n                            )\n                            return\n                        if paid_a:\n                            with job.lock:\n                                if not job.paid_fallback_used:\n                                    job.paid_fallback_used = True\n                                    self._log(f"job {job.job_id}: Class A 已超出免费额度，自动切换付费模式继续列举")\n\n                        page = client.list_objects_v2(\n                            bucket,\n                            prefix=prefix,\n                            start_after=cursor_key,\n                            max_keys=self.page_size,\n                        )\n                        now_ts = time.time()\n                        rows_to_insert: list[tuple[Any, ...]] = []\n                        for entry in page.objects:\n                            rel_path, is_dir = map_object_to_drive_relpath(\n                                bucket, entry.key, size=entry.size, include_bucket=include_bucket\n                            )\n                            if (\n                                not is_dir\n                                and entry.size > 0\n                                and entry.storage_class == "STANDARD_IA"\n                                and cost_mode in ("free-first", "free-only")\n                            ):\n                                st_val = "ia_deferred"\n                            else:\n                                st_val = "pending"\n                            rows_to_insert.append(\n                                (\n                                    bucket,\n                                    entry.key,\n                                    entry.size,\n                                    entry.etag,\n                                    entry.storage_class,\n                                    rel_path,\n                                    1 if is_dir else 0,\n                                    st_val,\n                                    "",\n                                    now_ts,\n                                )\n                            )\n                        new_cursor = page.last_key or cursor_key\n                        done_flag = 1 if (not page.is_truncated or not page.objects) else 0\n                        list_deltas: dict[str, tuple[int, int]] = {}\n                        with job.db_lock:\n                            if rows_to_insert:\n                                existing_keys: set[str] = set()\n                                keys_page = [r[1] for r in rows_to_insert]\n                                for k_idx in range(0, len(keys_page), 500):\n                                    k_chunk = keys_page[k_idx : k_idx + 500]\n                                    q_marks = ",".join("?" for _ in k_chunk)\n                                    existing_keys.update(\n                                        str(r[0])\n                                        for r in conn.execute(\n                                            f"SELECT key FROM objects WHERE bucket = ? AND key IN ({q_marks})",\n                                            [bucket, *k_chunk],\n                                        ).fetchall()\n                                    )\n                                new_rows = [r for r in rows_to_insert if r[1] not in existing_keys]\n                                conn.executemany(\n                                    """\n                                    INSERT OR IGNORE INTO objects\n                                    (bucket, key, size, etag, storage_class, rel_path, is_dir, status, error, updated_at)\n                                    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)\n                                    """,\n                                    rows_to_insert,\n                                )\n                                added_cnt = len(new_rows)\n                                added_bytes = sum(int(r[2]) for r in new_rows)\n                                for nr in new_rows:\n                                    st_k = str(nr[7])\n                                    dc, db = list_deltas.get(st_k, (0, 0))\n                                    list_deltas[st_k] = (dc + 1, db + int(nr[2]))\n                            else:\n                                added_cnt = 0\n                                added_bytes = 0\n                            conn.execute(\n                                """\n                                UPDATE buckets\n                                SET last_key = ?, listing_done = ?,\n                                    objects_listed = objects_listed + ?,\n                                    bytes_listed = bytes_listed + ?\n                                WHERE bucket = ?\n                                """,\n                                (new_cursor, done_flag, added_cnt, added_bytes, bucket),\n                            )\n                            conn.commit()\n                        if list_deltas:\n                            self._adjust_status_counts(job, list_deltas)\n                        cursor_key = new_cursor\n                        with work_cond:\n                            work_cond.notify_all()\n                        if done_flag:\n                            break\n            except Exception as exc:\n                self._log(f"job {job.job_id} lister error: {exc}")\n                lister_error.append(str(exc))\n                with job.lock:\n                    job.recent_errors.append({"key": "", "error": f"列举失败: {exc}"})\n                internal_stop.set()\n                job.stop_event.set()\n            finally:\n                listing_finished.set()\n                with work_cond:\n                    work_cond.notify_all()\n\n        # --- Background R2 DeleteObjects thread ---\n        def deleter_loop() -> None:\n            del_after_keys: dict[str, str] = {}\n            del_verifier = _DriveCloudSyncVerifier(job.dest_root, log=self._log)\n            while not stop_deleter.is_set() and not is_stopping():\n                deferred_cycle = 0\n                deleted_cycle = 0\n                try:\n                    with job.lock:\n                        copied_total = job.status_counts.get("copied", {}).get("count", 0)\n                    if copied_total >= 1000:\n                        with job.db_lock:\n                            b_names = [\n                                str(r[0])\n                                for r in conn.execute("SELECT bucket FROM buckets ORDER BY bucket").fetchall()\n                            ]\n                            ready_buckets: list[str] = []\n                            for b_name in b_names:\n                                ak = del_after_keys.get(b_name, "")\n                                if (\n                                    conn.execute(\n                                        "SELECT 1 FROM objects WHERE bucket=? AND status=\'copied\' AND key > ? "\n                                        "ORDER BY key LIMIT 1 OFFSET 999",\n                                        (b_name, ak),\n                                    ).fetchone()\n                                    is not None\n                                ):\n                                    ready_buckets.append(b_name)\n                                elif ak:\n                                    del_after_keys[b_name] = ""\n                        for r_b in ready_buckets:\n                            if stop_deleter.is_set() or is_stopping():\n                                break\n                            d_cnt, _, s_cnt = self._delete_copied_chunk(\n                                job,\n                                conn,\n                                client,\n                                bucket_filter=r_b,\n                                batch_limit=1000,\n                                after_keys=del_after_keys,\n                                single_pass=False,\n                                cloud_verifier=del_verifier,\n                            )\n                            deleted_cycle += d_cnt\n                            deferred_cycle += s_cnt\n                except Exception as exc:\n                    self._log(f"job {job.job_id}: background deleter transient error (retrying): {exc}")\n                    delete_wake.wait(timeout=3.0)\n                    delete_wake.clear()\n                    continue\n                wait_sec = 5.0 if (deferred_cycle > 0 and deleted_cycle == 0) else 1.5\n                delete_wake.wait(timeout=wait_sec)\n                delete_wake.clear()\n\n        # --- DriveFS direct writer workers (files mode) ---\n        def writer_loop() -> None:\n            while True:\n                item = write_q.get()\n                if item is None:\n                    write_q.task_done()\n                    return\n                bucket, key, rel_path, size, spool_tmp, budget_held = item\n                try:\n                    actual_rel = dir_cache.write_file_direct(\n                        rel_path, spool_tmp, size, on_relocate=record_relocate\n                    )\n                    outcomes_q.put((bucket, key, actual_rel, "copied", "", size, "pending"))\n                except OSError as exc:\n                    if exc.errno == errno.ENOSPC or "No space left on device" in str(exc):\n                        trigger_quota_pause(\n                            "Google Drive 挂载或本地缓存空间不足（可能已触发单账号 750 GB / 24h 上传限流 userRateLimitExceeded）；"\n                            "任务已安全暂停，待云端后台上传释放缓存后执行 resume 继续"\n                        )\n                        outcomes_q.put((bucket, key, rel_path, "pending", "", size, "pending"))\n                    else:\n                        outcomes_q.put((bucket, key, rel_path, "failed", f"Drive write failed: {exc}", size, "pending"))\n                except Exception as exc:\n                    outcomes_q.put((bucket, key, rel_path, "failed", f"Drive write failed: {exc}", size, "pending"))\n                finally:\n                    with contextlib.suppress(OSError):\n                        spool_tmp.unlink(missing_ok=True)\n                    spool_budget.release(budget_held)\n                    with in_flight_lock:\n                        in_flight_writers[0] -= 1\n                    write_q.task_done()\n                    with work_cond:\n                        work_cond.notify_all()\n\n        # --- TAR pipeline callbacks (tar mode) ---\n        def on_tar_committed(\n            entries: list[_ArchivedEntry],\n            tar_bytes: int,\n            deltas: dict[str, tuple[int, int]],\n            cons_cnt: int,\n            cons_bytes: int,\n            sess_cnt: int,\n            sess_bytes: int,\n        ) -> None:\n            self._adjust_status_counts(job, deltas)\n            if tar_bytes > 0 or cons_cnt or sess_cnt:\n                with job.lock:\n                    if tar_bytes > 0:\n                        job.tar_shards_count += 1\n                        job.tar_bytes_total += int(tar_bytes)\n                    job.consolidated_files += cons_cnt\n                    job.consolidated_bytes += cons_bytes\n                    job.files_done_session += sess_cnt\n                    job.bytes_done_session += sess_bytes\n            with in_flight_lock:\n                for e in entries:\n                    if e.bucket and e.key:\n                        dispatched_keys.discard((e.bucket, e.key))\n            if deltas.get("copied", (0, 0))[0] > 0:\n                delete_wake.set()\n            with work_cond:\n                work_cond.notify_all()\n\n        def on_tar_failed(entries: list[_ArchivedEntry], err_msg: str) -> None:\n            if not err_msg:\n                with in_flight_lock:\n                    for e in entries:\n                        if e.bucket and e.key:\n                            dispatched_keys.discard((e.bucket, e.key))\n                with work_cond:\n                    work_cond.notify_all()\n                return\n            now_ts = time.time()\n            upd: list[tuple[str, float, str, str]] = []\n            deltas: dict[str, tuple[int, int]] = {}\n            for e in entries:\n                if e.bucket and e.key:\n                    upd.append((err_msg, now_ts, e.bucket, e.key))\n                    if e.source == "r2_download":\n                        dc, db = deltas.get("pending", (0, 0))\n                        deltas["pending"] = (dc - 1, db - int(e.size))\n                        fc, fb = deltas.get("failed", (0, 0))\n                        deltas["failed"] = (fc + 1, fb + int(e.size))\n            if upd:\n                with job.db_lock:\n                    conn.executemany(\n                        "UPDATE objects SET status=\'failed\', error=?, updated_at=? WHERE bucket=? AND key=?",\n                        upd,\n                    )\n                    conn.commit()\n                self._adjust_status_counts(job, deltas)\n            with job.lock:\n                job.recent_errors.append({"key": entries[0].orig_rel_path if entries else "", "error": err_msg})\n            with in_flight_lock:\n                for e in entries:\n                    if e.bucket and e.key:\n                        dispatched_keys.discard((e.bucket, e.key))\n            with work_cond:\n                work_cond.notify_all()\n\n        def on_cons_archived_transition(\n            deltas: dict[str, tuple[int, int]], sess_cnt: int, sess_bytes: int\n        ) -> None:\n            self._adjust_status_counts(job, deltas)\n            with job.lock:\n                job.files_done_session += sess_cnt\n                job.bytes_done_session += sess_bytes\n            if deltas.get("copied", (0, 0))[0] > 0:\n                delete_wake.set()\n\n        tar_pipeline: _TarArchivePipeline | None = None\n        writer_threads: list[threading.Thread] = []\n        if storage_mode == "tar":\n            tar_pipeline = _TarArchivePipeline(\n                job=job,\n                conn=conn,\n                tar_shard_mb=tar_shard_mb,\n                on_committed=on_tar_committed,\n                on_failed=on_tar_failed,\n                on_quota_pause=trigger_quota_pause,\n                log=self._log,\n            )\n            with job.db_lock:\n                self._refresh_counts_locked(job, conn)\n        else:\n            writer_threads = [\n                threading.Thread(target=writer_loop, name=f"r2-writer-{job.job_id}-{i}", daemon=True)\n                for i in range(writer_workers)\n            ]\n            for wt in writer_threads:\n                wt.start()\n\n        lister_thread = threading.Thread(target=lister_loop, name=f"r2-lister-{job.job_id}", daemon=True)\n        lister_thread.start()\n\n        deleter_thread: threading.Thread | None = None\n        if delete_source:\n            deleter_thread = threading.Thread(target=deleter_loop, name=f"r2-del-{job.job_id}", daemon=True)\n            deleter_thread.start()\n\n        # --- Downloader worker function ---\n        def process_object(row: tuple[str, str, int, str, str, str, int]) -> None:\n            bucket, key, size, etag, sc, rel_path, is_dir = row\n            handed_off = False\n            spool_tmp: Path | None = None\n            held = 0\n            try:\n                if is_stopping():\n                    return\n\n                if storage_mode == "tar":\n                    assert tar_pipeline is not None\n                    # 1. Pre-check if consolidated / previously archived into a TAR shard (0 Class B ops!)\n                    if not overwrite:\n                        found = tar_pipeline.find_archived(rel_path, None if is_dir else size)\n                        if found is not None:\n                            arch_rel, arch_sz, f_b, _ = found\n                            if not f_b:\n                                tar_pipeline.backfill_archived_bk(arch_rel, bucket, key)\n                            if is_dir or size == 0:\n                                self.ledger.record_free(zero_byte=1)\n                            else:\n                                self.ledger.record_free(skipped_existing=1, bytes_transferred=size)\n                            outcomes_q.put((bucket, key, arch_rel, "copied", "", size, "pending"))\n                            handed_off = True\n                            return\n                        if not consolidate_existing and not is_dir:\n                            actual_rel, existing_sz = dir_cache.preexisting_file_size(rel_path)\n                            if existing_sz is not None and existing_sz == size:\n                                if size == 0:\n                                    self.ledger.record_free(zero_byte=1)\n                                else:\n                                    self.ledger.record_free(skipped_existing=1, bytes_transferred=size)\n                                outcomes_q.put((bucket, key, actual_rel, "copied", "", size, "pending"))\n                                handed_off = True\n                                return\n\n                    # 2. Directory markers & 0-byte files pack straight into TAR (0 Class B ops, 0 FUSE ops!)\n                    if is_dir:\n                        self.ledger.record_free(zero_byte=1)\n                        tar_pipeline.submit(\n                            _ArchiveTask(\n                                bucket=bucket,\n                                key=key,\n                                rel_path=rel_path,\n                                size=0,\n                                is_dir=True,\n                                source="r2_download",\n                            )\n                        )\n                        handed_off = True\n                        return\n                    if size == 0:\n                        self.ledger.record_free(zero_byte=1)\n                        tar_pipeline.submit(\n                            _ArchiveTask(\n                                bucket=bucket,\n                                key=key,\n                                rel_path=rel_path,\n                                size=0,\n                                is_dir=False,\n                                source="r2_download",\n                                payload_bytes=b"",\n                            )\n                        )\n                        handed_off = True\n                        return\n\n                    # 3. Budget check & R2 download -> NVMe TAR packer\n                    pub_url = public_urls.get(bucket)\n                    allowed_b, paid_b = self.ledger.can_spend_class_b(sc, cost_mode, max_free_b)\n                    if not allowed_b:\n                        trigger_budget_pause(\n                            f"已达到 free-only Class B 上限 ({max_free_b:,}) 或遇到低频存储 {sc}，任务已安全暂停"\n                        )\n                        return\n                    if paid_b:\n                        with job.lock:\n                            if not job.paid_fallback_used:\n                                job.paid_fallback_used = True\n                                self._log(f"job {job.job_id}: 已切换为付费 Class B/低频检索模式继续下载 ({sc})")\n\n                    held = spool_budget.acquire(size, stop_event=job.stop_event)\n                    spool_tmp = job.spool_dir / f"{uuid.uuid4().hex}.spool"\n                    client.download_object_to_file(\n                        bucket,\n                        key,\n                        spool_tmp,\n                        expected_size=size,\n                        expected_etag=etag,\n                        storage_class=sc,\n                        public_base_url=pub_url,\n                    )\n                    tar_pipeline.submit(\n                        _ArchiveTask(\n                            bucket=bucket,\n                            key=key,\n                            rel_path=rel_path,\n                            size=size,\n                            is_dir=False,\n                            source="r2_download",\n                            source_path=spool_tmp,\n                            unlink_source_after_pack=True,\n                            held_budget=held,\n                            release_fn=spool_budget.release,\n                        )\n                    )\n                    handed_off = True\n                    return\n\n                # --- files mode ---\n                if is_dir:\n                    with dir_cache.lock:\n                        dir_cache.ensure_dir_locked(rel_path, on_relocate=record_relocate)\n                    self.ledger.record_free(zero_byte=1)\n                    outcomes_q.put((bucket, key, rel_path, "copied", "", 0, "pending"))\n                    handed_off = True\n                    return\n\n                if not overwrite:\n                    actual_rel, existing_sz = dir_cache.preexisting_file_size(rel_path)\n                    if existing_sz is not None and existing_sz == size:\n                        self.ledger.record_free(skipped_existing=1, bytes_transferred=size)\n                        outcomes_q.put((bucket, key, actual_rel, "copied", "", size, "pending"))\n                        handed_off = True\n                        return\n\n                if size == 0:\n                    try:\n                        actual_rel = dir_cache.write_file_direct(rel_path, None, 0, on_relocate=record_relocate)\n                        self.ledger.record_free(zero_byte=1)\n                        outcomes_q.put((bucket, key, actual_rel, "copied", "", 0, "pending"))\n                    except Exception as exc:\n                        outcomes_q.put((bucket, key, rel_path, "failed", str(exc), 0, "pending"))\n                    handed_off = True\n                    return\n\n                pub_url = public_urls.get(bucket)\n                allowed_b, paid_b = self.ledger.can_spend_class_b(sc, cost_mode, max_free_b)\n                if not allowed_b:\n                    trigger_budget_pause(\n                        f"已达到 free-only Class A/B 上限 ({max_free_b:,}) 或遇到低频存储 {sc}，任务已安全暂停"\n                    )\n                    return\n                if paid_b:\n                    with job.lock:\n                        if not job.paid_fallback_used:\n                            job.paid_fallback_used = True\n                            self._log(f"job {job.job_id}: 已切换为付费 Class B/低频检索模式继续下载 ({sc})")\n\n                held = spool_budget.acquire(size, stop_event=job.stop_event)\n                spool_tmp = job.spool_dir / f"{uuid.uuid4().hex}.spool"\n                client.download_object_to_file(\n                    bucket,\n                    key,\n                    spool_tmp,\n                    expected_size=size,\n                    expected_etag=etag,\n                    storage_class=sc,\n                    public_base_url=pub_url,\n                )\n                with in_flight_lock:\n                    in_flight_writers[0] += 1\n                write_q.put((bucket, key, rel_path, size, spool_tmp, held))\n                handed_off = True\n            except _JobStopped:\n                pass\n            except Exception as exc:\n                if (isinstance(exc, OSError) and exc.errno == errno.ENOSPC) or "No space left on device" in str(exc):\n                    trigger_quota_pause(\n                        "Google Drive 挂载或本地缓存空间不足（可能已触发单账号 750 GB / 24h 上传限流 userRateLimitExceeded）；"\n                        "任务已安全暂停，待云端后台上传释放缓存后执行 resume 继续"\n                    )\n                else:\n                    outcomes_q.put((bucket, key, rel_path, "failed", str(exc), size, "pending"))\n                    handed_off = True\n            finally:\n                if not handed_off:\n                    if spool_tmp is not None:\n                        with contextlib.suppress(OSError):\n                            spool_tmp.unlink(missing_ok=True)\n                    if held:\n                        spool_budget.release(held)\n                    with in_flight_lock:\n                        dispatched_keys.discard((bucket, key))\n                with in_flight_lock:\n                    active_dl.discard((bucket, key))\n                with work_cond:\n                    work_cond.notify_all()\n\n        def _resolve_relpath(rel_p: str) -> str:\n            seen: set[str] = set()\n            cur = rel_p\n            while cur in relocated_paths and cur not in seen:\n                seen.add(cur)\n                cur = relocated_paths[cur]\n            return cur\n\n        def drain_outcomes() -> int:\n            raw_updates: list[tuple[str, str, str, str, str, int, str]] = []\n            relocates: list[tuple[str, str]] = []\n            while True:\n                try:\n                    raw_updates.append(outcomes_q.get_nowait())\n                    outcomes_q.task_done()\n                except queue.Empty:\n                    break\n            while True:\n                try:\n                    old_r, new_r = relocate_q.get_nowait()\n                    relocates.append((old_r, new_r))\n                    relocated_paths[old_r] = new_r\n                    relocate_q.task_done()\n                except queue.Empty:\n                    break\n\n            updates: list[tuple[str, str, str, float, str, str]] = []\n            deltas: dict[str, tuple[int, int]] = {}\n            session_files = 0\n            session_bytes = 0\n            err_items: list[dict[str, str]] = []\n            now_ts = time.time()\n            done_keys: list[tuple[str, str]] = []\n            for b, k, rpath, st_val, err_msg, sz, prev_st in raw_updates:\n                final_rel = _resolve_relpath(rpath)\n                updates.append((final_rel, st_val, err_msg, now_ts, b, k))\n                done_keys.append((b, k))\n                dc, db = deltas.get(prev_st, (0, 0))\n                deltas[prev_st] = (dc - 1, db - int(sz))\n                nc, nb = deltas.get(st_val, (0, 0))\n                deltas[st_val] = (nc + 1, nb + int(sz))\n                if st_val == "copied":\n                    session_files += 1\n                    session_bytes += int(sz)\n                elif st_val == "failed":\n                    err_items.append({"key": f"{b}/{k}", "error": err_msg})\n\n            if updates or relocates:\n                with job.db_lock:\n                    if updates:\n                        conn.executemany(\n                            "UPDATE objects SET rel_path=?, status=?, error=?, updated_at=? WHERE bucket=? AND key=?",\n                            updates,\n                        )\n                    for old_r, new_r in relocates:\n                        conn.execute(\n                            "UPDATE objects SET rel_path = ? WHERE rel_path = ?",\n                            (_resolve_relpath(new_r), old_r),\n                        )\n                    conn.commit()\n                if deltas:\n                    self._adjust_status_counts(job, deltas)\n            if done_keys:\n                with in_flight_lock:\n                    for bk in done_keys:\n                        dispatched_keys.discard(bk)\n            if session_files or err_items:\n                with job.lock:\n                    job.files_done_session += session_files\n                    job.bytes_done_session += session_bytes\n                    for item in err_items:\n                        job.recent_errors.append(item)\n                if deltas.get("copied", (0, 0))[0] > 0:\n                    delete_wake.set()\n            return len(updates)\n\n        try:\n            # Phase 1 (optional in tar mode): consolidate existing loose Drive files into TAR shards first.\n            if storage_mode == "tar" and tar_pipeline is not None and consolidate_existing and not is_stopping():\n                self._consolidate_existing_drive_files(\n                    job,\n                    conn,\n                    tar_pipeline,\n                    include_bucket=include_bucket,\n                    spool_budget=spool_budget,\n                    is_stopping=is_stopping,\n                    on_archived_transition=on_cons_archived_transition,\n                )\n                with job.db_lock:\n                    self._refresh_counts_locked(job, conn)\n                    self._refresh_buckets_locked(job, conn)\n                if not is_stopping():\n                    with job.lock:\n                        job.message = "正在高速迁移 R2 对象为 TAR 分卷"\n\n            # Phase 2: concurrent R2 downloads -> DriveFS / TAR pipeline\n            ckpt_interval = (\n                TAR_DEFAULT_CHECKPOINT_INTERVAL\n                if (storage_mode == "tar" and self.checkpoint_interval == DEFAULT_CHECKPOINT_INTERVAL)\n                else self.checkpoint_interval\n            )\n            last_ckpt_time = time.monotonic()\n            last_ckpt_changes = conn.total_changes\n            last_ui_time = time.monotonic()\n            last_disk_check = 0.0\n            pending_futs: set[concurrent.futures.Future[None]] = set()\n            dispatch_cursor: tuple[str, str] = ("", "")\n\n            if not is_stopping():\n                with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as dl_pool:\n                    while True:\n                        drain_outcomes()\n\n                        now_m = time.monotonic()\n                        if (now_m - last_ckpt_time) >= ckpt_interval:\n                            if conn.total_changes != last_ckpt_changes:\n                                self._flush_checkpoint(job, conn)\n                                last_ckpt_changes = conn.total_changes\n                                last_ui_time = time.monotonic()\n                            last_ckpt_time = time.monotonic()\n                        elif (now_m - last_ui_time) >= 30.0:\n                            with job.db_lock:\n                                self._refresh_counts_locked(job, conn)\n                                self._refresh_buckets_locked(job, conn)\n                            last_ui_time = now_m\n\n                        pending_futs = {f for f in pending_futs if not f.done()}\n\n                        if is_fuse_drive and (now_m - last_disk_check) >= 5.0 and not is_stopping():\n                            last_disk_check = now_m\n                            with contextlib.suppress(OSError):\n                                min_free = max(_MIN_DRIVEFS_WORK_FREE_BYTES, int(self.spool_bytes_cap) + 2 * 1024 * 1024 * 1024)\n                                if shutil.disk_usage(self.work_dir).free < min_free:\n                                    trigger_quota_pause(\n                                        "Google Drive 本地上传缓存空间不足（可能已触发单账号 750 GB / 24h 上传限流 userRateLimitExceeded）；"\n                                        "任务已安全暂停，待云端后台上传释放缓存后执行 resume 继续"\n                                    )\n                                    break\n\n                        if is_stopping():\n                            break\n\n                        max_queue = workers * 4\n                        with in_flight_lock:\n                            slots = max(0, max_queue - len(active_dl))\n                        if slots > 0:\n                            fetch_lim = max(slots * 2, 200)\n                            with job.db_lock:\n                                candidates = conn.execute(\n                                    "SELECT bucket, key, size, etag, storage_class, rel_path, is_dir "\n                                    "FROM objects WHERE status=\'pending\' AND (bucket, key) > (?, ?) "\n                                    "ORDER BY bucket, key LIMIT ?",\n                                    (dispatch_cursor[0], dispatch_cursor[1], fetch_lim),\n                                ).fetchall()\n                            dispatched = 0\n                            for cand in candidates:\n                                bk = (str(cand[0]), str(cand[1]))\n                                queue_full = False\n                                with in_flight_lock:\n                                    if bk in dispatched_keys or bk in active_dl:\n                                        dispatch_cursor = bk\n                                        continue\n                                    if len(active_dl) >= max_queue:\n                                        queue_full = True\n                                    else:\n                                        dispatch_cursor = bk\n                                        dispatched_keys.add(bk)\n                                        active_dl.add(bk)\n                                if queue_full:\n                                    break\n                                row_tuple = (\n                                    str(cand[0]),\n                                    str(cand[1]),\n                                    int(cand[2]),\n                                    str(cand[3]),\n                                    str(cand[4]),\n                                    str(cand[5]),\n                                    int(cand[6]),\n                                )\n                                pending_futs.add(dl_pool.submit(process_object, row_tuple))\n                                dispatched += 1\n                            if dispatched > 0:\n                                continue\n\n                        # Nothing newly dispatched from cursor right now; inspect active pipeline state.\n                        with in_flight_lock:\n                            busy_dl = len(active_dl)\n                            busy_wr = in_flight_writers[0]\n                            busy_disp = len(dispatched_keys)\n                        tar_pending = tar_pipeline.pending_count() if tar_pipeline is not None else 0\n\n                        if busy_dl == 0 and busy_wr == 0 and outcomes_q.empty():\n                            drain_outcomes()\n                            # Rewind cursor once when no active downloader exists, to confirm zero stragglers behind cursor.\n                            if dispatch_cursor != ("", "") and busy_disp == 0:\n                                dispatch_cursor = ("", "")\n                                continue\n\n                            # In tar mode, seal & flush open <shard_mb shards once all undispatched pending items are consumed.\n                            if (\n                                tar_pipeline is not None\n                                and tar_pipeline.has_open_shards()\n                                and listing_finished.is_set()\n                            ):\n                                with job.db_lock:\n                                    probe = conn.execute(\n                                        "SELECT bucket, key FROM objects WHERE status=\'pending\' ORDER BY bucket, key LIMIT 512"\n                                    ).fetchall()\n                                with in_flight_lock:\n                                    undispatched_left = any((str(r[0]), str(r[1])) not in dispatched_keys for r in probe)\n                                if not undispatched_left:\n                                    tar_pipeline.flush_open_shards(wait=True)\n                                    dispatch_cursor = ("", "")\n                                    continue\n\n                            if tar_pending == 0 and busy_disp == 0:\n                                with job.db_lock:\n                                    rem_pending = conn.execute(\n                                        "SELECT COUNT(*) FROM objects WHERE status=\'pending\'"\n                                    ).fetchone()[0]\n                                    rem_ia = conn.execute(\n                                        "SELECT COUNT(*) FROM objects WHERE status=\'ia_deferred\'"\n                                    ).fetchone()[0]\n                                if rem_pending > 0:\n                                    dispatch_cursor = ("", "")\n                                    continue\n                                if listing_finished.is_set():\n                                    if rem_ia > 0 and cost_mode == "free-first" and not is_stopping():\n                                        self._log(\n                                            f"job {job.job_id}: Pass 1 免费层级对象完成；切换付费 STANDARD_IA 检索 ({rem_ia} 个对象)"\n                                        )\n                                        with job.lock:\n                                            job.paid_fallback_used = True\n                                            job.message = (\n                                                f"免费 STANDARD 对象已完成，正在付费迁移 {rem_ia} 个 STANDARD_IA 对象"\n                                            )\n                                        with job.db_lock:\n                                            conn.execute(\n                                                "UPDATE objects SET status=\'pending\' WHERE status=\'ia_deferred\'"\n                                            )\n                                            conn.commit()\n                                            self._refresh_counts_locked(job, conn)\n                                        dispatch_cursor = ("", "")\n                                        continue\n                                    break\n\n                        with work_cond:\n                            work_cond.wait(timeout=0.4)\n\n                    for fut in list(pending_futs):\n                        fut.result()\n\n            # Shut down writers / TAR pipeline & lister cleanly before final R2 deletion drain.\n            if writer_threads:\n                for _ in writer_threads:\n                    write_q.put(None)\n                for wt in writer_threads:\n                    wt.join()\n            if tar_pipeline is not None:\n                tar_pipeline.close(wait=True)\n                tar_pipeline = None\n            drain_outcomes()\n            lister_thread.join(timeout=15.0)\n            drain_outcomes()\n\n            stop_deleter.set()\n            delete_wake.set()\n            if deleter_thread is not None:\n                deleter_thread.join(timeout=60.0)\n\n            if (\n                delete_source\n                and not job.cancel_requested\n                and not job.pause_requested\n                and not self._shutdown\n            ):\n                end_after_keys: dict[str, str] = {}\n                end_verifier = _DriveCloudSyncVerifier(job.dest_root, log=self._log)\n                while not self._shutdown:\n                    d_cnt, f_cnt, s_cnt = self._delete_copied_chunk(\n                        job,\n                        conn,\n                        client,\n                        batch_limit=1000,\n                        after_keys=end_after_keys,\n                        single_pass=True,\n                        cloud_verifier=end_verifier,\n                    )\n                    if d_cnt == 0 and f_cnt == 0 and s_cnt == 0:\n                        break\n\n            with job.db_lock:\n                self._refresh_counts_locked(job, conn)\n                self._refresh_buckets_locked(job, conn)\n            counts = job.status_counts\n            pending_cnt = counts.get("pending", {}).get("count", 0)\n            failed_cnt = counts.get("failed", {}).get("count", 0)\n            ia_cnt = counts.get("ia_deferred", {}).get("count", 0)\n            copied_cnt = counts.get("copied", {}).get("count", 0)\n            deleted_cnt = counts.get("deleted", {}).get("count", 0)\n\n            now_done = time.time()\n            with job.lock:\n                if job.cancel_requested:\n                    job.status = "cancelled"\n                    job.message = "任务已取消并保存断点检查点"\n                elif budget_pause_msg:\n                    job.status = "paused_budget"\n                    job.message = budget_pause_msg[0]\n                elif quota_pause_msg:\n                    job.status = "paused_quota"\n                    job.message = quota_pause_msg[0]\n                elif job.pause_requested or self._shutdown:\n                    job.status = "paused"\n                    job.message = "任务已暂停并保存断点检查点，可通过 resume 继续"\n                elif lister_error:\n                    job.status = "failed"\n                    job.message = f"列举对象失败: {lister_error[0]}"\n                elif ia_cnt > 0 and cost_mode == "free-only":\n                    job.status = "paused_budget"\n                    job.message = (\n                        f"已严格保持免费额度：已完成常规对象；剩余 {ia_cnt} 个 STANDARD_IA 低频对象未读取。"\n                        f"可执行 `resume {job.job_id} --cost-mode free-first` 继续。"\n                    )\n                elif failed_cnt > 0:\n                    job.status = "failed"\n                    job.message = f"部分对象失败 ({failed_cnt} 个)，可执行 `resume {job.job_id}` 重试失败项"\n                elif delete_source and copied_cnt > 0 and pending_cnt == 0 and ia_cnt == 0:\n                    job.status = "paused_quota"\n                    job.message = (\n                        f"全部对象已打包写入本地 Drive 挂载，并已清理云端核验通过的 {deleted_cnt} 个 R2 原文件；"\n                        f"剩余 {copied_cnt} 个对象正等待 Google Drive 云端同步 (750 GB/24h 限流)，"\n                        f"同步完成后执行 cleanup {job.job_id} 即可删空"\n                    )\n                else:\n                    job.status = "completed"\n                    if delete_source:\n                        job.message = f"迁移完成：已迁移并清理 R2 原文件 {deleted_cnt} 个"\n                    else:\n                        job.message = f"迁移完成：已写入 Drive {copied_cnt + deleted_cnt} 个对象（R2 原文件已保留）"\n                job.finished_at = now_done\n                job.session_finished_at = now_done\n            self._flush_checkpoint(job, conn)\n        finally:\n            stop_deleter.set()\n            delete_wake.set()\n            for _ in range(len(writer_threads)):\n                with contextlib.suppress(queue.Full):\n                    write_q.put_nowait(None)\n            if tar_pipeline is not None:\n                with contextlib.suppress(Exception):\n                    tar_pipeline.close(wait=True)\n\n    def _delete_copied_chunk(\n        self,\n        job: _RuntimeJob,\n        conn: sqlite3.Connection,\n        client: R2Client,\n        *,\n        bucket_filter: str | None = None,\n        batch_limit: int = 1000,\n        after_keys: dict[str, str] | None = None,\n        single_pass: bool = False,\n        cloud_verifier: _DriveCloudSyncVerifier | None = None,\n        delete_concurrency: int = 4,\n        skip_buckets: set[str] | None = None,\n        transient_bucket_errors: dict[str, str] | None = None,\n    ) -> tuple[int, int, int]:\n        """Re-verify Drive copies (TAR shard or loose file, plus Cloud sync on fuse.drive) and batch-delete up to `batch_limit` keys via Free DeleteObjects.\n\n        Returns `(deleted_total, failed_total, deferred_sync_total)`.\n        """\n        with job.db_lock:\n            if bucket_filter:\n                buckets = [bucket_filter]\n            else:\n                buckets = [\n                    str(r[0])\n                    for r in conn.execute("SELECT bucket FROM buckets ORDER BY bucket").fetchall()\n                ]\n                if not buckets:\n                    buckets = [\n                        str(r[0])\n                        for r in conn.execute("SELECT DISTINCT bucket FROM objects WHERE status=\'copied\' ORDER BY bucket").fetchall()\n                    ]\n        if not buckets:\n            return 0, 0, 0\n\n        verifier = cloud_verifier or _DriveCloudSyncVerifier(job.dest_root, log=self._log)\n        deleted_total = 0\n        failed_total = 0\n        deferred_sync_total = 0\n        shard_state_cache: dict[str, str] = {}\n        now_ts = time.time()\n        for bucket in buckets:\n            if job.stop_event.is_set() or self._shutdown:\n                break\n            if skip_buckets is not None and bucket in skip_buckets:\n                continue\n            after_key = (after_keys or {}).get(bucket, "")\n            with job.db_lock:\n                rows = conn.execute(\n                    "SELECT key, size, rel_path, is_dir FROM objects WHERE bucket=? AND status=\'copied\' AND key > ? ORDER BY key LIMIT ?",\n                    (bucket, after_key, batch_limit),\n                ).fetchall()\n                if not rows:\n                    if after_keys is not None and not single_pass:\n                        after_keys[bucket] = ""\n                    continue\n                arch_map: dict[str, tuple[int, str, str, int]] = {}\n                rel_cands: list[str] = []\n                for _, _, rp, _ in rows:\n                    s_rp = str(rp)\n                    rel_cands.append(s_rp)\n                    if not s_rp.endswith("/__r2_object__"):\n                        rel_cands.append(f"{s_rp}/__r2_object__")\n                for idx in range(0, len(rel_cands), 400):\n                    chunk_rp = rel_cands[idx : idx + 400]\n                    q_marks = ",".join("?" for _ in chunk_rp)\n                    for arp, asz, sname, rtar, tbytes in conn.execute(\n                        f"""\n                        SELECT a.rel_path, a.size, a.shard_name, t.rel_tar_path, t.tar_bytes\n                        FROM archived_files a\n                        JOIN tar_shards t ON a.shard_name = t.shard_name\n                        WHERE a.rel_path IN ({q_marks})\n                        """,\n                        chunk_rp,\n                    ).fetchall():\n                        arch_map[str(arp)] = (int(asz), str(sname), str(rtar), int(tbytes))\n\n            key_sizes: dict[str, int] = {str(k): int(sz) for k, sz, _, _ in rows}\n            verified_keys: list[str] = []\n            verify_failed: list[tuple[str, str]] = []\n            deferred_here = 0\n            for key, size, rel_path, is_dir in rows:\n                s_rel = str(rel_path)\n                arch_entry = arch_map.get(s_rel) or arch_map.get(f"{s_rel}/__r2_object__")\n                state = "missing"\n                if arch_entry is not None:\n                    asz, sname, rel_tar_path, tar_bytes = arch_entry\n                    if is_dir or int(asz) == int(size):\n                        s_state = shard_state_cache.get(sname)\n                        if s_state is None:\n                            tar_p = job.dest_root / rel_tar_path\n                            s_state = verifier.verify_file(tar_p, int(tar_bytes))\n                            shard_state_cache[sname] = s_state\n                        state = s_state\n                if state == "missing":\n                    target = job.dest_root / s_rel\n                    if is_dir:\n                        if target.is_dir():\n                            state = "synced"\n                    else:\n                        st1 = verifier.verify_file(target, int(size))\n                        if st1 != "missing":\n                            state = st1\n                        elif target.is_dir():\n                            state = verifier.verify_file(target / "__r2_object__", int(size))\n                if state == "synced":\n                    verified_keys.append(str(key))\n                elif state == "pending_sync":\n                    deferred_sync_total += 1\n                    deferred_here += 1\n                else:\n                    verify_failed.append((str(key), f"Drive pre-delete verification failed: {job.dest_root / s_rel}"))\n\n            if after_keys is not None:\n                if single_pass or deferred_here > 0:\n                    after_keys[bucket] = str(rows[-1][0])\n                elif len(rows) < batch_limit:\n                    after_keys[bucket] = ""\n\n            deltas: dict[str, tuple[int, int]] = {}\n            if verify_failed:\n                with job.db_lock:\n                    conn.executemany(\n                        "UPDATE objects SET status=\'failed\', error=?, updated_at=? WHERE bucket=? AND key=?",\n                        [(err, now_ts, bucket, k) for k, err in verify_failed],\n                    )\n                    conn.commit()\n                for k, _ in verify_failed:\n                    sz = key_sizes.get(k, 0)\n                    cc, cb = deltas.get("copied", (0, 0))\n                    deltas["copied"] = (cc - 1, cb - sz)\n                    fc, fb = deltas.get("failed", (0, 0))\n                    deltas["failed"] = (fc + 1, fb + sz)\n                failed_total += len(verify_failed)\n                with job.lock:\n                    for k, err in verify_failed[:10]:\n                        job.recent_errors.append({"key": f"{bucket}/{k}", "error": err})\n\n            if verified_keys:\n                try:\n                    res: DeleteBatchResult = client.delete_objects(\n                        bucket,\n                        verified_keys,\n                        batch_size=250,\n                        quiet=True,\n                        concurrency=delete_concurrency,\n                        log=self._log,\n                    )\n                except R2Error as err:\n                    if isinstance(err, R2HttpError) and err.status in (401, 403):\n                        raise\n                    self._log(\n                        f"job {job.job_id}: transient R2 delete failure on bucket {bucket} (deferring bucket): {err}"\n                    )\n                    with job.lock:\n                        job.recent_errors.append({"key": f"{bucket}/*", "error": str(err)})\n                    if skip_buckets is not None:\n                        skip_buckets.add(bucket)\n                    if transient_bucket_errors is not None:\n                        transient_bucket_errors[bucket] = str(err)\n                    if deltas:\n                        self._adjust_status_counts(job, deltas)\n                    continue\n                with job.db_lock:\n                    if res.deleted:\n                        conn.executemany(\n                            "UPDATE objects SET status=\'deleted\', error=\'\', updated_at=? WHERE bucket=? AND key=?",\n                            [(now_ts, bucket, k) for k in res.deleted],\n                        )\n                    if res.errors:\n                        conn.executemany(\n                            "UPDATE objects SET status=\'failed\', error=?, updated_at=? WHERE bucket=? AND key=?",\n                            [(err, now_ts, bucket, k) for k, err in res.errors.items()],\n                        )\n                    conn.commit()\n                del_bytes_chunk = 0\n                for k in res.deleted:\n                    sz = key_sizes.get(k, 0)\n                    del_bytes_chunk += sz\n                    cc, cb = deltas.get("copied", (0, 0))\n                    deltas["copied"] = (cc - 1, cb - sz)\n                    dc, db = deltas.get("deleted", (0, 0))\n                    deltas["deleted"] = (dc + 1, db + sz)\n                if single_pass and res.deleted:\n                    with job.lock:\n                        job.files_done_session += len(res.deleted)\n                        job.bytes_done_session += del_bytes_chunk\n                        job.updated_at = time.time()\n                for k in res.errors:\n                    sz = key_sizes.get(k, 0)\n                    cc, cb = deltas.get("copied", (0, 0))\n                    deltas["copied"] = (cc - 1, cb - sz)\n                    fc, fb = deltas.get("failed", (0, 0))\n                    deltas["failed"] = (fc + 1, fb + sz)\n                deleted_total += len(res.deleted)\n                failed_total += len(res.errors)\n                if res.errors:\n                    with job.lock:\n                        for k, err in list(res.errors.items())[:10]:\n                            job.recent_errors.append({"key": f"{bucket}/{k}", "error": err})\n            if deltas:\n                self._adjust_status_counts(job, deltas)\n        return deleted_total, failed_total, deferred_sync_total\n\n    def list_drive(self, inputs: list[str] | None = None, *, recursive: bool = True, limit: int = 500) -> dict[str, Any]:\n        """List migrated folders/files safely within Drive mount (or local fallback)."""\n        mount = self.drive_mount if self.drive_mount.is_dir() else (self.data_dir / "outputs")\n        mount.mkdir(parents=True, exist_ok=True)\n        mount_res = mount.resolve()\n        specs = inputs or ["r2-backup"]\n        files: list[dict[str, Any]] = []\n        dirs: list[dict[str, Any]] = []\n        truncated = False\n        for spec in specs:\n            target = resolve_drive_output_root(mount_res, spec, default_subdir="r2-backup", fallback_root=self.data_dir / "outputs")\n            if not target.exists():\n                continue\n            if target.is_file():\n                rel = str(target.relative_to(mount_res)).replace("\\\\", "/")\n                files.append({"path": rel, "size": target.stat().st_size})\n                continue\n            iterator = target.rglob("*") if recursive else target.iterdir()\n            for item in sorted(iterator):\n                if len(files) + len(dirs) >= limit:\n                    truncated = True\n                    break\n                if item.name.startswith(".") and ".ctpart." in item.name:\n                    continue\n                rel = str(item.relative_to(mount_res)).replace("\\\\", "/")\n                if item.is_dir():\n                    dirs.append({"path": rel + "/"})\n                elif item.is_file():\n                    with contextlib.suppress(OSError):\n                        files.append({"path": rel, "size": item.stat().st_size})\n            if truncated:\n                break\n        return {\n            "drive_root": str(mount_res),\n            "directories": dirs,\n            "files": files,\n            "truncated": truncated,\n        }\n', encoding="utf-8")
(R2_SRC / 'r2_migrate_app.py').write_text('"""Loopback HTTP API and built-in Web UI for the r2-migrate Colab service."""\nfrom __future__ import annotations\n\nimport contextlib\nimport http.server\nimport json\nimport os\nfrom pathlib import Path\nimport secrets\nimport signal\nimport sys\nimport threading\nfrom typing import Any\nimport urllib.parse\n\nfrom r2_migrate_engine import (\n    DEFAULT_SPOOL_BYTES,\n    DEFAULT_WORKERS,\n    DEFAULT_WRITER_WORKERS,\n    MigrationManager,\n)\nfrom r2_s3 import (\n    DEFAULT_SAFE_FREE_CLASS_A,\n    DEFAULT_SAFE_FREE_CLASS_B,\n    R2Client,\n    R2Credentials,\n)\n\nMAX_BODY_BYTES = 2 * 1024 * 1024\n\n\ndef _make_csp(nonce: str) -> str:\n    return (\n        f"default-src \'self\'; "\n        f"script-src \'nonce-{nonce}\'; "\n        f"style-src \'nonce-{nonce}\'; "\n        f"img-src \'self\' data:; "\n        f"connect-src \'self\'; "\n        f"frame-ancestors \'none\'; "\n        f"base-uri \'none\'"\n    )\n\n\nHTML_TEMPLATE = """<!doctype html>\n<html lang="zh-CN">\n<head>\n  <meta charset="utf-8">\n  <meta name="viewport" content="width=device-width, initial-scale=1">\n  <meta name="color-scheme" content="light dark">\n  <title>Cloudflare R2 → Google Drive 迁移控制台</title>\n  <style nonce="__NONCE__">\n    :root {\n      --bg-light: #f6f8fb;\n      --bg-dark: #0e131b;\n      --surface-light: #ffffff;\n      --surface-dark: #161d29;\n      --surface-2-light: #eef2f7;\n      --surface-2-dark: #1e2736;\n      --text-light: #17202b;\n      --text-dark: #e8eef7;\n      --muted-light: #526071;\n      --muted-dark: #9ba9bd;\n      --border-light: #cfd8e3;\n      --border-dark: #2c384a;\n      --brand-light: #0b57d0;\n      --brand-dark: #7cb3ff;\n      --brand-text-light: #ffffff;\n      --brand-text-dark: #071936;\n      --danger-light: #b91c1c;\n      --danger-dark: #fca5a5;\n      --danger-bg-light: #fef2f2;\n      --danger-bg-dark: #3b1518;\n      --ok-light: #15803d;\n      --ok-dark: #86efac;\n      --warn-light: #a16207;\n      --warn-dark: #fcd34d;\n\n      --bg: var(--bg-light);\n      --surface: var(--surface-light);\n      --surface-2: var(--surface-2-light);\n      --text: var(--text-light);\n      --muted: var(--muted-light);\n      --border: var(--border-light);\n      --brand: var(--brand-light);\n      --brand-text: var(--brand-text-light);\n      --danger: var(--danger-light);\n      --danger-bg: var(--danger-bg-light);\n      --ok: var(--ok-light);\n      --warn: var(--warn-light);\n\n      @media (prefers-color-scheme: dark) {\n        --bg: var(--bg-dark);\n        --surface: var(--surface-dark);\n        --surface-2: var(--surface-2-dark);\n        --text: var(--text-dark);\n        --muted: var(--muted-dark);\n        --border: var(--border-dark);\n        --brand: var(--brand-dark);\n        --brand-text: var(--brand-text-dark);\n        --danger: var(--danger-dark);\n        --danger-bg: var(--danger-bg-dark);\n        --ok: var(--ok-dark);\n        --warn: var(--warn-dark);\n      }\n\n      @supports (color: light-dark(white, black)) {\n        --bg: light-dark(var(--bg-light), var(--bg-dark));\n        --surface: light-dark(var(--surface-light), var(--surface-dark));\n        --surface-2: light-dark(var(--surface-2-light), var(--surface-2-dark));\n        --text: light-dark(var(--text-light), var(--text-dark));\n        --muted: light-dark(var(--muted-light), var(--muted-dark));\n        --border: light-dark(var(--border-light), var(--border-dark));\n        --brand: light-dark(var(--brand-light), var(--brand-dark));\n        --brand-text: light-dark(var(--brand-text-light), var(--brand-text-dark));\n        --danger: light-dark(var(--danger-light), var(--danger-dark));\n        --danger-bg: light-dark(var(--danger-bg-light), var(--danger-bg-dark));\n        --ok: light-dark(var(--ok-light), var(--ok-dark));\n        --warn: light-dark(var(--warn-light), var(--warn-dark));\n      }\n\n      color-scheme: light dark;\n      accent-color: var(--brand);\n    }\n\n    *, *::before, *::after { box-sizing: border-box; }\n    body {\n      margin: 0;\n      font-family: system-ui, -apple-system, "Segoe UI", Roboto, sans-serif;\n      line-height: 1.5;\n      background: var(--bg);\n      color: var(--text);\n    }\n    .page {\n      max-inline-size: 76rem;\n      margin-inline: auto;\n      padding: 1.25rem;\n      display: grid;\n      gap: 1.25rem;\n    }\n    .topbar {\n      display: flex;\n      flex-wrap: wrap;\n      justify-content: space-between;\n      align-items: center;\n      gap: 0.75rem;\n      padding: 1rem 1.25rem;\n      background: var(--surface);\n      border: 1px solid var(--border);\n      border-radius: 0.75rem;\n    }\n    .topbar h1 { margin: 0; font-size: 1.35rem; }\n    .subtle { color: var(--muted); font-size: 0.9rem; margin: 0.15rem 0 0; }\n    .cards {\n      display: grid;\n      grid-template-columns: repeat(auto-fit, minmax(13.5rem, 1fr));\n      gap: 0.9rem;\n    }\n    .metric {\n      background: var(--surface);\n      border: 1px solid var(--border);\n      border-radius: 0.75rem;\n      padding: 0.9rem 1rem;\n    }\n    .metric-label { font-size: 0.85rem; color: var(--muted); }\n    .metric-val { font-size: 1.35rem; font-weight: 700; margin-block: 0.2rem; font-variant-numeric: tabular-nums; }\n    .metric-sub { font-size: 0.8rem; color: var(--muted); font-variant-numeric: tabular-nums; }\n    .grid-2 {\n      display: grid;\n      grid-template-columns: repeat(auto-fit, minmax(22rem, 1fr));\n      gap: 1.25rem;\n      align-items: start;\n    }\n    .panel {\n      background: var(--surface);\n      border: 1px solid var(--border);\n      border-radius: 0.75rem;\n      padding: 1.15rem 1.25rem;\n    }\n    .panel h2 { margin: 0 0 0.9rem; font-size: 1.1rem; }\n    .panel-header {\n      display: flex;\n      justify-content: space-between;\n      align-items: center;\n      gap: 0.75rem;\n      flex-wrap: wrap;\n      margin-block-end: 0.85rem;\n    }\n    .panel-header h2 { margin: 0; }\n\n    .field {\n      display: flex;\n      flex-direction: column;\n      align-items: stretch;\n      gap: 0.25rem;\n      margin-block-end: 1rem;\n    }\n    .field label {\n      font-weight: 600;\n      font-size: 0.92rem;\n    }\n    .hint {\n      font-size: 0.82rem;\n      color: var(--muted);\n    }\n    input[type="text"],\n    input[type="password"],\n    input[type="url"],\n    input[type="number"],\n    textarea,\n    select {\n      font: inherit;\n      font-size: max(1rem, 1em);\n      padding: 0.6rem 0.75rem;\n      min-height: 48px;\n      border-radius: 0.5rem;\n      border: 1px solid var(--border);\n      background: var(--bg);\n      color: var(--text);\n      inline-size: 100%;\n    }\n    [hidden] {\n      display: none !important;\n    }\n    .tar-suboptions {\n      margin-block-start: 0.65rem;\n      padding: 0.75rem 0.85rem;\n      border-radius: 0.5rem;\n      border: 1px dashed var(--border);\n      background: var(--bg);\n    }\n    .input-row {\n      display: flex;\n      gap: 0.5rem;\n      align-items: center;\n    }\n    .input-row > input { flex: 1 1 auto; }\n    .inline-2 {\n      display: grid;\n      grid-template-columns: repeat(auto-fit, minmax(10rem, 1fr));\n      gap: 0.75rem;\n    }\n\n    fieldset {\n      border: 1px solid var(--border);\n      border-radius: 0.6rem;\n      padding: 0.75rem 0.9rem;\n      margin-block-end: 1rem;\n    }\n    legend {\n      font-weight: 600;\n      font-size: 0.9rem;\n      padding-inline: 0.35rem;\n    }\n    .choice-row {\n      display: flex;\n      align-items: flex-start;\n      gap: 0.55rem;\n      margin-block: 0.45rem;\n    }\n    .choice-row input[type="radio"],\n    .choice-row input[type="checkbox"] {\n      margin-block-start: 0.22rem;\n      inline-size: 1.15rem;\n      block-size: 1.15rem;\n      flex-shrink: 0;\n    }\n    .choice-row label {\n      font-weight: 500;\n      cursor: pointer;\n    }\n\n    .error-msg {\n      display: none;\n      color: var(--danger);\n      font-size: 0.84rem;\n      margin-block-start: 0.2rem;\n    }\n    :is(input, textarea, select):is(:user-invalid, .user-invalid-fallback) {\n      border-color: var(--danger);\n      background: var(--danger-bg);\n    }\n    :is(input, textarea, select):is(:user-invalid, .user-invalid-fallback) + .error-msg {\n      display: block;\n    }\n    :is(input, textarea, select):focus-visible,\n    button:focus-visible,\n    .table-wrapper:focus-visible {\n      outline: 3px solid var(--brand);\n      outline-offset: 2px;\n    }\n\n    .btn-row {\n      display: flex;\n      flex-wrap: wrap;\n      gap: 0.6rem;\n      align-items: center;\n    }\n    button {\n      font: inherit;\n      font-weight: 600;\n      min-height: 48px;\n      padding: 0.5rem 0.95rem;\n      border-radius: 0.5rem;\n      border: 1px solid var(--border);\n      background: var(--surface-2);\n      color: var(--text);\n      cursor: pointer;\n    }\n    @media (pointer: coarse) {\n      input[type="text"],\n      input[type="password"],\n      input[type="url"],\n      input[type="number"],\n      textarea,\n      select,\n      button {\n        min-height: 52px;\n      }\n    }\n    button.primary {\n      background: var(--brand);\n      color: var(--brand-text);\n      border-color: var(--brand);\n    }\n    button.danger {\n      color: var(--danger);\n      border-color: var(--danger);\n    }\n    button.sm {\n      min-height: 36px;\n      padding: 0.3rem 0.65rem;\n      font-size: 0.85rem;\n    }\n    button:disabled { opacity: 0.55; cursor: not-allowed; }\n\n    .status-banner {\n      padding: 0.7rem 0.9rem;\n      border-radius: 0.5rem;\n      background: var(--surface-2);\n      border: 1px solid var(--border);\n      font-size: 0.9rem;\n      min-height: 2.5rem;\n    }\n    .badge {\n      display: inline-block;\n      padding: 0.15rem 0.5rem;\n      border-radius: 999px;\n      font-size: 0.78rem;\n      font-weight: 600;\n      border: 1px solid var(--border);\n      background: var(--surface-2);\n      white-space: nowrap;\n    }\n    .badge.ok { color: var(--ok); border-color: var(--ok); }\n    .badge.warn { color: var(--warn); border-color: var(--warn); }\n    .badge.err { color: var(--danger); border-color: var(--danger); }\n\n    /* Responsive container-queried data tables */\n    .table-wrapper {\n      overflow: auto;\n      max-inline-size: 100%;\n      max-block-size: min(460px, 72vh);\n      container-type: inline-size;\n      border: 1px solid var(--border);\n      border-radius: 0.6rem;\n    }\n    table {\n      inline-size: 100%;\n      border-collapse: separate;\n      border-spacing: 0;\n      font-size: 0.88rem;\n      font-variant-numeric: tabular-nums;\n    }\n    thead {\n      position: sticky;\n      inset-block-start: 0;\n      z-index: 2;\n      background: var(--surface-2);\n    }\n    th, td {\n      text-align: start;\n      padding: 0.6rem 0.75rem;\n      border-block-end: 1px solid var(--border);\n      vertical-align: top;\n    }\n    tbody th {\n      position: sticky;\n      inset-inline-start: 0;\n      z-index: 1;\n      background: var(--surface);\n      font-weight: 600;\n    }\n    .jobs-table {\n      --label-2: "状态";\n      --label-3: "目标路径";\n      --label-4: "已迁移/已列举";\n      --label-5: "速率与详情";\n      --label-6: "操作";\n    }\n    .buckets-table {\n      --label-2: "创建时间";\n      --label-3: "已迁移/已列举";\n      --label-4: "已删除/失败/IA";\n    }\n    @container (width < 640px) {\n      table, thead, tbody, tr, th, td { display: block; }\n      thead { display: none; }\n      tr {\n        margin-block-end: 0.85rem;\n        border-block-end: 2px solid var(--border);\n        background: var(--surface);\n      }\n      tbody th {\n        position: sticky;\n        inset-block-start: 0;\n        background: var(--surface-2);\n        z-index: 2;\n      }\n      td { border-block-end: 1px solid var(--border); }\n      td:nth-child(2)::before { content: var(--label-2) ": "; font-weight: 600; color: var(--muted); }\n      td:nth-child(3)::before { content: var(--label-3) ": "; font-weight: 600; color: var(--muted); }\n      td:nth-child(4)::before { content: var(--label-4) ": "; font-weight: 600; color: var(--muted); }\n      td:nth-child(5)::before { content: var(--label-5) ": "; font-weight: 600; color: var(--muted); }\n      td:nth-child(6)::before { content: var(--label-6) ": "; font-weight: 600; color: var(--muted); }\n    }\n\n    dialog {\n      border: 1px solid var(--border);\n      border-radius: 0.75rem;\n      background: var(--surface);\n      color: var(--text);\n      padding: 1.25rem;\n      max-inline-size: min(32rem, 92vw);\n    }\n    dialog:modal {\n      overscroll-behavior: contain;\n    }\n    dialog::backdrop {\n      background: rgba(0, 0, 0, 0.55);\n      overflow: hidden;\n      overscroll-behavior: contain;\n    }\n    dialog h3 { margin: 0 0 0.6rem; }\n    .dialog-actions {\n      display: flex;\n      justify-content: flex-end;\n      gap: 0.6rem;\n      margin-block-start: 1rem;\n    }\n\n    .visually-hidden:where(:not(:focus-within, :active)) {\n      position: absolute !important;\n      clip-path: inset(50%) !important;\n      overflow: hidden !important;\n      width: 1px !important;\n      height: 1px !important;\n      margin: -1px !important;\n      padding: 0 !important;\n      border: 0 !important;\n      white-space: nowrap !important;\n    }\n  </style>\n</head>\n<body>\n  <main class="page">\n    <header class="topbar">\n      <div>\n        <h1>Cloudflare R2 → Google Drive 迁移控制台</h1>\n        <p class="subtle" id="drive-subtitle">加载运行环境信息中...</p>\n      </div>\n      <div class="btn-row">\n        <span class="badge" id="cred-badge">凭据未检查</span>\n        <button type="button" id="refresh-all-btn">刷新状态</button>\n      </div>\n    </header>\n\n    <section class="cards" aria-label="R2 月度调用额度与估算费用">\n      <article class="metric">\n        <div class="metric-label">Class A 调用 (List)</div>\n        <div class="metric-val" id="m-class-a">0</div>\n        <div class="metric-sub" id="m-class-a-sub">免费额度 1,000,000 / 月</div>\n      </article>\n      <article class="metric">\n        <div class="metric-label">Class B 调用 (Standard Get)</div>\n        <div class="metric-val" id="m-class-b">0</div>\n        <div class="metric-sub" id="m-class-b-sub">免费额度 10,000,000 / 月</div>\n      </article>\n      <article class="metric">\n        <div class="metric-label">零成本优化命中 ($0.00)</div>\n        <div class="metric-val" id="m-free-ops">0</div>\n        <div class="metric-sub" id="m-free-sub">CDN命中 / Drive跳过 / 批量删除</div>\n      </article>\n      <article class="metric">\n        <div class="metric-label">本月预估费用 (USD)</div>\n        <div class="metric-val" id="m-cost">$0.0000</div>\n        <div class="metric-sub" id="m-transferred">已迁移 0 B</div>\n      </article>\n    </section>\n\n    <div role="status" aria-live="polite" class="status-banner" id="action-status">就绪。</div>\n\n    <div class="grid-2">\n      <section class="panel" aria-labelledby="creds-heading">\n        <div class="panel-header">\n          <h2 id="creds-heading">1. Cloudflare R2 连接凭据</h2>\n          <button type="button" class="sm danger" id="clear-creds-btn">清除凭据</button>\n        </div>\n        <form id="creds-form" method="post" action="/api/credentials" aria-labelledby="creds-heading" novalidate>\n          <div class="inline-2">\n            <div class="field">\n              <label for="account_id">Account ID</label>\n              <span id="account_id-hint" class="hint">Cloudflare 账户 ID；或留空改填自定义 Endpoint</span>\n              <input type="text" id="account_id" name="account_id" pattern="[A-Za-z0-9_-]{8,64}"\n                     autocorrect="off" autocapitalize="none" spellcheck="false" enterkeyhint="next"\n                     aria-describedby="account_id-hint" data-error="account_id-error">\n              <div id="account_id-error" class="error-msg"><span role="img" aria-label="Error">❌</span> 请输入 8–64 位字母/数字/下划线 Account ID。</div>\n            </div>\n            <div class="field">\n              <label for="jurisdiction">存储管辖区 (Jurisdiction)</label>\n              <span id="jurisdiction-hint" class="hint">默认全球，欧盟可选 eu</span>\n              <select id="jurisdiction" name="jurisdiction">\n                <option value="">默认 (Global)</option>\n                <option value="eu">欧盟 (eu)</option>\n                <option value="fedramp">FedRAMP (fedramp)</option>\n              </select>\n            </div>\n          </div>\n\n          <div class="field">\n            <label for="endpoint_url">自定义 S3 Endpoint URL（可选）</label>\n            <span id="endpoint_url-hint" class="hint">示例：https://&lt;account&gt;.r2.cloudflarestorage.com</span>\n            <input type="url" id="endpoint_url" name="endpoint_url"\n                   autocorrect="off" autocapitalize="none" spellcheck="false" enterkeyhint="next"\n                   aria-describedby="endpoint_url-hint" data-error="endpoint_url-error">\n            <div id="endpoint_url-error" class="error-msg"><span role="img" aria-label="Error">❌</span> 请输入合法的 https:// 或本地回环 URL。</div>\n          </div>\n\n          <div class="field">\n            <label for="access_key_id">R2 Access Key ID <span aria-hidden="true">*</span></label>\n            <span id="access_key_id-hint" class="hint">仅迁移建议 Object Read Only；需删原文件请选 Object Read &amp; Write</span>\n            <input type="text" id="access_key_id" name="access_key_id" required minlength="6"\n                   autocomplete="username" autocorrect="off" autocapitalize="none" spellcheck="false" enterkeyhint="next"\n                   aria-describedby="access_key_id-hint" data-error="access_key_id-error">\n            <div id="access_key_id-error" class="error-msg"><span role="img" aria-label="Error">❌</span> Access Key ID 不能为空（至少 6 位）。</div>\n          </div>\n\n          <div class="field">\n            <label for="secret_access_key">R2 Secret Access Key <span aria-hidden="true">*</span></label>\n            <span id="secret_access_key-hint" class="hint">以 0600 权限保存于挂载 Drive，服务端不会在页面回显明文</span>\n            <div class="input-row">\n              <input type="password" id="secret_access_key" name="secret_access_key" required minlength="6"\n                     autocomplete="current-password" enterkeyhint="done"\n                     aria-describedby="secret_access_key-hint" data-error="secret_access_key-error">\n              <button type="button" id="toggle-secret" class="sm" aria-pressed="false" aria-label="显示 Secret Access Key">显示</button>\n            </div>\n            <div id="secret_access_key-error" class="error-msg"><span role="img" aria-label="Error">❌</span> Secret Access Key 不能为空（至少 6 位）。</div>\n          </div>\n\n          <div class="choice-row">\n            <input type="checkbox" id="persist_creds" name="persist" checked>\n            <label for="persist_creds">保存到 Drive (0600)，Colab 抢占重启后自动断点续传</label>\n          </div>\n\n          <div class="btn-row">\n            <button type="submit" class="primary" id="save-creds-btn">保存并测试连接</button>\n            <button type="button" id="list-buckets-btn">拉取 Bucket 列表</button>\n          </div>\n        </form>\n      </section>\n\n      <section class="panel" aria-labelledby="job-heading">\n        <div class="panel-header">\n          <h2 id="job-heading">2. 创建或断点续传迁移任务</h2>\n        </div>\n        <form id="job-form" method="post" action="/api/jobs" aria-labelledby="job-heading" novalidate>\n          <div class="inline-2">\n            <div class="field">\n              <label for="buckets">指定 Buckets（留空=全部）</label>\n              <span id="buckets-hint" class="hint">多个桶用英文逗号分隔，如 assets,logs</span>\n              <input type="text" id="buckets" name="buckets" autocorrect="off" autocapitalize="none" spellcheck="false"\n                     aria-describedby="buckets-hint">\n            </div>\n            <div class="field">\n              <label for="exclude_buckets">排除 Buckets（可选）</label>\n              <span id="exclude_buckets-hint" class="hint">如 temp-cache</span>\n              <input type="text" id="exclude_buckets" name="exclude_buckets" autocorrect="off" autocapitalize="none" spellcheck="false"\n                     aria-describedby="exclude_buckets-hint">\n            </div>\n          </div>\n\n          <div class="inline-2">\n            <div class="field">\n              <label for="output_dir">Google Drive 目标目录</label>\n              <span id="output_dir-hint" class="hint">相对 MyDrive 路径，自动按 &lt;dir&gt;/&lt;bucket&gt;/&lt;key&gt; 保留结构</span>\n              <input type="text" id="output_dir" name="output_dir" value="r2-backup" required\n                     autocorrect="off" autocapitalize="none" spellcheck="false"\n                     aria-describedby="output_dir-hint" data-error="output_dir-error">\n              <div id="output_dir-error" class="error-msg"><span role="img" aria-label="Error">❌</span> 请填写 Google Drive 目标目录名。</div>\n            </div>\n            <div class="field">\n              <label for="prefix">Key 前缀过滤（可选）</label>\n              <span id="prefix-hint" class="hint">仅迁移该前缀下的对象，如 2026/</span>\n              <input type="text" id="prefix" name="prefix" autocorrect="off" autocapitalize="none" spellcheck="false"\n                     aria-describedby="prefix-hint">\n            </div>\n          </div>\n\n          <div class="field">\n            <label for="public_urls">自定义 CDN 公共域名映射（可选，零 Class B 下载）</label>\n            <span id="public_urls-hint" class="hint">每行一条 bucket=https://cdn.example.com；边缘缓存命中 0 Class B，未命中自动回退 S3 签名读取</span>\n            <textarea id="public_urls" name="public_urls" rows="2" autocorrect="off" autocapitalize="none" spellcheck="false"\n                      aria-describedby="public_urls-hint"></textarea>\n          </div>\n\n          <fieldset>\n            <legend>R2 API 调用费用模式</legend>\n            <div class="choice-row">\n              <input type="radio" id="mode-free-first" name="cost_mode" value="free-first" checked>\n              <label for="mode-free-first"><strong>free-first（推荐）</strong>：先用 CDN / 免费额度 / STANDARD 层，耗尽后自动回退付费读取</label>\n            </div>\n            <div class="choice-row">\n              <input type="radio" id="mode-free-only" name="cost_mode" value="free-only">\n              <label for="mode-free-only"><strong>free-only</strong>：严格 $0.00，触及免费上限或低频 IA 对象前自动安全暂停</label>\n            </div>\n            <div class="choice-row">\n              <input type="radio" id="mode-unlimited" name="cost_mode" value="unlimited">\n              <label for="mode-unlimited"><strong>unlimited</strong>：所有存储类型一次跑完（仍启用零费用优化与成本统计）</label>\n            </div>\n          </fieldset>\n\n          <fieldset>\n            <legend>Drive 目标存储模式</legend>\n            <div class="choice-row">\n              <input type="radio" id="storage-tar" name="storage_mode" value="tar" checked>\n              <label for="storage-tar"><strong>tar 分卷归档（推荐）</strong>：本地 NVMe 打包 512 MiB 无压缩 .tar 分卷整块写入 Drive，百万小文件绕过 DriveFUSE RPC 限流</label>\n            </div>\n            <div class="choice-row">\n              <input type="radio" id="storage-files" name="storage_mode" value="files">\n              <label for="storage-files"><strong>files 散列直写</strong>：逐文件按原目录结构直接写入 Drive（适合少量大文件或需即时浏览单文件）</label>\n            </div>\n            <div id="tar-options" class="tar-suboptions">\n              <div class="field">\n                <label for="tar_shard_mb">单卷 TAR 目标大小 (MiB)</label>\n                <span id="tar-shard-hint" class="hint">达到该体积即切分下一卷（默认 512 MiB，支持 1-4096 MiB）</span>\n                <input type="number" id="tar_shard_mb" name="tar_shard_mb" min="1" max="4096" value="512" inputmode="numeric" aria-describedby="tar-shard-hint">\n              </div>\n              <div class="choice-row">\n                <input type="checkbox" id="consolidate_existing" name="consolidate_existing" checked>\n                <label for="consolidate_existing">开始迁移前先将 Drive 目标目录中已下载的散列文件整合打包进 .tar 分卷并清理散列副本（保持完整性）</label>\n              </div>\n            </div>\n          </fieldset>\n\n          <div class="inline-2">\n            <div class="field">\n              <label for="workers">并发下载线程数</label>\n              <input type="number" id="workers" name="workers" min="1" max="128" value="48" inputmode="numeric">\n            </div>\n            <div class="field">\n              <label for="writer_workers">DriveFS 写入线程数</label>\n              <input type="number" id="writer_workers" name="writer_workers" min="1" max="64" value="16" inputmode="numeric">\n            </div>\n          </div>\n\n          <div class="choice-row">\n            <input type="checkbox" id="flatten_single_bucket" name="flatten_single_bucket">\n            <label for="flatten_single_bucket">仅迁移单个桶时省略顶层 bucket 目录</label>\n          </div>\n          <div class="choice-row">\n            <input type="checkbox" id="overwrite" name="overwrite">\n            <label for="overwrite">覆盖 Drive 中大小相同的已存在文件</label>\n          </div>\n          <div class="choice-row">\n            <input type="checkbox" id="delete_source" name="delete_source">\n            <label for="delete_source">核验大小与 MD5 ETag 后删除 R2 原文件（Batch DeleteObjects 为 $0.00）</label>\n          </div>\n\n          <div class="btn-row">\n            <button type="submit" class="primary" id="start-job-btn">启动 / 断点续传迁移</button>\n          </div>\n        </form>\n      </section>\n    </div>\n\n    <section class="panel" aria-labelledby="buckets-table-title">\n      <div class="panel-header">\n        <h2 id="buckets-table-title">已发现 Buckets</h2>\n      </div>\n      <div tabindex="0" role="region" aria-labelledby="buckets-table-title" class="table-wrapper">\n        <table class="buckets-table">\n          <thead>\n            <tr>\n              <th scope="col">Bucket 名称</th>\n              <th scope="col">创建时间</th>\n              <th scope="col">已迁移 / 已列举</th>\n              <th scope="col">已删除 / 失败 / IA挂起</th>\n            </tr>\n          </thead>\n          <tbody id="buckets-tbody">\n            <tr><th scope="row">—</th><td colspan="3">点击“拉取 Bucket 列表”或启动迁移任务后显示</td></tr>\n          </tbody>\n        </table>\n      </div>\n    </section>\n\n    <section class="panel" aria-labelledby="jobs-table-title">\n      <div class="panel-header">\n        <h2 id="jobs-table-title">迁移任务队列与断点检查点</h2>\n      </div>\n      <div tabindex="0" role="region" aria-labelledby="jobs-table-title" class="table-wrapper">\n        <table class="jobs-table">\n          <thead>\n            <tr>\n              <th scope="col">任务 ID</th>\n              <th scope="col">状态</th>\n              <th scope="col">目标路径</th>\n              <th scope="col">已迁移 / 已列举</th>\n              <th scope="col">速率与说明</th>\n              <th scope="col">操作</th>\n            </tr>\n          </thead>\n          <tbody id="jobs-tbody">\n            <tr><th scope="row">—</th><td colspan="5">暂无迁移任务</td></tr>\n          </tbody>\n        </table>\n      </div>\n    </section>\n  </main>\n\n  <dialog id="confirm-dialog" aria-labelledby="confirm-title">\n    <h3 id="confirm-title">确认操作</h3>\n    <p id="confirm-body"></p>\n    <form method="dialog">\n      <div class="dialog-actions">\n        <button value="cancel" autofocus>取消</button>\n        <button value="confirm" class="danger" id="confirm-ok-btn">确认执行</button>\n      </div>\n    </form>\n  </dialog>\n\n  <script nonce="__NONCE__">\n    const fmtBytes = (n) => {\n      const v = Number(n || 0);\n      if (v < 1024) return `${v} B`;\n      const units = [\'KB\', \'MB\', \'GB\', \'TB\'];\n      let val = v / 1024;\n      let idx = 0;\n      while (val >= 1024 && idx < units.length - 1) { val /= 1024; idx += 1; }\n      return `${val.toFixed(2)} ${units[idx]}`;\n    };\n\n    const statusBanner = document.getElementById(\'action-status\');\n    const setStatus = (msg) => { statusBanner.textContent = msg; };\n\n    // Polite single live region for form field validation errors + aria-invalid synchronization\n    const announcer = document.createElement(\'div\');\n    announcer.setAttribute(\'aria-live\', \'polite\');\n    announcer.classList.add(\'visually-hidden\');\n    document.body.append(announcer);\n    let announceTimer;\n    let announcedField;\n\n    const spokenText = (node) => {\n      const copy = node.cloneNode(true);\n      copy.querySelectorAll(\'[aria-hidden="true"]\').forEach((n) => n.remove());\n      copy.querySelectorAll(\'[role="img"][aria-label]\').forEach((n) => n.replaceWith(`${n.getAttribute(\'aria-label\')}: `));\n      return copy.textContent.replace(/\\\\s+/g, \' \').trim();\n    };\n    const linkError = (el, shown) => {\n      const errorId = el.dataset.error;\n      if (!errorId) return;\n      const ids = (el.getAttribute(\'aria-describedby\') || \'\').split(/\\\\s+/).filter((id) => id && id !== errorId);\n      if (shown) ids.push(errorId);\n      if (ids.length) el.setAttribute(\'aria-describedby\', ids.join(\' \'));\n      else el.removeAttribute(\'aria-describedby\');\n    };\n    const announceError = (el) => {\n      const error = document.getElementById(el.dataset.error || \'\');\n      if (!error) return;\n      const label = el.labels?.[0] ? `${spokenText(el.labels[0])}: ` : \'\';\n      clearTimeout(announceTimer);\n      announcer.textContent = \'\';\n      announcedField = el;\n      announceTimer = setTimeout(() => { announcer.textContent = label + spokenText(error); }, 500);\n    };\n\n    const fallbackDirty = new WeakMap();\n    const supportsUserInvalid = typeof CSS !== \'undefined\' && CSS.supports && CSS.supports(\'selector(:user-invalid)\');\n    const syncAria = (el, announce = false) => {\n      if (!el.matches?.(\'input, select, textarea\') || el.type === \'checkbox\' || el.type === \'radio\') return;\n      let invalid;\n      if (supportsUserInvalid) {\n        invalid = el.matches(\':user-invalid\');\n      } else {\n        const st = fallbackDirty.get(el);\n        invalid = Boolean(st && st.submittedOrBlurred && !el.validity.valid);\n        el.classList.toggle(\'user-invalid-fallback\', invalid);\n      }\n      if (invalid) {\n        if (announce) announceError(el);\n        el.setAttribute(\'aria-invalid\', \'true\');\n        linkError(el, true);\n      } else {\n        el.setAttribute(\'aria-invalid\', \'false\');\n        linkError(el, false);\n        if (announcedField === el) {\n          clearTimeout(announceTimer);\n          announcer.textContent = \'\';\n        }\n      }\n    };\n    document.querySelectorAll(\'input, select, textarea\').forEach((el) => syncAria(el));\n    document.addEventListener(\'blur\', (e) => {\n      if (!supportsUserInvalid && e.target?.matches?.(\'input, select, textarea\')) {\n        const st = fallbackDirty.get(e.target) || { touched: false, submittedOrBlurred: false };\n        st.submittedOrBlurred = true;\n        fallbackDirty.set(e.target, st);\n      }\n      syncAria(e.target, true);\n    }, true);\n    document.addEventListener(\'input\', (e) => {\n      if (!supportsUserInvalid && e.target?.matches?.(\'input, select, textarea\')) {\n        const st = fallbackDirty.get(e.target) || { touched: false, submittedOrBlurred: false };\n        st.touched = true;\n        fallbackDirty.set(e.target, st);\n      }\n      if (e.target?.hasAttribute?.(\'aria-invalid\')) syncAria(e.target);\n    });\n    document.addEventListener(\'invalid\', (e) => {\n      if (!supportsUserInvalid) {\n        fallbackDirty.set(e.target, { touched: true, submittedOrBlurred: true });\n      }\n      syncAria(e.target, false);\n    }, true);\n\n    // Password reveal toggle\n    const secretInput = document.getElementById(\'secret_access_key\');\n    const toggleSecretBtn = document.getElementById(\'toggle-secret\');\n    toggleSecretBtn.addEventListener(\'click\', () => {\n      const showing = secretInput.type === \'text\';\n      secretInput.type = showing ? \'password\' : \'text\';\n      toggleSecretBtn.setAttribute(\'aria-pressed\', showing ? \'false\' : \'true\');\n      toggleSecretBtn.textContent = showing ? \'显示\' : \'隐藏\';\n    });\n\n    // Native modal confirmation dialog\n    const confirmDialog = document.getElementById(\'confirm-dialog\');\n    const confirmTitle = document.getElementById(\'confirm-title\');\n    const confirmBody = document.getElementById(\'confirm-body\');\n    const confirmModal = (title, body) => new Promise((resolve) => {\n      confirmTitle.textContent = title;\n      confirmBody.textContent = body;\n      const onClose = () => {\n        const result = confirmDialog.returnValue;\n        confirmDialog.returnValue = \'\';\n        confirmDialog.removeEventListener(\'close\', onClose);\n        resolve(result === \'confirm\');\n      };\n      confirmDialog.addEventListener(\'close\', onClose);\n      confirmDialog.showModal();\n    });\n\n    async function apiJson(path, opts = {}) {\n      const res = await fetch(path, {\n        headers: { \'Content-Type\': \'application/json\', ...(opts.headers || {}) },\n        ...opts,\n      });\n      const data = await res.json().catch(() => ({ error: `HTTP ${res.status}` }));\n      if (!res.ok) throw new Error(data.error || `HTTP ${res.status}`);\n      return data;\n    }\n\n    let knownBuckets = [];\n\n    function renderMetrics(usage, health) {\n      if (health) {\n        const driveTag = health.drive_mounted ? \'Google Drive 已挂载\' : \'本地回退模式 (未挂载 Drive)\';\n        document.getElementById(\'drive-subtitle\').textContent =\n          `${driveTag} · 数据目录: ${health.data_dir} · 月份: ${usage?.month || \'\'}`;\n        const credBadge = document.getElementById(\'cred-badge\');\n        if (health.credentials?.configured) {\n          credBadge.className = \'badge ok\';\n          credBadge.textContent = `R2 已配置 (${health.credentials.access_key_id_masked || health.credentials.source})`;\n          if (!document.getElementById(\'account_id\').value && health.credentials.account_id) {\n            document.getElementById(\'account_id\').value = health.credentials.account_id;\n          }\n          if (!document.getElementById(\'endpoint_url\').value && health.credentials.endpoint_url) {\n            document.getElementById(\'endpoint_url\').value = health.credentials.endpoint_url;\n          }\n        } else {\n          credBadge.className = \'badge warn\';\n          credBadge.textContent = \'未配置 R2 凭据\';\n        }\n      }\n      if (!usage) return;\n      document.getElementById(\'m-class-a\').textContent = Number(usage.class_a_ops || 0).toLocaleString();\n      document.getElementById(\'m-class-a-sub\').textContent =\n        `本月免费剩余 ${Number(usage.free_class_a_remaining || 0).toLocaleString()} / 1,000,000`;\n      const stdB = Number(usage.class_b_ops || 0);\n      const iaB = Number(usage.ia_class_b_ops || 0);\n      document.getElementById(\'m-class-b\').textContent = (stdB + iaB).toLocaleString();\n      document.getElementById(\'m-class-b-sub\').textContent =\n        `标准剩余 ${Number(usage.free_class_b_remaining || 0).toLocaleString()} · IA检索 ${iaB.toLocaleString()}`;\n      const freeTotal = Number(usage.cdn_hit_ops || 0) + Number(usage.skipped_existing_ops || 0)\n        + Number(usage.zero_byte_ops || 0) + Number(usage.free_delete_ops || 0);\n      document.getElementById(\'m-free-ops\').textContent = freeTotal.toLocaleString();\n      document.getElementById(\'m-free-sub\').textContent =\n        `CDN:${usage.cdn_hit_ops || 0} 跳过:${usage.skipped_existing_ops || 0} 零字节:${usage.zero_byte_ops || 0} 批量删:${usage.free_delete_ops || 0}`;\n      document.getElementById(\'m-cost\').textContent = `$${Number(usage.estimated_cost_usd || 0).toFixed(4)}`;\n      document.getElementById(\'m-transferred\').textContent = `已传输 ${fmtBytes(usage.bytes_transferred || 0)}`;\n    }\n\n    function renderBuckets(jobs) {\n      const tbody = document.getElementById(\'buckets-tbody\');\n      tbody.replaceChildren();\n      const statsMap = new Map();\n      for (const j of jobs || []) {\n        for (const b of j.buckets || []) {\n          statsMap.set(b.bucket, b);\n        }\n      }\n      const names = Array.from(new Set([...knownBuckets.map((b) => b.name), ...statsMap.keys()]));\n      if (!names.length) {\n        const tr = document.createElement(\'tr\');\n        const th = document.createElement(\'th\');\n        th.scope = \'row\';\n        th.textContent = \'—\';\n        const td = document.createElement(\'td\');\n        td.colSpan = 3;\n        td.textContent = \'点击“拉取 Bucket 列表”或启动迁移任务后显示\';\n        tr.append(th, td);\n        tbody.appendChild(tr);\n        return;\n      }\n      const createdMap = new Map(knownBuckets.map((b) => [b.name, b.creation_date || \'—\']));\n      for (const name of names) {\n        const st = statsMap.get(name);\n        const tr = document.createElement(\'tr\');\n        const th = document.createElement(\'th\');\n        th.scope = \'row\';\n        th.textContent = name;\n        const c1 = document.createElement(\'td\');\n        c1.textContent = createdMap.get(name) || \'—\';\n        const c2 = document.createElement(\'td\');\n        c2.textContent = st ? `${st.objects_migrated} / ${st.objects_listed}` : \'—\';\n        const c3 = document.createElement(\'td\');\n        c3.textContent = st ? `${st.objects_deleted} / ${st.objects_failed} / ${st.objects_ia_deferred}` : \'—\';\n        tr.append(th, c1, c2, c3);\n        tbody.appendChild(tr);\n      }\n    }\n\n    function makeActionButton(label, cls, onClick) {\n      const btn = document.createElement(\'button\');\n      btn.type = \'button\';\n      btn.className = `sm ${cls || \'\'}`.trim();\n      btn.textContent = label;\n      btn.addEventListener(\'click\', onClick);\n      return btn;\n    }\n\n    function renderJobs(jobs) {\n      const tbody = document.getElementById(\'jobs-tbody\');\n      tbody.replaceChildren();\n      if (!jobs || !jobs.length) {\n        const tr = document.createElement(\'tr\');\n        const th = document.createElement(\'th\');\n        th.scope = \'row\';\n        th.textContent = \'—\';\n        const td = document.createElement(\'td\');\n        td.colSpan = 5;\n        td.textContent = \'暂无迁移任务\';\n        tr.append(th, td);\n        tbody.appendChild(tr);\n        return;\n      }\n      for (const j of jobs) {\n        const tr = document.createElement(\'tr\');\n        const th = document.createElement(\'th\');\n        th.scope = \'row\';\n        th.textContent = j.id;\n\n        const tdStatus = document.createElement(\'td\');\n        const badge = document.createElement(\'span\');\n        const cls = j.status === \'completed\' ? \'ok\' : (j.status === \'failed\' || j.status === \'cancelled\' ? \'err\' : \'warn\');\n        badge.className = `badge ${cls}`;\n        badge.textContent = j.status + (j.paid_fallback_used ? \' · 含付费回退\' : \'\');\n        tdStatus.appendChild(badge);\n        const modeVal = j.progress?.storage_mode || j.config?.storage_mode || \'files\';\n        if (modeVal === \'tar\') {\n          const tarBadge = document.createElement(\'span\');\n          tarBadge.className = \'badge ok\';\n          tarBadge.style.marginInlineStart = \'0.35rem\';\n          tarBadge.textContent = \'TAR 分卷\';\n          tdStatus.appendChild(tarBadge);\n        }\n\n        const tdDest = document.createElement(\'td\');\n        tdDest.textContent = `${j.config?.dest_root || \'\'}` + (j.config?.delete_source ? \' (删原)\' : \'\');\n\n        const p = j.progress || {};\n        const tdProg = document.createElement(\'td\');\n        tdProg.textContent = `${p.objects_migrated || 0} / ${p.objects_listed || 0} (${fmtBytes(p.bytes_migrated || 0)})`\n          + (p.tar_shards ? ` · 分卷:${p.tar_shards} (${fmtBytes(p.tar_bytes || 0)})` : \'\')\n          + (p.consolidated_existing_files ? ` · 已归并散列:${p.consolidated_existing_files}` : \'\')\n          + (p.objects_deleted ? ` · 已删:${p.objects_deleted}` : \'\')\n          + (p.objects_copied_pending_delete > 0 ? ` · 待删R2:${p.objects_copied_pending_delete}` : \'\')\n          + (p.objects_pending > 0 ? ` · 待迁:${p.objects_pending}` : \'\')\n          + (p.objects_failed ? ` · 失败:${p.objects_failed}` : \'\');\n\n        const tdMsg = document.createElement(\'td\');\n        tdMsg.textContent = `${p.files_per_sec || 0} files/s · ${p.mb_per_sec || 0} MB/s · ${j.message || \'\'}`;\n\n        const tdOps = document.createElement(\'td\');\n        const wrap = document.createElement(\'div\');\n        wrap.className = \'btn-row\';\n        if (j.status === \'running\' || j.status === \'queued\') {\n          wrap.appendChild(makeActionButton(\'暂停\', \'\', async () => {\n            await apiJson(`/api/jobs/${encodeURIComponent(j.id)}/pause`, { method: \'POST\', body: \'{}\' });\n            setStatus(`已请求暂停任务 ${j.id}`);\n            await refreshAll();\n          }));\n        } else if (j.status !== \'completed\' && j.status !== \'cancelled\') {\n          wrap.appendChild(makeActionButton(\'继续\', \'primary\', async () => {\n            await apiJson(`/api/jobs/${encodeURIComponent(j.id)}/resume`, { method: \'POST\', body: \'{}\' });\n            setStatus(`已恢复任务 ${j.id}`);\n            await refreshAll();\n          }));\n        }\n        if (modeVal !== \'tar\' && j.status !== \'completed\' && j.status !== \'cancelled\') {\n          wrap.appendChild(makeActionButton(\'转 TAR 高速续传\', \'primary\', async () => {\n            const ok = await confirmModal(\n              \'切换为 TAR 分卷归档并整合散列文件\',\n              `将把任务 ${j.id} 切换为 512 MiB TAR 分卷模式（48 下载并发 + 16 写入并发）：若当前正在运行会先安全暂停，恢复后自动将 Drive 目标目录中已落盘的散列文件校验打包进 .tar 分卷并清理散列副本，再继续迁移剩余 R2 对象。确认切换？`\n            );\n            if (!ok) return;\n            if (j.status === \'running\' || j.status === \'queued\') {\n              await apiJson(`/api/jobs/${encodeURIComponent(j.id)}/pause`, { method: \'POST\', body: \'{}\' });\n            }\n            await apiJson(`/api/jobs/${encodeURIComponent(j.id)}/resume`, {\n              method: \'POST\',\n              body: JSON.stringify({\n                storage_mode: \'tar\',\n                tar_shard_mb: 512,\n                consolidate_existing: true,\n                workers: 48,\n                writer_workers: 16,\n              }),\n            });\n            setStatus(`任务 ${j.id} 已切换为 TAR 分卷归档模式并恢复执行`);\n            await refreshAll();\n          }));\n        }\n        if (p.objects_copied_pending_delete > 0 && j.status !== \'running\' && j.status !== \'queued\') {\n          wrap.appendChild(makeActionButton(`清理R2(${p.objects_copied_pending_delete})`, \'danger\', async () => {\n            const ok = await confirmModal(\n              \'确认批量删除 R2 原文件\',\n              `将核验 Drive 中已拷贝的 ${p.objects_copied_pending_delete} 个文件是否存在且大小一致，然后通过 Free DeleteObjects 从 Cloudflare R2 删除对应原对象。确认执行？`\n            );\n            if (!ok) return;\n            const cleanRes = await apiJson(`/api/jobs/${encodeURIComponent(j.id)}/cleanup`, { method: \'POST\', body: \'{}\' });\n            setStatus(cleanRes.status === \'running\' ? `已启动任务 ${j.id} 的 R2 核验清理（后台执行中）` : `已完成任务 ${j.id} 的 R2 原文件清理`);\n            await refreshAll();\n          }));\n        }\n        if (j.status !== \'completed\' && j.status !== \'cancelled\') {\n          wrap.appendChild(makeActionButton(\'取消\', \'danger\', async () => {\n            const ok = await confirmModal(\'取消迁移任务\', `确定要取消任务 ${j.id} 吗？已保存断点检查点，Drive 中已拷贝文件保留。`);\n            if (!ok) return;\n            await apiJson(`/api/jobs/${encodeURIComponent(j.id)}/cancel`, { method: \'POST\', body: \'{}\' });\n            setStatus(`已取消任务 ${j.id}`);\n            await refreshAll();\n          }));\n        }\n        tdOps.appendChild(wrap);\n        tr.append(th, tdStatus, tdDest, tdProg, tdMsg, tdOps);\n        tbody.appendChild(tr);\n      }\n    }\n\n    async function refreshAll() {\n      try {\n        const [health, jobsResp] = await Promise.all([apiJson(\'/health\'), apiJson(\'/api/jobs\')]);\n        renderMetrics(jobsResp.usage || health.usage, health);\n        renderJobs(jobsResp.jobs || []);\n        renderBuckets(jobsResp.jobs || []);\n      } catch (err) {\n        setStatus(`刷新失败: ${err.message}`);\n      }\n    }\n\n    document.getElementById(\'refresh-all-btn\').addEventListener(\'click\', refreshAll);\n\n    document.getElementById(\'clear-creds-btn\').addEventListener(\'click\', async () => {\n      const ok = await confirmModal(\'清除已保存凭据\', \'将删除内存及 Google Drive (0600) 上保存的 R2 API 凭据。确认清除？\');\n      if (!ok) return;\n      try {\n        await apiJson(\'/api/credentials\', { method: \'DELETE\' });\n        setStatus(\'已清除 R2 凭据\');\n        await refreshAll();\n      } catch (err) {\n        setStatus(`清除失败: ${err.message}`);\n      }\n    });\n\n    const acctInput = document.getElementById(\'account_id\');\n    const epInput = document.getElementById(\'endpoint_url\');\n    const clearEndpointCustomErr = () => {\n      acctInput.setCustomValidity(\'\');\n      epInput.setCustomValidity(\'\');\n      syncAria(acctInput);\n      syncAria(epInput);\n    };\n    acctInput.addEventListener(\'input\', clearEndpointCustomErr);\n    epInput.addEventListener(\'input\', clearEndpointCustomErr);\n\n    document.getElementById(\'creds-form\').addEventListener(\'submit\', async (e) => {\n      e.preventDefault();\n      const form = e.currentTarget;\n      acctInput.setCustomValidity(\'\');\n      epInput.setCustomValidity(\'\');\n      if (!acctInput.value.trim() && !epInput.value.trim()) {\n        acctInput.setCustomValidity(\'请填写 Account ID 或自定义 Endpoint URL\');\n      }\n      if (!form.reportValidity()) {\n        form.querySelector(\':invalid\')?.focus();\n        return;\n      }\n      const btn = document.getElementById(\'save-creds-btn\');\n      btn.disabled = true;\n      try {\n        const fd = new FormData(form);\n        const payload = {\n          account_id: String(fd.get(\'account_id\') || \'\').trim(),\n          jurisdiction: String(fd.get(\'jurisdiction\') || \'\').trim(),\n          endpoint_url: String(fd.get(\'endpoint_url\') || \'\').trim(),\n          access_key_id: String(fd.get(\'access_key_id\') || \'\').trim(),\n          secret_access_key: String(fd.get(\'secret_access_key\') || \'\').trim(),\n          persist: Boolean(document.getElementById(\'persist_creds\').checked),\n          verify: true,\n        };\n        const res = await apiJson(\'/api/credentials\', { method: \'POST\', body: JSON.stringify(payload) });\n        knownBuckets = res.buckets || [];\n        document.getElementById(\'secret_access_key\').value = \'\';\n        setStatus(`凭据验证成功，发现 ${knownBuckets.length} 个 Bucket`);\n        await refreshAll();\n      } catch (err) {\n        setStatus(`凭据验证/保存失败: ${err.message}`);\n      } finally {\n        btn.disabled = false;\n      }\n    });\n\n    document.getElementById(\'list-buckets-btn\').addEventListener(\'click\', async () => {\n      try {\n        const res = await apiJson(\'/api/buckets\', { method: \'POST\', body: \'{}\' });\n        knownBuckets = res.buckets || [];\n        setStatus(`已拉取 ${knownBuckets.length} 个 Bucket`);\n        await refreshAll();\n      } catch (err) {\n        setStatus(`拉取 Bucket 失败: ${err.message}`);\n      }\n    });\n\n    function syncTarOptionsVisibility() {\n      const box = document.getElementById(\'tar-options\');\n      const isTar = document.getElementById(\'storage-tar\')?.checked;\n      if (!box) return;\n      if (isTar) {\n        box.hidden = false;\n        box.inert = false;\n      } else {\n        box.hidden = true;\n        box.inert = true;\n      }\n    }\n    document.querySelectorAll(\'input[name="storage_mode"]\').forEach((el) => {\n      el.addEventListener(\'change\', syncTarOptionsVisibility);\n    });\n    syncTarOptionsVisibility();\n\n    document.getElementById(\'job-form\').addEventListener(\'submit\', async (e) => {\n      e.preventDefault();\n      const form = e.currentTarget;\n      if (!form.reportValidity()) {\n        form.querySelector(\':invalid\')?.focus();\n        return;\n      }\n      const deleteSource = document.getElementById(\'delete_source\').checked;\n      if (deleteSource) {\n        const ok = await confirmModal(\n          \'确认开启“迁移后删除 R2 原文件”\',\n          \'迁移线程会在核验文件大小与 MD5 ETag 并写入 Google Drive 后，使用 Batch DeleteObjects 删除 Cloudflare R2 上的原始文件。是否继续？\'\n        );\n        if (!ok) return;\n      }\n      const btn = document.getElementById(\'start-job-btn\');\n      btn.disabled = true;\n      try {\n        const fd = new FormData(form);\n        const splitCsv = (s) => String(s || \'\').split(\',\').map((x) => x.trim()).filter(Boolean);\n        const pubLines = String(fd.get(\'public_urls\') || \'\').split(/\\\\r?\\\\n/).map((x) => x.trim()).filter(Boolean);\n        const storageMode = String(fd.get(\'storage_mode\') || \'tar\');\n        const payload = {\n          buckets: splitCsv(fd.get(\'buckets\')),\n          exclude_buckets: splitCsv(fd.get(\'exclude_buckets\')),\n          output_dir: String(fd.get(\'output_dir\') || \'r2-backup\').trim(),\n          prefix: String(fd.get(\'prefix\') || \'\').trim(),\n          public_urls: pubLines,\n          cost_mode: String(fd.get(\'cost_mode\') || \'free-first\'),\n          storage_mode: storageMode,\n          tar_shard_mb: Number(fd.get(\'tar_shard_mb\') || 512),\n          consolidate_existing: Boolean(document.getElementById(\'consolidate_existing\').checked),\n          workers: Number(fd.get(\'workers\') || 48),\n          writer_workers: Number(fd.get(\'writer_workers\') || 16),\n          flatten_single_bucket: Boolean(document.getElementById(\'flatten_single_bucket\').checked),\n          overwrite: Boolean(document.getElementById(\'overwrite\').checked),\n          delete_source: deleteSource,\n        };\n        const res = await apiJson(\'/api/jobs\', { method: \'POST\', body: JSON.stringify(payload) });\n        setStatus(`任务 ${res.id} 已启动 (${res.resumed_from_checkpoint ? \'从检查点续传\' : \'新任务\'})`);\n        await refreshAll();\n      } catch (err) {\n        setStatus(`启动任务失败: ${err.message}`);\n      } finally {\n        btn.disabled = false;\n      }\n    });\n\n    refreshAll();\n    setInterval(refreshAll, 3000);\n  </script>\n</body>\n</html>\n"""\n\n\nclass R2MigrateHttpServer(http.server.ThreadingHTTPServer):\n    daemon_threads = True\n    allow_reuse_address = True\n\n    def __init__(self, server_address: tuple[str, int], manager: MigrationManager) -> None:\n        self.manager = manager\n        self.drive_mounted = manager.drive_mount.is_dir()\n        super().__init__(server_address, R2MigrateRequestHandler)\n\n\nclass R2MigrateRequestHandler(http.server.BaseHTTPRequestHandler):\n    server: R2MigrateHttpServer\n\n    def log_message(self, format: str, *args: Any) -> None:\n        return\n\n    def _send_json(self, status: int, payload: Any) -> None:\n        raw = json.dumps(payload, ensure_ascii=False, indent=2).encode("utf-8")\n        self.send_response(status)\n        self.send_header("Content-Type", "application/json; charset=utf-8")\n        self.send_header("Cache-Control", "no-store")\n        self.send_header("Content-Length", str(len(raw)))\n        self.end_headers()\n        with contextlib.suppress(BrokenPipeError, ConnectionResetError):\n            self.wfile.write(raw)\n\n    def _send_html(self, status: int, html: str, nonce: str) -> None:\n        raw = html.encode("utf-8")\n        self.send_response(status)\n        self.send_header("Content-Type", "text/html; charset=utf-8")\n        self.send_header("Cache-Control", "no-store")\n        self.send_header("Content-Security-Policy", _make_csp(nonce))\n        self.send_header("X-Content-Type-Options", "nosniff")\n        self.send_header("Content-Length", str(len(raw)))\n        self.end_headers()\n        with contextlib.suppress(BrokenPipeError, ConnectionResetError):\n            self.wfile.write(raw)\n\n    def _read_json(self) -> dict[str, Any]:\n        raw_len = self.headers.get("Content-Length")\n        if not raw_len:\n            return {}\n        try:\n            length = int(raw_len)\n        except ValueError:\n            raise ValueError("Invalid Content-Length") from None\n        if length <= 0:\n            return {}\n        if length > MAX_BODY_BYTES:\n            raise ValueError(f"Request body exceeds {MAX_BODY_BYTES} bytes")\n        data = self.rfile.read(length)\n        if not data:\n            return {}\n        parsed = json.loads(data.decode("utf-8"))\n        if not isinstance(parsed, dict):\n            raise ValueError("JSON body must be an object")\n        return parsed\n\n    def do_GET(self) -> None:\n        parsed = urllib.parse.urlsplit(self.path)\n        path = parsed.path.rstrip("/") or "/"\n        mgr = self.server.manager\n        if path == "/":\n            nonce = secrets.token_urlsafe(16)\n            self._send_html(200, HTML_TEMPLATE.replace("__NONCE__", nonce), nonce)\n            return\n        if path == "/health":\n            self._send_json(\n                200,\n                {\n                    "ok": True,\n                    "service": "r2-migrate",\n                    "version": "2.1",\n                    "capabilities": ["files", "tar", "consolidate_existing", "cloud_sync_verify", "adaptive_delete", "fast_cleanup"],\n                    "drive_mounted": mgr.drive_mount.is_dir(),\n                    "data_dir": str(mgr.data_dir),\n                    "work_dir": str(mgr.work_dir),\n                    "credentials": mgr.cred_store.summary(),\n                    "active_jobs": mgr.active_jobs_count(),\n                    "usage": mgr.ledger.snapshot(),\n                },\n            )\n            return\n        if path == "/api/credentials":\n            self._send_json(200, mgr.cred_store.summary())\n            return\n        if path == "/api/usage":\n            self._send_json(200, {"usage": mgr.ledger.snapshot()})\n            return\n        if path == "/api/jobs":\n            self._send_json(200, {"jobs": mgr.list_jobs(), "usage": mgr.ledger.snapshot()})\n            return\n        if path.startswith("/api/jobs/"):\n            job_id = urllib.parse.unquote(path[len("/api/jobs/") :])\n            if "/" in job_id or not job_id:\n                self._send_json(404, {"error": "Not found"})\n                return\n            try:\n                self._send_json(200, mgr.get_job(job_id))\n            except KeyError as err:\n                self._send_json(404, {"error": str(err)})\n            return\n        self._send_json(404, {"error": "Not found"})\n\n    def do_DELETE(self) -> None:\n        parsed = urllib.parse.urlsplit(self.path)\n        path = parsed.path.rstrip("/")\n        if path == "/api/credentials":\n            self.server.manager.cred_store.clear()\n            self._send_json(200, {"cleared": True, "credentials": self.server.manager.cred_store.summary()})\n            return\n        self._send_json(404, {"error": "Not found"})\n\n    def do_POST(self) -> None:\n        parsed = urllib.parse.urlsplit(self.path)\n        path = parsed.path.rstrip("/")\n        mgr = self.server.manager\n        try:\n            body = self._read_json()\n        except ValueError as err:\n            self._send_json(400, {"error": str(err)})\n            return\n\n        try:\n            if path == "/api/credentials":\n                creds = R2Credentials.from_dict(body)\n                creds.validate()\n                buckets: list[dict[str, str]] | None = None\n                if body.get("verify", True):\n                    client = R2Client(creds, ledger=mgr.ledger, timeout=mgr.client_timeout)\n                    try:\n                        buckets = client.list_buckets()\n                    finally:\n                        client.close()\n                        mgr.ledger.flush()\n                summary = mgr.cred_store.set_credentials(creds, persist=bool(body.get("persist", True)))\n                self._send_json(\n                    200,\n                    {"credentials": summary, "buckets": buckets, "usage": mgr.ledger.snapshot()},\n                )\n                return\n\n            if path == "/api/buckets":\n                buckets = mgr.list_buckets(body.get("credentials"))\n                self._send_json(200, {"buckets": buckets, "usage": mgr.ledger.snapshot()})\n                return\n\n            if path == "/api/usage":\n                usage = mgr.ledger.calibrate(\n                    class_a_ops=body.get("class_a_ops"),\n                    class_b_ops=body.get("class_b_ops"),\n                    reset=bool(body.get("reset", False)),\n                )\n                self._send_json(200, {"usage": usage})\n                return\n\n            if path == "/api/jobs":\n                raw_max_a = body.get("max_free_class_a")\n                raw_max_b = body.get("max_free_class_b")\n                raw_consolidate = body.get("consolidate_existing")\n                job = mgr.create_or_resume_job(\n                    buckets=body.get("buckets"),\n                    exclude_buckets=body.get("exclude_buckets"),\n                    prefix=str(body.get("prefix") or ""),\n                    output_dir=body.get("output_dir"),\n                    flatten_single_bucket=bool(body.get("flatten_single_bucket", False)),\n                    delete_source=bool(body.get("delete_source", False)),\n                    cost_mode=str(body.get("cost_mode") or "free-first"),\n                    public_urls=body.get("public_urls"),\n                    max_free_class_a=DEFAULT_SAFE_FREE_CLASS_A if raw_max_a is None else int(raw_max_a),\n                    max_free_class_b=DEFAULT_SAFE_FREE_CLASS_B if raw_max_b is None else int(raw_max_b),\n                    workers=body.get("workers"),\n                    writer_workers=body.get("writer_workers"),\n                    overwrite=bool(body.get("overwrite", False)),\n                    storage_mode=body.get("storage_mode"),\n                    tar_shard_mb=body.get("tar_shard_mb"),\n                    consolidate_existing=None if raw_consolidate is None else bool(raw_consolidate),\n                    new_job=bool(body.get("new_job", False)),\n                    credentials=body.get("credentials"),\n                )\n                self._send_json(201, job)\n                return\n\n            if path == "/api/drive-list":\n                res = mgr.list_drive(\n                    inputs=body.get("inputs"),\n                    recursive=bool(body.get("recursive", True)),\n                    limit=max(1, min(5000, int(body.get("limit") or 500))),\n                )\n                self._send_json(200, res)\n                return\n\n            if path.startswith("/api/jobs/"):\n                rest = path[len("/api/jobs/") :]\n                if "/" not in rest:\n                    self._send_json(404, {"error": "Not found"})\n                    return\n                raw_id, action = rest.split("/", 1)\n                job_id = urllib.parse.unquote(raw_id)\n                if action == "pause":\n                    self._send_json(200, mgr.pause_job(job_id))\n                    return\n                if action == "resume":\n                    raw_consolidate = body.get("consolidate_existing")\n                    self._send_json(\n                        200,\n                        mgr.resume_job(\n                            job_id,\n                            cost_mode=body.get("cost_mode"),\n                            delete_source=body.get("delete_source"),\n                            workers=body.get("workers"),\n                            writer_workers=body.get("writer_workers"),\n                            max_free_class_a=body.get("max_free_class_a"),\n                            max_free_class_b=body.get("max_free_class_b"),\n                            overwrite=body.get("overwrite"),\n                            public_urls=body.get("public_urls"),\n                            storage_mode=body.get("storage_mode"),\n                            tar_shard_mb=body.get("tar_shard_mb"),\n                            consolidate_existing=None if raw_consolidate is None else bool(raw_consolidate),\n                            credentials=body.get("credentials"),\n                        ),\n                    )\n                    return\n                if action == "cleanup":\n                    self._send_json(\n                        200,\n                        mgr.cleanup_job(\n                            job_id,\n                            credentials=body.get("credentials"),\n                            background=bool(body.get("background", True)),\n                        ),\n                    )\n                    return\n                if action == "cancel":\n                    self._send_json(200, mgr.cancel_job(job_id))\n                    return\n                self._send_json(404, {"error": "Unknown job action"})\n                return\n\n            self._send_json(404, {"error": "Not found"})\n        except KeyError as err:\n            self._send_json(404, {"error": str(err)})\n        except (ValueError, TypeError) as err:\n            self._send_json(400, {"error": str(err)})\n        except Exception as err:\n            self._send_json(502, {"error": str(err)})\n\n\ndef create_server(\n    *,\n    host: str = "127.0.0.1",\n    port: int | None = None,\n    data_dir: Path | None = None,\n    work_dir: Path | None = None,\n    drive_mount: Path | None = None,\n) -> tuple[R2MigrateHttpServer, MigrationManager]:\n    svc_port = int(os.environ.get("CT_SERVICE_PORT", "8000")) if port is None else int(port)\n    d_dir = Path(data_dir or os.environ.get("CT_DATA_DIR") or "/content/drive/MyDrive/colab-llm/r2-migrate")\n    w_dir = Path(work_dir or os.environ.get("CT_R2_WORK_DIR") or "/content/r2-migrate")\n    mount = Path(drive_mount or os.environ.get("CT_DRIVE_MOUNT") or "/content/drive/MyDrive")\n    workers = max(1, int(os.environ.get("CT_R2_WORKERS", str(DEFAULT_WORKERS))))\n    writers = max(1, int(os.environ.get("CT_R2_WRITER_WORKERS", str(DEFAULT_WRITER_WORKERS))))\n    spool_gb = max(0.25, float(os.environ.get("CT_R2_SPOOL_GB", "4")))\n    mgr = MigrationManager(\n        data_dir=d_dir,\n        work_dir=w_dir,\n        drive_mount=mount,\n        default_workers=workers,\n        default_writer_workers=writers,\n        spool_bytes_cap=int(spool_gb * 1024 * 1024 * 1024),\n    )\n    mgr.recover_on_startup()\n    srv = R2MigrateHttpServer((host, svc_port), mgr)\n    return srv, mgr\n\n\ndef main() -> None:\n    srv, mgr = create_server()\n\n    def handle_sig(signum: int, _frame: Any) -> None:\n        threading.Thread(target=srv.shutdown, daemon=True).start()\n\n    for sig in (signal.SIGTERM, signal.SIGINT):\n        with contextlib.suppress(ValueError, OSError):\n            signal.signal(sig, handle_sig)\n\n    try:\n        srv.serve_forever()\n    finally:\n        mgr.shutdown()\n        srv.server_close()\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")

R2_APP = R2_SRC / 'r2_migrate_app.py'
R2_DRIVE_MOUNT = Path('/content/drive/MyDrive') if (MOUNT_DRIVE and os.path.isdir('/content/drive/MyDrive')) else (Path(DATA_DIR) / 'outputs')
R2_DRIVE_MOUNT.mkdir(parents=True, exist_ok=True)
print(f"r2-migrate setup ready: cpu_count={os.cpu_count()} free_ssd={shutil.disk_usage('/content').free / 1e9:.1f}GB "
      f"data={DATA_DIR} drive_mount={R2_DRIVE_MOUNT}", flush=True)


In [ ]:
# @title 2. Service -- start loopback R2 -> Drive migration API & Web UI on 127.0.0.1:8000
if "R2_APP" not in globals():
    raise RuntimeError("r2-migrate setup cell did not run; run Cell 1 first.")

subprocess.run(["pkill", "-f", str(R2_APP)], check=False)

def start_service():
    return ct_spawn(
        "service",
        [sys.executable, str(R2_APP)],
        cwd=str(R2_SRC),
        env={
            "PYTHONPATH": str(R2_SRC),
            "PYTHONUNBUFFERED": "1",
            "CT_SERVICE_PORT": str(SERVICE_PORT),
            "CT_DATA_DIR": str(DATA_DIR),
            "CT_R2_WORK_DIR": str(R2_WORK),
            "CT_DRIVE_MOUNT": str(R2_DRIVE_MOUNT),
            "CT_R2_WORKERS": os.environ.get("CT_R2_WORKERS", "48"),
            "CT_R2_WRITER_WORKERS": os.environ.get("CT_R2_WRITER_WORKERS", "16"),
            "CT_R2_SPOOL_GB": os.environ.get("CT_R2_SPOOL_GB", "4"),
        },
    )

PROCS["service"] = (start_service(), start_service)
ct_wait_http(f"http://127.0.0.1:{SERVICE_PORT}/health", timeout=120, ok=lambda c: c == 200)
print(f"r2-migrate API & Web UI listening on 127.0.0.1:{SERVICE_PORT}", flush=True)


In [ ]:
# @title 3. Auth layer -- Caddy: basic auth for browsers, `Authorization: Bearer <token>` for API clients
# PUBLIC_PORT must be free on the VM: Colab itself uses 8080 (runtime server) and 8012 (colab dfs shim).
# Only used when the service has no auth of its own (CT_AUTH=caddy). The service itself listens on
# 127.0.0.1 only, cloudflared targets Caddy, so every public request passes through auth.
REWRITE_HOST = True   # True: send Host: 127.0.0.1:<port> upstream (Ollama & co. reject foreign Host headers)
import platform
CADDY_ARCH = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")

def start_caddy():
    if not os.path.exists("/usr/local/bin/caddy"):
        tag = json.loads(urllib.request.urlopen(
            "https://api.github.com/repos/caddyserver/caddy/releases/latest", timeout=30).read())["tag_name"]
        tgz = "/tmp/caddy.tgz"
        with urllib.request.urlopen(f"https://github.com/caddyserver/caddy/releases/download/{tag}/"
                                    f"caddy_{tag.lstrip('v')}_linux_{CADDY_ARCH}.tar.gz", timeout=60) as src, \
                open(tgz, "wb") as dst:
            shutil.copyfileobj(src, dst)
        subprocess.run(["tar", "-xzf", tgz, "-C", "/usr/local/bin", "caddy"], check=True)
    import hashlib
    cookie_tok = hashlib.sha256(f"ct-cookie-v2:{AUTH_USER}:{AUTH_PASS}:{API_TOKEN}".encode()).hexdigest()
    hashed = subprocess.check_output(["caddy", "hash-password", "--plaintext", AUTH_PASS], text=True).strip()
    upstream = f"""reverse_proxy 127.0.0.1:{SERVICE_PORT} {{
            {"header_up Host {upstream_hostport}" if REWRITE_HOST else ""}
            {"header_up Origin http://{upstream_hostport}" if REWRITE_HOST else ""}
            header_up -Authorization
            flush_interval -1
        }}"""
    caddyfile = f"""{{
    admin off
    auto_https off
}}
:{PUBLIC_PORT} {{
    encode zstd gzip
    @bearer header Authorization "Bearer {API_TOKEN}"
    handle @bearer {{
        {upstream}
    }}
    @cookie header Cookie *ct_session={cookie_tok}*
    handle @cookie {{
        {upstream}
    }}
    handle {{
        route {{
            basic_auth {{
                "{AUTH_USER}" {hashed}
            }}
            header +Set-Cookie "ct_session={cookie_tok}; Path=/; HttpOnly; SameSite=Lax"
            {upstream}
        }}
    }}
}}
"""
    open(f"{WORK}/Caddyfile", "w").write(caddyfile)
    subprocess.run(["pkill", "-f", "caddy run"], check=False)
    return ct_spawn("caddy", ["caddy", "run", "--config", f"{WORK}/Caddyfile", "--adapter", "caddyfile"])

AUTH_OK = False   # cell 4 refuses to open the tunnel unless this becomes True (fail closed)
if AUTH == "caddy":
    PROCS["caddy"] = (start_caddy(), start_caddy)
    time.sleep(2)
    if PROCS["caddy"][0].poll() is not None:   # e.g. port taken (Colab uses 8080 itself)
        raise RuntimeError(f"caddy exited -- is :{PUBLIC_PORT} in use?\n" + ct_tail("caddy"))
    # Must be 401 from *our* Caddy and must NOT leak Set-Cookie on unauthenticated 401 responses.
    ct_wait_http(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=60, ok=lambda c: c == 401)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=5)
    except urllib.error.HTTPError as _e:
        if _e.headers.get("Set-Cookie"):
            raise RuntimeError("Caddy leaked Set-Cookie on unauthenticated 401 response!")
    AUTH_OK = True
    print(f"caddy on :{PUBLIC_PORT} -> 127.0.0.1:{SERVICE_PORT} (unauthenticated request -> 401)")
else:
    AUTH_OK = True
    print(f"auth={AUTH}: no proxy, cloudflared targets the service on :{PUBLIC_PORT} directly")


In [ ]:
# @title 4. Tunnel -- Cloudflare by default, or project-configured FRP + HTTPS
"""Self-contained Colab transport cell; injected into every service notebook."""
import hashlib
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path


def render_frpc(config, local_port, ca_path):
    """TOML strings use JSON quoting, never shell interpolation."""
    return f'''serverAddr = {json.dumps(config['server_ip'])}
serverPort = {int(config['server_port'])}
loginFailExit = false
auth.method = "token"
auth.token = {json.dumps(config['token'])}
auth.additionalScopes = {json.dumps(config.get('auth_scopes',['HeartBeats','NewWorkConns']))}
transport.tls.enable = true
transport.tls.trustedCaFile = {json.dumps(str(ca_path))}
transport.tls.serverName = {json.dumps(config.get('server_name') or config['server_ip'])}
log.to = "console"
log.level = "info"
[[proxies]]
name = {json.dumps(config['proxy_name'])}
type = "tcp"
localIP = "127.0.0.1"
localPort = {int(local_port)}
remotePort = {int(config['remote_port'])}
'''


def install_frpc(destination, version):
    if destination.exists():
        return str(destination)
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise ValueError('Unsupported FRP client architecture')
    if not re.fullmatch(r'\d+\.\d+\.\d+',version) or tuple(map(int,version.split('.')))<(0,61,0):
        raise ValueError('Unsupported FRP client version; expected stable >= 0.61.0')
    asset=f'frp_{version}_linux_{arch}.tar.gz'
    base=f'https://github.com/fatedier/frp/releases/download/v{version}/'
    with tempfile.TemporaryDirectory() as tmp:
        archive=Path(tmp)/asset
        with urllib.request.urlopen(base+asset,timeout=60) as response, open(archive,'wb') as out:
            shutil.copyfileobj(response,out)
        checks=urllib.request.urlopen(base+'frp_sha256_checksums.txt',timeout=60).read().decode()
        expected=next((line.split()[0] for line in checks.splitlines()
                       if line.split() and line.split()[-1].lstrip('*')==asset),None)
        if expected is None or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected:
            raise ValueError('FRP client checksum verification failed')
        with tarfile.open(archive) as tar:
            entry=tar.getmember(f'frp_{version}_linux_{arch}/frpc')
            if not entry.isfile():
                raise ValueError('Invalid FRP client binary')
            destination.parent.mkdir(parents=True,exist_ok=True)
            with tar.extractfile(entry) as source:
                destination.write_bytes(source.read())
            destination.chmod(0o755)
    return str(destination)


def run_tunnel(ctx):
    if not ctx.get('AUTH_OK'):
        raise RuntimeError('Auth layer not ready; refusing to expose the service')
    work=Path(ctx['WORK'])
    transport=os.environ.get('CT_TRANSPORT','cloudflare')
    if transport not in ('cloudflare','frp'):
        raise ValueError('Unsupported tunnel transport')
    # Stop only the tunnel previously started by this notebook (PID + proc
    # start time prevent accidental termination after PID reuse).
    marker=work/'tunnel-process.json'
    def identity(pid):
        try:
            stat=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
            return stat[19]
        except OSError:
            return None
    if marker.exists():
        previous=json.loads(marker.read_text())
        if previous.get('start_time') and identity(previous['pid'])==previous['start_time']:
            os.kill(previous['pid'],signal.SIGTERM)
            for _ in range(50):
                if identity(previous['pid']) is None:
                    break
                time.sleep(.1)
    def spawn(name,args):
        (work/(name+'.log')).write_text('')
        proc=ctx['ct_spawn'](name,args)
        marker.write_text(json.dumps(dict(pid=proc.pid,start_time=identity(proc.pid))))
        return proc

    if transport=='frp':
        config={key:os.environ['CT_FRP_'+key.upper()] for key in
                ('server_ip','server_port','remote_port','proxy_name','token')}
        config['server_name']=os.environ.get('CT_FRP_SERVER_NAME') or config['server_ip']
        config['auth_scopes']=json.loads(os.environ.get('CT_FRP_AUTH_SCOPES') or '["HeartBeats","NewWorkConns"]')
        public_host=ctx['PUBLIC_HOST']
        if not public_host:
            raise ValueError('FRP requires a public hostname')
        ca_path=work/'frp-ca.crt'
        ca_path.write_text(os.environ['CT_FRP_CA_CERT'])
        config_path=work/'frpc.toml'
        config_path.touch(mode=0o600,exist_ok=True)
        config_path.chmod(0o600)
        config_path.write_text(render_frpc(config,ctx['PUBLIC_PORT'],ca_path))
        binary=install_frpc(work/('frpc-'+os.environ['CT_FRP_VERSION']),os.environ['CT_FRP_VERSION'])
        result=subprocess.run([binary,'verify','-c',str(config_path)],capture_output=True)
        if result.returncode:
            raise RuntimeError('FRP client configuration validation failed')
        def start():
            return spawn('frpc',[binary,'-c',str(config_path)])
        key='frpc'
    else:
        cf_arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine(),'amd64')
        binary=ctx['ct_fetch_binary']('/usr/local/bin/cloudflared',
            f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{cf_arch}')
        def refresh_quick_url(proc_ref, wait_seconds=60):
            deadline_q=time.monotonic()+wait_seconds
            while time.monotonic()<deadline_q:
                if proc_ref.poll() is not None:
                    return None
                match_q=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',ctx['ct_tail']('cloudflared',100))
                if match_q:
                    new_url=match_q.group(0)
                    if new_url!=ctx.get('PUBLIC_URL'):
                        ctx['PUBLIC_URL']=new_url
                        print(f'CT_PUBLIC_URL={new_url}',flush=True)
                    return new_url
                time.sleep(1)
            return None
        ctx['ct_refresh_public_url']=refresh_quick_url
        def start():
            args=[binary,'tunnel','--no-autoupdate','--protocol','http2']
            if ctx['TUNNEL_TOKEN']:
                args+=['run','--token',ctx['TUNNEL_TOKEN']]
            else:
                args+=['--url',f'http://127.0.0.1:{ctx["PUBLIC_PORT"]}']
            proc_new=spawn('cloudflared',args)
            if not ctx['TUNNEL_TOKEN'] and ctx.get('PUBLIC_URL'):
                threading.Thread(target=refresh_quick_url,args=(proc_new,),daemon=True).start()
            return proc_new
        key='cloudflared'
    proc=start()
    ctx['PROCS'][key]=(proc,start)
    ctx['TUNNEL_PROCESS_KEY']=key
    deadline=time.monotonic()+180
    registered=False
    while time.monotonic()<deadline:
        if proc.poll() is not None:
            raise RuntimeError(f'{key} exited before registration; see /content/ct/{key}.log')
        log=ctx['ct_tail'](key,100)
        if transport=='frp':
            registered=registered or ('start proxy success' in log and config['proxy_name'] in log)
            if registered:
                url='https://'+ctx['PUBLIC_HOST']
                try:
                    with urllib.request.urlopen(url,timeout=5) as response:
                        if response.status<600:
                            break
                except urllib.error.HTTPError as error:
                    # A valid public TLS response proves the transport/site.
                    # 502 is expected while a large model is still loading;
                    # the service's keep-alive cell gates CT_READY on health.
                    code=error.code
                    error.close()
                    if code<600:
                        break
                except OSError:
                    pass
        elif ctx['TUNNEL_TOKEN'] and 'Registered tunnel connection' in log:
            url='https://'+ctx['PUBLIC_HOST']
            break
        elif not ctx['TUNNEL_TOKEN']:
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
            if match:
                url=match.group(0)
                break
        time.sleep(1)
    else:
        raise RuntimeError(f'{key} not ready; check tunnel logs, DNS, HTTPS and server ports')
    ctx['PUBLIC_URL']=url
    print(f'CT_PUBLIC_URL={url}',flush=True)

run_tunnel(globals())


In [ ]:
# @title 5. Ready + keep-alive -- verify public Caddy Basic/Bearer auth before CT_READY
def _r2_log(msg):
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] [r2-migrate] {msg}", flush=True)

def _r2_public_probe():
    req = urllib.request.Request(
        PUBLIC_URL + "/health",
        headers={"Authorization": f"Bearer {API_TOKEN}", "User-Agent": "Mozilla/5.0"},
    )
    with urllib.request.urlopen(req, timeout=20) as resp:
        data = json.loads(resp.read().decode("utf-8"))
        if not data.get("ok"):
            raise RuntimeError(f"Public /health returned not ok: {data}")
        return resp.status

def _r2_verify_auth():
    req_unauth = urllib.request.Request(PUBLIC_URL + "/", headers={"User-Agent": "Mozilla/5.0"})
    try:
        urllib.request.urlopen(req_unauth, timeout=15)
        raise RuntimeError("Unauthenticated public GET / unexpectedly succeeded!")
    except urllib.error.HTTPError as err:
        if err.code != 401:
            raise RuntimeError(f"Expected HTTP 401 without credentials, got {err.code}")
        if err.headers.get("Set-Cookie"):
            raise RuntimeError("Public 401 leaked Set-Cookie")
    req_bad = urllib.request.Request(
        PUBLIC_URL + "/health",
        headers={"Authorization": "Bearer invalid-token", "User-Agent": "Mozilla/5.0"},
    )
    try:
        urllib.request.urlopen(req_bad, timeout=15)
        raise RuntimeError("Invalid Bearer token unexpectedly succeeded!")
    except urllib.error.HTTPError as err:
        if err.code != 401:
            raise RuntimeError(f"Expected HTTP 401 for bad Bearer token, got {err.code}")

ct_public_gate(PUBLIC_URL, _r2_public_probe, _r2_verify_auth, _r2_log, timeout=180, interval=8)
if not CREDS_FROM_ENV:
    print(f"CT_CREDENTIALS user={AUTH_USER} pass={AUTH_PASS}")
    print(f"API token: {API_TOKEN}")
print(f"{SERVICE} is up: {PUBLIC_URL}")
print("CT_READY", flush=True)

def _r2_local_health():
    with urllib.request.urlopen(f"http://127.0.0.1:{SERVICE_PORT}/health", timeout=10) as resp:
        return resp.status == 200 and json.loads(resp.read().decode("utf-8")).get("ok")

def _r2_heartbeat(uptime_s):
    with urllib.request.urlopen(f"http://127.0.0.1:{SERVICE_PORT}/health", timeout=10) as resp:
        info = json.loads(resp.read().decode("utf-8"))
    u = info.get("usage") or {}
    _r2_log(
        f"heartbeat | up={uptime_s}s | active_jobs={info.get('active_jobs', 0)} | "
        f"A={u.get('class_a_ops', 0)} B={u.get('class_b_ops', 0)} est=${u.get('estimated_cost_usd', 0.0):.4f} | "
        f"free_ssd={shutil.disk_usage('/content').free / 1e9:.1f}GB"
    )

ct_watchdog(
    _r2_local_health,
    start_service,
    _r2_log,
    service_key="service",
    log_name="service",
    grace=60,
    heartbeat=_r2_heartbeat,
)
